# Client Load Exp for Shabdiz

In [ ]:
# import subprocess
# import concurrent.futures
# import posixpath
# import shutil
# import time
# from pathlib import Path


# def run_shell(command):
#     return subprocess.call(command, shell=True)


# def run_parallel(func, iterable, max_workers):
#     with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
#         return list(executor.map(func, iterable))


# # latencies: 50, 90 150, 210
# default_region = ['us-central1-c']
# # regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# # regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']

# regions = ['us-central1-c', 'us-central1-c', 'us-central1-c', 'us-central1-c']


# # Regions

# # num_nodes = 4
# zone_no = 0

# # Use 2 extra 2-core machines as client machines.
# n_clients = 2

# # Client experiment settings.
# CLIENT_DURATION_SEC = 360
# CLIENT_WAIT_AFTER_START_SEC = CLIENT_DURATION_SEC + 45
# CLIENT_TOTAL_REQUESTS = 100000000
# CLIENT_MAX_IN_FLIGHT = 400
# SERVER_BATCH_SIZE_HINT = 100
# SEND_INTERVAL_US = 0

# # Throughput/latency load points.
# # This gives enough points for a throughput-vs-latency plot without too many subruns.
# LOAD_POINTS = [
#     # {"active_clients": 1, "client_threads": 1, "max_in_flight": 100},   # total inflight 100
#     {"active_clients": 1, "client_threads": 1, "max_in_flight": 150},   # total inflight 150
#     # {"active_clients": 1, "client_threads": 2, "max_in_flight": 100},   # total inflight 200
#     {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},   # total inflight 300
#     # {"active_clients": 1, "client_threads": 4, "max_in_flight": 100},   # total inflight 400
#     # {"active_clients": 1, "client_threads": 8, "max_in_flight": 100},   # total inflight 800
#     # {"active_clients": 2, "client_threads": 8, "max_in_flight": 100},   # total inflight 1600
# ]


# for num_nodes in [8]:
# # for zone_no in  [0,1,2,3, 4]:

#     project = "research-488322"
#     zone = "us-central1-c"
#     machine_type = "e2-standard-2"
#     image_family = "tsm-sc-family"  # your custom image
#     subnet = "default"
#     gcp_username = "tejas"

#     def get_zone_for_instance(i):
#         if i < int(num_nodes / 2):
#             return default_region[0]
#         else:
#             return regions[zone_no]

#     # Fetch all tsm-sc-* instances across ALL zones
#     fetch_cmd = f'''
#     gcloud compute instances list \
#         --project={project} \
#         --filter="name~'^tsm-sc-'" \
#         --format="value(name,zone)"
#     '''

#     output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
#     instances = []

#     for line in output.splitlines():
#         if line.strip():
#             name, inst_zone = line.split()
#             instances.append((name, inst_zone))

#     print("\n➡ Existing instances to delete:")
#     for name, inst_zone in instances:
#         print(f"  - {name} ({inst_zone})")

#     def delete_instance(instance):
#         name, inst_zone = instance
#         cmd = f'''
#         gcloud compute instances delete {name} \
#             --zone={inst_zone} \
#             --project={project} \
#             --quiet
#         '''
#         print(f"🗑️ Deleting {name} in {inst_zone}")
#         return subprocess.call(cmd, shell=True)

#     # if instances:
#     #     run_parallel(delete_instance, instances, max_workers=32)
#     #     print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
#     # else:
#     #     print("\n✔ No tsm-sc-* instances found.\n")

#     # Create commands list
#     commands = []

#     # Create replica nodes.
#     for i in range(num_nodes):
#         inst_zone = get_zone_for_instance(i)

#         cmd = f'''
#         gcloud compute instances create tsm-sc-{i:03} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())

#     # Create client machines after replica nodes.
#     for i in range(n_clients):
#         client_idx = num_nodes + i
#         inst_zone = get_zone_for_instance(client_idx)

#         cmd = f'''
#         gcloud compute instances create tsm-sc-{client_idx:03} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())

#     def run_command(command):
#         print(f"Running: {command}")
#         return subprocess.call(command, shell=True)

#     # run_parallel(run_command, commands, max_workers=48)

#     print("All instances launched.")

#     # Get sorted node and client IPs.
#     ip_cmd = f'''
#     gcloud compute instances list \
#         --project={project} \
#         --filter="name~'^tsm-sc-'" \
#         --sort-by=name \
#         --format="value(name,zone,networkInterfaces[0].networkIP)"
#     '''

#     ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

#     instance_records = []

#     for line in ip_output.splitlines():
#         if line.strip():
#             name, inst_zone, ip = line.split()
#             idx = int(name.rsplit("-", 1)[1])
#             instance_records.append((idx, name, inst_zone, ip))

#     instance_records.sort()

#     node_records = [r for r in instance_records if r[0] < num_nodes]
#     client_records = [r for r in instance_records if num_nodes <= r[0] < num_nodes + n_clients]

#     iplist = [r[3] for r in node_records]

#     with open("tsm_ips.txt", "w") as f:
#         for ip in iplist:
#             f.write(ip + "\n")

#     print("🎯 Node IPs:", iplist)
#     print("🎯 Client instances:", client_records)

#     node1_ip = iplist[0]
#     print(f"Clients will connect to leader/node1 at: {node1_ip}")

#     subprocess.call('git add .; git commit -m "testing"; git push', shell=True)

#     n_collection = 100
#     subprocess.call('make -j8', shell=True)

#     def kill_stellar_private(i):
#         inst_zone = get_zone_for_instance(i)

#         remote_command = f"""\
# cd /home/tejas/stellar-private; \
# sudo pkill -9 stellar-core; sudo pkill -9 shab_client; \
# """

#         command = (
#             f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#             f'--project "{project}" --command "{remote_command}"'
#         )

#         print(f"Executing: {command}")
#         output = subprocess.call(command, shell=True)
#         print(f"Return code for tsm-sc-{i:03}: {output}")
#         return output

#     results = run_parallel(
#         kill_stellar_private,
#         range(num_nodes + n_clients),
#         max_workers=48
#     )

#     def git_pull_stellar(i):
#         inst_zone = get_zone_for_instance(i)

#         command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# git pull"'''

#         print(command)
#         output = subprocess.call(command, shell=True)
#         print(output)
#         return output

#     results = run_parallel(
#         git_pull_stellar,
#         range(num_nodes + n_clients),
#         max_workers=48
#     )
#     print(results)

#     stellar_private_path = Path('../stellar-private')
#     if stellar_private_path.exists():
#         shutil.rmtree(stellar_private_path)
#     stellar_private_path.mkdir()

#     subprocess.call(
#         'cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     subprocess.call(
#         'cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; '
#         './gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     # Enable custom message only on leader.
#     line_to_add = "SEND_CUSTOM_MESSAGE=true"
#     target_file = "../stellar-private/node1/stellar-core.cfg"

#     subprocess.call(
#         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#         shell=True
#     )

#     # Optional memory profiling on node2.
#     target_file = "../stellar-private/node2/stellar-core.cfg"
#     line_to_add = "MEMORY_PROF=true"

#     subprocess.call(
#         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#         shell=True
#     )

#     print(f"The line '{line_to_add}' has been prepended to {target_file}.")

#     def compile_stellar(i):
#         inst_zone = get_zone_for_instance(i)

#         command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# g++ -O2 -std=c++17 -pthread \
# -I/home/tejas/stellar-core/src \
# /home/tejas/stellar-core/shab_client.cpp \
# -o /home/tejas/stellar-core/shab_client; \
# make -j16; \
# cd; \
# sudo rm -rf stellar-private"'''

#         print(command)
#         output = subprocess.call(command, shell=True)
#         print(output)
#         return output

#     # results = run_parallel(
#     #     compile_stellar,
#     #     range(num_nodes + n_clients),
#     #     max_workers=48
#     # )
#     print(results)

#     # -------------------------------------------------------------------------
#     # Throughput/latency experiment loop.
#     # No sleep intervals. SEND_INTERVAL_US is fixed at 0.
#     # We vary client concurrency to create the throughput-vs-latency points.
#     # -------------------------------------------------------------------------
#     for load in LOAD_POINTS:
#         active_clients = load["active_clients"]
#         client_threads = load["client_threads"]
#         client_max_in_flight = load["max_in_flight"]
#         total_client_threads = active_clients * client_threads
#         aggregate_max_in_flight = active_clients * client_threads * client_max_in_flight
        
#         total_client_threads = active_clients * client_threads

#         run_label = (
#             f"clients_{active_clients}_threads_{client_threads}_"
#             f"inflight_{client_max_in_flight}_"
#             f"total_threads_{total_client_threads}_"
#             f"total_inflight_{aggregate_max_in_flight}"
#         )

#         print("\n" + "=" * 80)
#         print(f"🚀 Starting throughput/latency run: {run_label}")
#         print("=" * 80)

#         def clean_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             remote_command = f"""\
# cd /home/tejas; \
# sudo rm -rf stellar-private; \
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")
#             output = subprocess.call(command, shell=True)
#             print(f"Return code for tsm-sc-{i:03}: {output}")
#             return output

#         results = run_parallel(
#             clean_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=48
#         )

#         def copy_folder_to_instance(
#             i,
#             source_folder="/home/tejas/stellar-private",
#             destination_path="/home/tejas/stellar-private"
#         ):
#             inst_zone = get_zone_for_instance(i)
#             instance_name = f"tsm-sc-{i:03}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{source_folder}" "{instance_name}:{destination_path}"'''

#             print(f"Executing command for {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Command for {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         results = run_parallel(
#             copy_folder_to_instance,
#             range(num_nodes + n_clients),
#             max_workers=48
#         )

#         def run_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             node_number = i + 1
#             instance_name = f"tsm-sc-{i:03}"

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Kill old processes on all replica and client machines.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=48
#         )

#         # Start consensus replicas.
#         results = run_parallel(
#             run_stellar_private,
#             range(num_nodes),
#             max_workers=48
#         )

#         print(results)
#         print("All Stellar nodes should be starting in the background.")

#         # Give nodes time to authenticate and start the client listener.
#         time.sleep(60)

#         def run_stellar_client(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
# {client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
# {CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
# > stellar-client-{client_id}.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing client command: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Start only the required number of client VMs for this load point.
#         active_client_indices = [
#             num_nodes + j for j in range(active_clients)
#         ]

#         results = run_parallel(
#             run_stellar_client,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print(results)
#         print(
#             f"Started {active_clients} client VM(s), "
#             f"each with {client_threads} client threads. "
#             f"Total client threads = {total_client_threads}."
#         )

#         # Wait for the duration run to produce stable per-second client logs.
#         # The clients may wait forever on final partial batches, so we kill them after this.
#         time.sleep(CLIENT_WAIT_AFTER_START_SEC)

#         # Stop all nodes and clients.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=48
#         )

#         remote_base_folder = "/home/tejas/stellar-private"

#         local_base_destination = (
#             "/home/tejas/work/experiments/shabdiz/"
#             + f"shab_tput_latency_{num_nodes}_{run_label}"
#         )

#         Path(local_base_destination).mkdir(parents=True, exist_ok=True)

#         # Save run metadata.
#         with open(Path(local_base_destination) / "run_config.txt", "w") as f:
#             f.write(f"num_nodes={num_nodes}\n")
#             f.write(f"active_clients={active_clients}\n")
#             f.write(f"client_threads_per_vm={client_threads}\n")
#             f.write(f"total_client_threads={total_client_threads}\n")
#             f.write(f"client_max_in_flight_per_thread={CLIENT_MAX_IN_FLIGHT}\n")
#             f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
#             f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
#             f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
#             f.write(f"leader_ip={node1_ip}\n")
#             f.write(f"machine_type={machine_type}\n")
#             f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
#             f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

#         def copy_folder_from_instance(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"

#             node_number = i + 1
#             node_folder = f"node{node_number}"

#             remote_source_path = posixpath.join(remote_base_folder, node_folder)

#             local_destination_path = Path(local_base_destination) / instance_name
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             remote_source = f"{instance_name}:{remote_source_path}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{remote_source}" "{local_destination_path}"'''

#             print(f"Executing command to copy {node_folder} from {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy from {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         # Copy only a few node logs to reduce time.
#         # Change range(min(3, num_nodes)) to range(num_nodes) if you want all logs.
#         node_copy_results = run_parallel(
#             copy_folder_from_instance,
#             range(min(3, num_nodes)),
#             max_workers=48
#         )

#         def copy_client_log(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_source = (
#                 f"{instance_name}:/home/tejas/stellar-private/"
#                 f"stellar-client-{client_id}.log"
#             )

#             local_destination_path = Path(local_base_destination)
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

#             print(f"Copying client log from {instance_name}...")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy finished with exit code: {output}")

#             return (instance_name, output)

#         client_copy_results = run_parallel(
#             copy_client_log,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print("\n--- Summary of Download Results ---")
#         print("Node log copies:", node_copy_results)
#         print("Client log copies:", client_copy_results)
#         print(f"Saved run to: {local_base_destination}")

# Main num nodes experiment

In [22]:
# import subprocess
# import concurrent.futures
# import posixpath
# import shutil
# import time
# from pathlib import Path


# def run_shell(command):
#     return subprocess.call(command, shell=True)


# def run_parallel(func, iterable, max_workers):
#     items = list(iterable)
#     if not items:
#         return []
#     with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
#         return list(executor.map(func, items))


# # latencies: 50, 90 150, 210
# default_region = ['us-central1-a']
# # regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# # regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']

# regions = ['us-central1-a', 'us-central1-a', 'us-central1-a', 'us-central1-a']


# # Regions

# zone_no = 0

# # Run throughput/latency vs num_nodes for these system sizes.
# NUM_NODES_LIST = [4]
# # NUM_NODES_LIST = [32]

# # Use 1 extra 2-core machine as client machine.
# # For num_nodes = N, client is tsm-sc-N.
# n_clients = 1

# # Start clean only once, before the largest run.
# # After that, keep the lower-index VMs and delete only the extra higher-index VMs.
# DELETE_BEFORE_FIRST_RUN = True

# # Since NUM_NODES_LIST goes 48 -> 32 -> 16 -> 8 -> 4, delete only the VMs
# # that will not be needed by the next smaller run.
# DELETE_UNUSED_AFTER_EACH_RUN = True

# # Delete the remaining 4 replica VMs + 1 client VM after the final run.
# DELETE_ALL_AFTER_FINAL_RUN = True

# # Code is unchanged across node-count runs, so compile only on the first run.
# COMPILE_ONLY_FIRST_RUN = True

# MAX_NUM_NODES = max(NUM_NODES_LIST)

# # Client experiment settings.
# CLIENT_DURATION_SEC = 360
# CLIENT_WAIT_AFTER_START_SEC = 210
# CLIENT_TOTAL_REQUESTS = 100000000
# SERVER_BATCH_SIZE_HINT = 100
# SEND_INTERVAL_US = 0

# # Fixed load point for throughput-vs-num_nodes and latency-vs-num_nodes.
# # Aggregate max in-flight = active_clients * client_threads * max_in_flight = 200.
# LOAD_POINTS = [
#     {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},   # total inflight 300
# ]


# for run_idx, num_nodes in enumerate(NUM_NODES_LIST):
# # for zone_no in  [0,1,2,3, 4]:

#     project = "research-488322"
#     zone = "us-central1-a"
#     machine_type = "e2-standard-2"
#     image_family = "tsm-sc-family"  # your custom image
#     subnet = "default"
#     gcp_username = "tejas"

#     print("\n" + "#" * 100)
#     print(f"Starting experiment for num_nodes={num_nodes}")
#     print("#" * 100)

#     def get_zone_for_instance(i):
#         # IMPORTANT: use MAX_NUM_NODES instead of current num_nodes.
#         # We reuse VMs while moving 48 -> 32 -> 16 -> 8 -> 4, so the zone for
#         # tsm-sc-016, tsm-sc-032, etc. must stay the same across runs.
#         if i < int(MAX_NUM_NODES / 2):
#             return default_region[0]
#         else:
#             return regions[zone_no]

#     def fetch_existing_instances():
#         fetch_cmd = f'''
#         gcloud compute instances list \
#             --project={project} \
#             --filter="name~'^tsm-sc-'" \
#             --format="value(name,zone)"
#         '''

#         output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
#         instances = []

#         for line in output.splitlines():
#             if line.strip():
#                 name, inst_zone = line.split()
#                 instances.append((name, inst_zone))

#         return instances

#     def delete_instance(instance):
#         name, inst_zone = instance

#         cmd = f'''
#         gcloud compute instances delete {name} \
#             --zone={inst_zone} \
#             --project={project} \
#             --quiet
#         '''

#         print(f"🗑️ Deleting {name} in {inst_zone}")
#         return subprocess.call(cmd, shell=True)

#     def parse_tsm_index(name):
#         return int(name.rsplit("-", 1)[1])

#     # -------------------------------------------------------------------------
#     # Delete existing tsm-sc-* instances only before the first/largest run.
#     # Later runs reuse tsm-sc-000 ... tsm-sc-(next_num_nodes).
#     # -------------------------------------------------------------------------
#     if DELETE_BEFORE_FIRST_RUN and run_idx == 0:
#         instances = fetch_existing_instances()

#         print("\n➡ Existing instances to delete before first run:")
#         for name, inst_zone in instances:
#             print(f"  - {name} ({inst_zone})")

#         if instances:
#             run_parallel(
#                 delete_instance,
#                 instances,
#                 max_workers=min(32, len(instances))
#             )
#             print("\n🧹 All existing tsm-sc-* instances deleted before first run.\n")
#         else:
#             print("\n✔ No existing tsm-sc-* instances found before first run.\n")

#     # -------------------------------------------------------------------------
#     # Create only the missing replica/client machines for this num_nodes run.
#     # When moving 48 -> 32 -> 16 -> 8 -> 4, the lower-index machines are reused.
#     # Replicas: tsm-sc-000 ... tsm-sc-(num_nodes-1)
#     # Client:   tsm-sc-num_nodes
#     # -------------------------------------------------------------------------
#     existing_instances = fetch_existing_instances()
#     existing_names = {name for name, _ in existing_instances}

#     commands = []
#     newly_created_indices = []

#     # Create replica nodes only if they do not already exist.
#     for i in range(num_nodes):
#         instance_name = f"tsm-sc-{i:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing replica {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(i)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(i)

#     # Create client machine only if it does not already exist.
#     for i in range(n_clients):
#         client_idx = num_nodes + i
#         instance_name = f"tsm-sc-{client_idx:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing client {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(client_idx)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(client_idx)

#     def run_command(command):
#         print(f"Running: {command}")
#         return subprocess.call(command, shell=True)

#     if commands:
#         run_parallel(
#             run_command,
#             commands,
#             max_workers=min(48, len(commands))
#         )

#         print("All missing instances launched.")

#         # Give GCP/SSH a little time after VM creation.
#         time.sleep(30)
#     else:
#         print("✔ All required instances already exist; no VM creation needed.")

#     # -------------------------------------------------------------------------
#     # Get sorted node and client IPs.
#     # IMPORTANT: tsm_ips.txt must include only server/replica IPs.
#     # Client IPs must not be included.
#     # -------------------------------------------------------------------------
#     ip_cmd = f'''
#     gcloud compute instances list \
#         --project={project} \
#         --filter="name~'^tsm-sc-'" \
#         --sort-by=name \
#         --format="value(name,zone,networkInterfaces[0].networkIP)"
#     '''

#     ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

#     instance_records = []

#     for line in ip_output.splitlines():
#         if line.strip():
#             name, inst_zone, ip = line.split()
#             idx = int(name.rsplit("-", 1)[1])
#             instance_records.append((idx, name, inst_zone, ip))

#     instance_records.sort()

#     node_records = [r for r in instance_records if r[0] < num_nodes]
#     client_records = [r for r in instance_records if num_nodes <= r[0] < num_nodes + n_clients]

#     if len(node_records) != num_nodes:
#         raise RuntimeError(
#             f"Expected {num_nodes} replica nodes, but found {len(node_records)}: {node_records}"
#         )

#     if len(client_records) != n_clients:
#         raise RuntimeError(
#             f"Expected {n_clients} client nodes, but found {len(client_records)}: {client_records}"
#         )

#     iplist = [r[3] for r in node_records]

#     with open("tsm_ips.txt", "w") as f:
#         for ip in iplist:
#             f.write(ip + "\n")

#     print("🎯 Node IPs:", iplist)
#     print("🎯 Client instances:", client_records)

#     node1_ip = iplist[0]
#     print(f"Clients will connect to leader/node1 at: {node1_ip}")

#     # -------------------------------------------------------------------------
#     # Push/pull/compile only for the first run.
#     # Later runs reuse the same lower-index VMs, and the code has not changed.
#     # -------------------------------------------------------------------------
#     def kill_stellar_private(i):
#         inst_zone = get_zone_for_instance(i)

#         remote_command = f"""\
# cd /home/tejas/stellar-private; \
# sudo pkill -9 stellar-core; sudo pkill -9 shab_client; \
# """

#         command = (
#             f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#             f'--project "{project}" --command "{remote_command}"'
#         )

#         print(f"Executing: {command}")
#         output = subprocess.call(command, shell=True)
#         print(f"Return code for tsm-sc-{i:03}: {output}")
#         return output

#     do_compile_this_run = (not COMPILE_ONLY_FIRST_RUN) or (run_idx == 0)

#     if do_compile_this_run:
#         subprocess.call('git add .; git commit -m "testing"; git push', shell=True)

#         n_collection = 100
#         subprocess.call('make -j8', shell=True)

#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def git_pull_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# git pull"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         results = run_parallel(
#             git_pull_stellar,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )
#         print(results)

#         # ---------------------------------------------------------------------
#         # Compile only on the first/largest run.
#         # Since later runs reuse a subset of these VMs, no recompilation is needed.
#         # ---------------------------------------------------------------------
#         def compile_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# g++ -O2 -std=c++17 -pthread \
# -I/home/tejas/stellar-core/src \
# /home/tejas/stellar-core/shab_client.cpp \
# -o /home/tejas/stellar-core/shab_client; \
# make -j4; \
# cd; \
# sudo rm -rf stellar-private"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         results = run_parallel(
#             compile_stellar,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )
#         print(results)
#     else:
#         print(f"⏭️ Skipping git pull and compilation for num_nodes={num_nodes}; code already compiled on first run.")

#     # -------------------------------------------------------------------------
#     # Generate stellar-private configs locally using only replica IPs.
#     # -------------------------------------------------------------------------
#     stellar_private_path = Path('../stellar-private')
#     if stellar_private_path.exists():
#         shutil.rmtree(stellar_private_path)
#     stellar_private_path.mkdir()

#     subprocess.call(
#         'cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     subprocess.call(
#         'cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; '
#         './gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     # Enable custom message only on leader.
#     line_to_add = "SEND_CUSTOM_MESSAGE=true"
#     target_file = "../stellar-private/node1/stellar-core.cfg"

#     subprocess.call(
#         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#         shell=True
#     )

#     # Optional memory profiling on node2.
#     # if num_nodes >= 2:
#     #     target_file = "../stellar-private/node2/stellar-core.cfg"
#     #     line_to_add = "MEMORY_PROF=true"

#     #     subprocess.call(
#     #         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#     #         shell=True
#     #     )

#     #     print(f"The line '{line_to_add}' has been prepended to {target_file}.")

#     # -------------------------------------------------------------------------
#     # Throughput/latency experiment loop.
#     # Fixed offered load for scalability:
#     # active_clients=1, client_threads=2, max_in_flight=100.
#     # Aggregate max in-flight = 200.
#     # -------------------------------------------------------------------------
#     for load in LOAD_POINTS:
#         active_clients = load["active_clients"]
#         client_threads = load["client_threads"]
#         client_max_in_flight = load["max_in_flight"]

#         if active_clients > n_clients:
#             raise RuntimeError(
#                 f"active_clients={active_clients} exceeds n_clients={n_clients}"
#             )

#         total_client_threads = active_clients * client_threads
#         aggregate_max_in_flight = active_clients * client_threads * client_max_in_flight

#         run_label = (
#             f"clients_{active_clients}_threads_{client_threads}_"
#             f"inflight_{client_max_in_flight}_"
#             f"total_threads_{total_client_threads}_"
#             f"total_inflight_{aggregate_max_in_flight}"
#         )

#         print("\n" + "=" * 80)
#         print(f"🚀 Starting throughput/latency run: num_nodes={num_nodes}, {run_label}")
#         print("=" * 80)

#         def clean_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             remote_command = f"""\
# cd /home/tejas; \
# sudo rm -rf stellar-private; \
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")
#             output = subprocess.call(command, shell=True)
#             print(f"Return code for tsm-sc-{i:03}: {output}")
#             return output

#         results = run_parallel(
#             clean_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def copy_folder_to_instance(
#             i,
#             source_folder="/home/tejas/stellar-private",
#             destination_path="/home/tejas/stellar-private"
#         ):
#             inst_zone = get_zone_for_instance(i)
#             instance_name = f"tsm-sc-{i:03}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{source_folder}" "{instance_name}:{destination_path}"'''

#             print(f"Executing command for {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Command for {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         results = run_parallel(
#             copy_folder_to_instance,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def run_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             node_number = i + 1
#             instance_name = f"tsm-sc-{i:03}"

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Kill old processes on all replica and client machines.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         # Start consensus replicas.
#         results = run_parallel(
#             run_stellar_private,
#             range(num_nodes),
#             max_workers=min(48, num_nodes)
#         )

#         print(results)
#         print("All Stellar nodes should be starting in the background.")

#         # Give nodes time to authenticate and start the client listener.
#         time.sleep(80)

#         def run_stellar_client(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
# {client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
# {CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
# > stellar-client-{client_id}.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing client command: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Start only the required number of client VMs for this load point.
#         active_client_indices = [
#             num_nodes + j for j in range(active_clients)
#         ]

#         results = run_parallel(
#             run_stellar_client,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print(results)
#         print(
#             f"Started {active_clients} client VM(s), "
#             f"each with {client_threads} client threads. "
#             f"Total client threads = {total_client_threads}. "
#             f"Aggregate max in-flight = {aggregate_max_in_flight}."
#         )

#         # Wait for the duration run to produce stable per-second client logs.
#         # The clients may wait forever on final partial batches, so we kill them after this.
#         time.sleep(CLIENT_WAIT_AFTER_START_SEC)

#         # Stop all nodes and clients.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         remote_base_folder = "/home/tejas/stellar-private"

#         local_base_destination = (
#             "/home/tejas/work/experiments/shabdiz/"
#             + f"PBFT_vs_num_nodes_{num_nodes}_{run_label}"
#         )

#         Path(local_base_destination).mkdir(parents=True, exist_ok=True)

#         # Save run metadata.
#         with open(Path(local_base_destination) / "run_config.txt", "w") as f:
#             f.write(f"num_nodes={num_nodes}\n")
#             f.write(f"active_clients={active_clients}\n")
#             f.write(f"client_threads_per_vm={client_threads}\n")
#             f.write(f"total_client_threads={total_client_threads}\n")
#             f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
#             f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
#             f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
#             f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
#             f.write(f"leader_ip={node1_ip}\n")
#             f.write(f"machine_type={machine_type}\n")
#             f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

#         def copy_folder_from_instance(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"

#             node_number = i + 1
#             node_folder = f"node{node_number}"

#             remote_source_path = posixpath.join(remote_base_folder, node_folder)

#             local_destination_path = Path(local_base_destination) / instance_name
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             remote_source = f"{instance_name}:{remote_source_path}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{remote_source}" "{local_destination_path}"'''

#             print(f"Executing command to copy {node_folder} from {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy from {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         # Copy only a few node logs to reduce time.
#         # Change range(min(3, num_nodes)) to range(num_nodes) if you want all logs.
#         node_copy_results = run_parallel(
#             copy_folder_from_instance,
#             range(min(3, num_nodes)),
#             max_workers=min(48, max(1, min(3, num_nodes)))
#         )

#         def copy_client_log(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_source = (
#                 f"{instance_name}:/home/tejas/stellar-private/"
#                 f"stellar-client-{client_id}.log"
#             )

#             local_destination_path = Path(local_base_destination)
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

#             print(f"Copying client log from {instance_name}...")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy finished with exit code: {output}")

#             return (instance_name, output)

#         client_copy_results = run_parallel(
#             copy_client_log,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print("\n--- Summary of Download Results ---")
#         print("Node log copies:", node_copy_results)
#         print("Client log copies:", client_copy_results)
#         print(f"Saved run to: {local_base_destination}")

#     # -------------------------------------------------------------------------
#     # After this run, delete only the machines that the next smaller run will
#     # not need. Example:
#     #   after 48-node run, keep 000..032 and delete 033..048
#     #   after 32-node run, keep 000..016 and delete 017..032
#     #   after 16-node run, keep 000..008 and delete 009..016
#     #   after 8-node run,  keep 000..004 and delete 005..008
#     # Final run optionally deletes everything.
#     # -------------------------------------------------------------------------
#     if DELETE_UNUSED_AFTER_EACH_RUN:
#         instances = fetch_existing_instances()

#         if run_idx + 1 < len(NUM_NODES_LIST):
#             next_num_nodes = NUM_NODES_LIST[run_idx + 1]
#             keep_count = next_num_nodes + n_clients

#             instances_to_delete = [
#                 (name, inst_zone)
#                 for name, inst_zone in instances
#                 if parse_tsm_index(name) >= keep_count
#             ]

#             print(
#                 f"\n➡ After num_nodes={num_nodes}, next run needs "
#                 f"indices 0..{keep_count - 1}. Deleting higher-index VMs:"
#             )
#         elif DELETE_ALL_AFTER_FINAL_RUN:
#             instances_to_delete = instances

#             print("\n➡ Final run complete. Deleting all remaining tsm-sc-* VMs:")
#         else:
#             instances_to_delete = []

#             print("\n✔ Final run complete. Leaving remaining tsm-sc-* VMs running.")

#         for name, inst_zone in instances_to_delete:
#             print(f"  - {name} ({inst_zone})")

#         if instances_to_delete:
#             run_parallel(
#                 delete_instance,
#                 instances_to_delete,
#                 max_workers=min(48, len(instances_to_delete))
#             )
#             print(f"\n🧹 Deleted {len(instances_to_delete)} unneeded instance(s) after num_nodes={num_nodes}.\n")
#         else:
#             print("\n✔ No unneeded instances to delete after this run.\n")



####################################################################################################
Starting experiment for num_nodes=4
####################################################################################################



➡ Existing instances to delete before first run:

✔ No existing tsm-sc-* instances found before first run.



Running: gcloud compute instances create tsm-sc-000             --project=research-488322             --zone=us-central1-a             --machine-type=e2-standard-2             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default             --can-ip-forward             --maintenance-policy=MIGRATE             --provisioning-model=STANDARD             --service-account=254510644191-compute@developer.gserviceaccount.com             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append             --tags=http-server,https-server             --create-disk=auto-delete=yes,boot=yes,image-family=tsm-sc-family,mode=rw,size=20,type=pd-balanced             --no-shielded-secure-boot             --shielded-vtpm             

Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-004].
 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert



NAME        ZONE           MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP     STATUS
tsm-sc-004  us-central1-a  e2-standard-2               10.128.0.15  146.148.91.103  RUNNING


...

2026-06-28T09:46:41.124 [default INFO] Config from /home/tejas/stellar-private/node4/stellar-core.cfg
2026-06-28T09:46:41.126 [default INFO] Generated QUORUM_SET: {
   "t" : 3,
   "v" : [ "GA7RZ", "node1", "node2", "node3" ]
}

2026-06-28T09:46:41.126 [default WARNING] Adjusted TARGET_PEER_CONNECTIONS to 336 due to insufficient MAX_ADDITIONAL_PEER_CONNECTIONS=1000
2026-06-28T09:46:41.126 [default INFO] Assigning calculated value of 1 to FAILURE_SAFETY



🚀 Starting throughput/latency run: num_nodes=4, clients_1_threads_2_inflight_150_total_threads_2_total_inflight_300
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-000" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-001" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-002" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-003" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-004" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Return code for tsm-sc-000: 0
Return code for tsm-sc-003: 0
Return code for tsm-sc-001: 0
Return code for tsm-sc-004: 0
Ret

Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-003].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-001].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-004].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-000].



🧹 Deleted 5 unneeded instance(s) after num_nodes=4.



Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-002].


# Failure Experiment

In [28]:
# import subprocess
# import concurrent.futures
# import posixpath
# import shutil
# import time
# from pathlib import Path


# def run_shell(command):
#     return subprocess.call(command, shell=True)


# def run_parallel(func, iterable, max_workers):
#     items = list(iterable)
#     if not items:
#         return []
#     with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
#         return list(executor.map(func, items))


# # latencies: 50, 90 150, 210
# default_region = ['us-central1-c']
# # regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# # regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']

# regions = ['us-central1-c', 'us-central1-c', 'us-central1-c', 'us-central1-c']


# # Regions

# zone_no = 0

# # Run throughput/latency vs num_nodes for these system sizes.
# NUM_NODES_LIST = [16]

# # Use 1 extra 2-core machine as client machine.
# # For num_nodes = N, client is tsm-sc-N.
# n_clients = 1

# # Start clean only once, before the largest run.
# # After that, keep the lower-index VMs and delete only the extra higher-index VMs.
# DELETE_BEFORE_FIRST_RUN = False

# # Since NUM_NODES_LIST goes 48 -> 32 -> 16 -> 8 -> 4, delete only the VMs
# # that will not be needed by the next smaller run.
# DELETE_UNUSED_AFTER_EACH_RUN = False

# # Delete the remaining 4 replica VMs + 1 client VM after the final run.
# DELETE_ALL_AFTER_FINAL_RUN = False

# # Code is unchanged across node-count runs, so compile only on the first run.
# COMPILE_ONLY_FIRST_RUN = True

# MAX_NUM_NODES = max(NUM_NODES_LIST)

# # Client experiment settings.
# CLIENT_DURATION_SEC = 360
# CLIENT_WAIT_AFTER_START_SEC = 180
# CLIENT_TOTAL_REQUESTS = 100000000
# SERVER_BATCH_SIZE_HINT = 100
# SEND_INTERVAL_US = 0

# # Fixed load point for throughput-vs-num_nodes and latency-vs-num_nodes.
# # Aggregate max in-flight = active_clients * client_threads * max_in_flight = 200.
# LOAD_POINTS = [
#     {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},   # total inflight 300
# ]


# for run_idx, num_nodes in enumerate(NUM_NODES_LIST):
# # for zone_no in  [0,1,2,3, 4]:

#     project = "research-488322"
#     zone = "us-central1-c"
#     machine_type = "e2-standard-2"
#     image_family = "tsm-sc-family"  # your custom image
#     subnet = "default"
#     gcp_username = "tejas"

#     print("\n" + "#" * 100)
#     print(f"Starting experiment for num_nodes={num_nodes}")
#     print("#" * 100)

#     def get_zone_for_instance(i):
#         # IMPORTANT: use MAX_NUM_NODES instead of current num_nodes.
#         # We reuse VMs while moving 48 -> 32 -> 16 -> 8 -> 4, so the zone for
#         # tsm-sc-016, tsm-sc-032, etc. must stay the same across runs.
#         if i < int(MAX_NUM_NODES / 2):
#             return default_region[0]
#         else:
#             return regions[zone_no]

#     def fetch_existing_instances():
#         fetch_cmd = f'''
#         gcloud compute instances list \
#             --project={project} \
#             --filter="name~'^tsm-sc-'" \
#             --format="value(name,zone)"
#         '''

#         output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
#         instances = []

#         for line in output.splitlines():
#             if line.strip():
#                 name, inst_zone = line.split()
#                 instances.append((name, inst_zone))

#         return instances

#     def delete_instance(instance):
#         name, inst_zone = instance

#         cmd = f'''
#         gcloud compute instances delete {name} \
#             --zone={inst_zone} \
#             --project={project} \
#             --quiet
#         '''

#         print(f"🗑️ Deleting {name} in {inst_zone}")
#         return subprocess.call(cmd, shell=True)

#     def parse_tsm_index(name):
#         return int(name.rsplit("-", 1)[1])

#     # -------------------------------------------------------------------------
#     # Delete existing tsm-sc-* instances only before the first/largest run.
#     # Later runs reuse tsm-sc-000 ... tsm-sc-(next_num_nodes).
#     # -------------------------------------------------------------------------
#     if DELETE_BEFORE_FIRST_RUN and run_idx == 0:
#         instances = fetch_existing_instances()

#         print("\n➡ Existing instances to delete before first run:")
#         for name, inst_zone in instances:
#             print(f"  - {name} ({inst_zone})")

#         if instances:
#             run_parallel(
#                 delete_instance,
#                 instances,
#                 max_workers=min(32, len(instances))
#             )
#             print("\n🧹 All existing tsm-sc-* instances deleted before first run.\n")
#         else:
#             print("\n✔ No existing tsm-sc-* instances found before first run.\n")

#     # -------------------------------------------------------------------------
#     # Create only the missing replica/client machines for this num_nodes run.
#     # When moving 48 -> 32 -> 16 -> 8 -> 4, the lower-index machines are reused.
#     # Replicas: tsm-sc-000 ... tsm-sc-(num_nodes-1)
#     # Client:   tsm-sc-num_nodes
#     # -------------------------------------------------------------------------
#     existing_instances = fetch_existing_instances()
#     existing_names = {name for name, _ in existing_instances}

#     commands = []
#     newly_created_indices = []

#     # Create replica nodes only if they do not already exist.
#     for i in range(num_nodes):
#         instance_name = f"tsm-sc-{i:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing replica {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(i)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(i)

#     # Create client machine only if it does not already exist.
#     for i in range(n_clients):
#         client_idx = num_nodes + i
#         instance_name = f"tsm-sc-{client_idx:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing client {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(client_idx)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(client_idx)

#     def run_command(command):
#         print(f"Running: {command}")
#         return subprocess.call(command, shell=True)

#     if commands:
#         run_parallel(
#             run_command,
#             commands,
#             max_workers=min(48, len(commands))
#         )

#         print("All missing instances launched.")

#         # Give GCP/SSH a little time after VM creation.
#         time.sleep(30)
#     else:
#         print("✔ All required instances already exist; no VM creation needed.")

#     # -------------------------------------------------------------------------
#     # Get sorted node and client IPs.
#     # IMPORTANT: tsm_ips.txt must include only server/replica IPs.
#     # Client IPs must not be included.
#     # -------------------------------------------------------------------------
#     ip_cmd = f'''
#     gcloud compute instances list \
#         --project={project} \
#         --filter="name~'^tsm-sc-'" \
#         --sort-by=name \
#         --format="value(name,zone,networkInterfaces[0].networkIP)"
#     '''

#     ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

#     instance_records = []

#     for line in ip_output.splitlines():
#         if line.strip():
#             name, inst_zone, ip = line.split()
#             idx = int(name.rsplit("-", 1)[1])
#             instance_records.append((idx, name, inst_zone, ip))

#     instance_records.sort()

#     node_records = [r for r in instance_records if r[0] < num_nodes]
#     client_records = [r for r in instance_records if num_nodes <= r[0] < num_nodes + n_clients]

#     if len(node_records) != num_nodes:
#         raise RuntimeError(
#             f"Expected {num_nodes} replica nodes, but found {len(node_records)}: {node_records}"
#         )

#     if len(client_records) != n_clients:
#         raise RuntimeError(
#             f"Expected {n_clients} client nodes, but found {len(client_records)}: {client_records}"
#         )

#     iplist = [r[3] for r in node_records]

#     with open("tsm_ips.txt", "w") as f:
#         for ip in iplist:
#             f.write(ip + "\n")

#     print("🎯 Node IPs:", iplist)
#     print("🎯 Client instances:", client_records)

#     node1_ip = iplist[0]
#     print(f"Clients will connect to leader/node1 at: {node1_ip}")

#     # -------------------------------------------------------------------------
#     # Push/pull/compile only for the first run.
#     # Later runs reuse the same lower-index VMs, and the code has not changed.
#     # -------------------------------------------------------------------------
#     def kill_stellar_private(i):
#         inst_zone = get_zone_for_instance(i)

#         remote_command = f"""\
# cd /home/tejas/stellar-private; \
# sudo pkill -9 stellar-core; sudo pkill -9 shab_client; \
# """

#         command = (
#             f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#             f'--project "{project}" --command "{remote_command}"'
#         )

#         print(f"Executing: {command}")
#         output = subprocess.call(command, shell=True)
#         print(f"Return code for tsm-sc-{i:03}: {output}")
#         return output

#     do_compile_this_run = (not COMPILE_ONLY_FIRST_RUN) or (run_idx == 0)

#     if do_compile_this_run:
#         subprocess.call('git add .; git commit -m "testing"; git push', shell=True)

#         n_collection = 100
#         subprocess.call('make -j8', shell=True)

#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def git_pull_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# git pull"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         results = run_parallel(
#             git_pull_stellar,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )
#         print(results)

#         # ---------------------------------------------------------------------
#         # Compile only on the first/largest run.
#         # Since later runs reuse a subset of these VMs, no recompilation is needed.
#         # ---------------------------------------------------------------------
#         def compile_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# g++ -O2 -std=c++17 -pthread \
# -I/home/tejas/stellar-core/src \
# /home/tejas/stellar-core/shab_client.cpp \
# -o /home/tejas/stellar-core/shab_client; \
# make -j16; \
# cd; \
# sudo rm -rf stellar-private"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         # results = run_parallel(
#         #     compile_stellar,
#         #     range(num_nodes + n_clients),
#         #     max_workers=min(48, num_nodes + n_clients)
#         # )


        
#         print(results)
#     else:
#         print(f"⏭️ Skipping git pull and compilation for num_nodes={num_nodes}; code already compiled on first run.")

#     # -------------------------------------------------------------------------
#     # Generate stellar-private configs locally using only replica IPs.
#     # -------------------------------------------------------------------------
#     stellar_private_path = Path('../stellar-private')
#     if stellar_private_path.exists():
#         shutil.rmtree(stellar_private_path)
#     stellar_private_path.mkdir()

#     subprocess.call(
#         'cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     subprocess.call(
#         'cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; '
#         './gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     # Enable custom message only on leader.
#     line_to_add = "SEND_CUSTOM_MESSAGE=true"
#     target_file = "../stellar-private/node1/stellar-core.cfg"

#     subprocess.call(
#         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#         shell=True
#     )

#     # Optional memory profiling on node2.
#     # if num_nodes >= 2:
#     #     target_file = "../stellar-private/node2/stellar-core.cfg"
#     #     line_to_add = "MEMORY_PROF=true"

#     #     subprocess.call(
#     #         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#     #         shell=True
#     #     )

#     #     print(f"The line '{line_to_add}' has been prepended to {target_file}.")

#     # -------------------------------------------------------------------------
#     # Throughput/latency experiment loop.
#     # Fixed offered load for scalability:
#     # active_clients=1, client_threads=2, max_in_flight=100.
#     # Aggregate max in-flight = 200.
#     # -------------------------------------------------------------------------
#     for load in LOAD_POINTS:
#         active_clients = load["active_clients"]
#         client_threads = load["client_threads"]
#         client_max_in_flight = load["max_in_flight"]

#         if active_clients > n_clients:
#             raise RuntimeError(
#                 f"active_clients={active_clients} exceeds n_clients={n_clients}"
#             )

#         total_client_threads = active_clients * client_threads
#         aggregate_max_in_flight = active_clients * client_threads * client_max_in_flight

#         run_label = (
#             f"clients_{active_clients}_threads_{client_threads}_"
#             f"inflight_{client_max_in_flight}_"
#             f"total_threads_{total_client_threads}_"
#             f"total_inflight_{aggregate_max_in_flight}"
#         )

#         print("\n" + "=" * 80)
#         print(f"🚀 Starting throughput/latency run: num_nodes={num_nodes}, {run_label}")
#         print("=" * 80)

#         def clean_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             remote_command = f"""\
# cd /home/tejas; \
# sudo rm -rf stellar-private; \
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")
#             output = subprocess.call(command, shell=True)
#             print(f"Return code for tsm-sc-{i:03}: {output}")
#             return output

#         results = run_parallel(
#             clean_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def copy_folder_to_instance(
#             i,
#             source_folder="/home/tejas/stellar-private",
#             destination_path="/home/tejas/stellar-private"
#         ):
#             inst_zone = get_zone_for_instance(i)
#             instance_name = f"tsm-sc-{i:03}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{source_folder}" "{instance_name}:{destination_path}"'''

#             print(f"Executing command for {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Command for {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         results = run_parallel(
#             copy_folder_to_instance,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def run_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             node_number = i + 1
#             instance_name = f"tsm-sc-{i:03}"

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Kill old processes on all replica and client machines.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         # Start consensus replicas.
#         results = run_parallel(
#             run_stellar_private,
#             range(num_nodes),
#             max_workers=min(48, num_nodes)
#         )

#         print(results)
#         print("All Stellar nodes should be starting in the background.")

#         # Give nodes time to authenticate and start the client listener.
#         time.sleep(80)

#         def run_stellar_client(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
# {client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
# {CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
# > stellar-client-{client_id}.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing client command: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Start only the required number of client VMs for this load point.
#         active_client_indices = [
#             num_nodes + j for j in range(active_clients)
#         ]

#         results = run_parallel(
#             run_stellar_client,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print(results)
#         print(
#             f"Started {active_clients} client VM(s), "
#             f"each with {client_threads} client threads. "
#             f"Total client threads = {total_client_threads}. "
#             f"Aggregate max in-flight = {aggregate_max_in_flight}."
#         )

#         # Wait for the duration run to produce stable per-second client logs.
#         # The clients may wait forever on final partial batches, so we kill them after this.
#         # time.sleep(CLIENT_WAIT_AFTER_START_SEC)

#         time.sleep(100)

#         results = run_parallel(
#             kill_stellar_private,
#             [4],
#             max_workers=min(48, num_nodes + n_clients)
#         )

        
#         time.sleep(20)

        


#         results = run_parallel(
#             kill_stellar_private,
#             [5],
#             max_workers=min(48, num_nodes + n_clients)
#         )

        
#         time.sleep(20)


#         results = run_parallel(
#             kill_stellar_private,
#             [6],
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         time.sleep(20)
        
#         results = run_parallel(
#             kill_stellar_private,
#             [7],
#             max_workers=min(48, num_nodes + n_clients)
#         )
        
#         time.sleep(20)
        
#         results = run_parallel(
#             kill_stellar_private,
#             [8],
#             max_workers=min(48, num_nodes + n_clients)
#         )
        
#         time.sleep(80)
        

#         # Stop all nodes and clients.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         remote_base_folder = "/home/tejas/stellar-private"

#         local_base_destination = (
#             "/home/tejas/work/experiments/shabdiz/"
#             + f"Failure_{num_nodes}_{run_label}"
#         )

#         Path(local_base_destination).mkdir(parents=True, exist_ok=True)

#         # Save run metadata.
#         with open(Path(local_base_destination) / "run_config.txt", "w") as f:
#             f.write(f"num_nodes={num_nodes}\n")
#             f.write(f"active_clients={active_clients}\n")
#             f.write(f"client_threads_per_vm={client_threads}\n")
#             f.write(f"total_client_threads={total_client_threads}\n")
#             f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
#             f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
#             f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
#             f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
#             f.write(f"leader_ip={node1_ip}\n")
#             f.write(f"machine_type={machine_type}\n")
#             f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

#         def copy_folder_from_instance(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"

#             node_number = i + 1
#             node_folder = f"node{node_number}"

#             remote_source_path = posixpath.join(remote_base_folder, node_folder)

#             local_destination_path = Path(local_base_destination) / instance_name
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             remote_source = f"{instance_name}:{remote_source_path}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{remote_source}" "{local_destination_path}"'''

#             print(f"Executing command to copy {node_folder} from {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy from {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         # Copy only a few node logs to reduce time.
#         # Change range(min(3, num_nodes)) to range(num_nodes) if you want all logs.
#         node_copy_results = run_parallel(
#             copy_folder_from_instance,
#             range((num_nodes)),
#             max_workers=min(48, max(1, min(3, num_nodes)))
#         )

#         def copy_client_log(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_source = (
#                 f"{instance_name}:/home/tejas/stellar-private/"
#                 f"stellar-client-{client_id}.log"
#             )

#             local_destination_path = Path(local_base_destination)
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

#             print(f"Copying client log from {instance_name}...")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy finished with exit code: {output}")

#             return (instance_name, output)

#         client_copy_results = run_parallel(
#             copy_client_log,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print("\n--- Summary of Download Results ---")
#         print("Node log copies:", node_copy_results)
#         print("Client log copies:", client_copy_results)
#         print(f"Saved run to: {local_base_destination}")

#     # -------------------------------------------------------------------------
#     # After this run, delete only the machines that the next smaller run will
#     # not need. Example:
#     #   after 48-node run, keep 000..032 and delete 033..048
#     #   after 32-node run, keep 000..016 and delete 017..032
#     #   after 16-node run, keep 000..008 and delete 009..016
#     #   after 8-node run,  keep 000..004 and delete 005..008
#     # Final run optionally deletes everything.
#     # -------------------------------------------------------------------------
#     if DELETE_UNUSED_AFTER_EACH_RUN:
#         instances = fetch_existing_instances()

#         if run_idx + 1 < len(NUM_NODES_LIST):
#             next_num_nodes = NUM_NODES_LIST[run_idx + 1]
#             keep_count = next_num_nodes + n_clients

#             instances_to_delete = [
#                 (name, inst_zone)
#                 for name, inst_zone in instances
#                 if parse_tsm_index(name) >= keep_count
#             ]

#             print(
#                 f"\n➡ After num_nodes={num_nodes}, next run needs "
#                 f"indices 0..{keep_count - 1}. Deleting higher-index VMs:"
#             )
#         elif DELETE_ALL_AFTER_FINAL_RUN:
#             instances_to_delete = instances

#             print("\n➡ Final run complete. Deleting all remaining tsm-sc-* VMs:")
#         else:
#             instances_to_delete = []

#             print("\n✔ Final run complete. Leaving remaining tsm-sc-* VMs running.")

#         for name, inst_zone in instances_to_delete:
#             print(f"  - {name} ({inst_zone})")

#         if instances_to_delete:
#             run_parallel(
#                 delete_instance,
#                 instances_to_delete,
#                 max_workers=min(32, len(instances_to_delete))
#             )
#             print(f"\n🧹 Deleted {len(instances_to_delete)} unneeded instance(s) after num_nodes={num_nodes}.\n")
#         else:
#             print("\n✔ No unneeded instances to delete after this run.\n")



####################################################################################################
Starting experiment for num_nodes=16
####################################################################################################
✔ Reusing existing replica tsm-sc-000
✔ Reusing existing replica tsm-sc-001
✔ Reusing existing replica tsm-sc-002
✔ Reusing existing replica tsm-sc-003
✔ Reusing existing replica tsm-sc-004
✔ Reusing existing replica tsm-sc-005
✔ Reusing existing replica tsm-sc-006
✔ Reusing existing replica tsm-sc-007
✔ Reusing existing replica tsm-sc-008
✔ Reusing existing replica tsm-sc-009
✔ Reusing existing replica tsm-sc-010
✔ Reusing existing replica tsm-sc-011
✔ Reusing existing replica tsm-sc-012
✔ Reusing existing replica tsm-sc-013
✔ Reusing existing replica tsm-sc-014
✔ Reusing existing replica tsm-sc-015
✔ Reusing existing client tsm-sc-016
✔ All required instances already exist; no VM creation needed.
🎯 Node IPs: ['10.128.0.108', '10.128.0.109', '10.128

To github.com:tejas-shivanand-mane/stellar-core.git
   a58b4b8..61d602b  main -> main


make  all-recursive
make[1]: Entering directory '/home/tejas/stellar-core'
Making all in lib
make[2]: Entering directory '/home/tejas/stellar-core/lib'
Making all in ../lib/libsodium
make[3]: Entering directory '/home/tejas/stellar-core/lib/libsodium'
Making all in builds
make[4]: Entering directory '/home/tejas/stellar-core/lib/libsodium/builds'
make[4]: Nothing to be done for 'all'.
make[4]: Leaving directory '/home/tejas/stellar-core/lib/libsodium/builds'
Making all in contrib
make[4]: Entering directory '/home/tejas/stellar-core/lib/libsodium/contrib'
make[4]: Nothing to be done for 'all'.
make[4]: Leaving directory '/home/tejas/stellar-core/lib/libsodium/contrib'
Making all in dist-build
make[4]: Entering directory '/home/tejas/stellar-core/lib/libsodium/dist-build'
make[4]: Nothing to be done for 'all'.
make[4]: Leaving directory '/home/tejas/stellar-core/lib/libsodium/dist-build'
Making all in msvc-scripts
make[4]: Entering directory '/home/tejas/stellar-core/lib/libsodium/msvc-

From https://github.com/tejas-shivanand-mane/stellar-core
   a58b4b8..61d602b  main       -> origin/main
From https://github.com/tejas-shivanand-mane/stellar-core
   a58b4b8..61d602b  main       -> origin/main
From https://github.com/tejas-shivanand-mane/stellar-core
   a58b4b8..61d602b  main       -> origin/main
From https://github.com/tejas-shivanand-mane/stellar-core
   a58b4b8..61d602b  main       -> origin/main
From https://github.com/tejas-shivanand-mane/stellar-core
   a58b4b8..61d602b  main       -> origin/main


Updating a58b4b8..61d602b
Fast-forward
Updating a58b4b8..61d602b
Fast-forward
 PostProcess.ipynb |   48 +-
 RunGCP.ipynb      | 4327 +++++++++--------------------------------------------
 2 files changed, 765 insertions(+), 3610 deletions(-)
 PostProcess.ipynb |   48 +-
 RunGCP.ipynb      | 4327 +++++++++--------------------------------------------
 2 files changed, 765 insertions(+), 3610 deletions(-)
Updating a58b4b8..61d602b
Fast-forward
Updating a58b4b8..61d602b
Fast-forward
Updating a58b4b8..61d602b
Fast-forward
 PostProcess.ipynb |   48 +-
 RunGCP.ipynb      | 4327 +++++++++--------------------------------------------
 2 files changed, 765 insertions(+), 3610 deletions(-)
 PostProcess.ipynb |   48 +-
 RunGCP.ipynb      | 4327 +++++++++--------------------------------------------
 2 files changed, 765 insertions(+), 3610 deletions(-)
 PostProcess.ipynb |   48 +-
 RunGCP.ipynb      | 4327 +++++++++--------------------------------------------
 2 files changed, 765 insertions(+), 361

...

Initializing database for node2...
Initializing database for node3...
Initializing database for node4...
Initializing database for node5...
Initializing database for node6...
Initializing database for node7...
Initializing database for node8...


2026-06-23T01:01:07.426 [default INFO] Config from /home/tejas/stellar-private/node6/stellar-core.cfg
2026-06-23T01:01:07.429 [default INFO] Generated QUORUM_SET: {
   "t" : 9,
   "v" : [
      "node14",
      "node1",
      "GAPE6",
      "node9",
      "node2",
      "node12",
      "node3",
      "node5",
      "node8",
      "node7",
      "node13",
      "node10",
      "node16",
      "node11",
      "node15",
      "node4"
   ]
}

2026-06-23T01:01:07.429 [default WARNING] Adjusted TARGET_PEER_CONNECTIONS to 336 due to insufficient MAX_ADDITIONAL_PEER_CONNECTIONS=1000
2026-06-23T01:01:07.429 [default INFO] Assigning calculated value of 7 to FAILURE_SAFETY
2026-06-23T01:01:07.461 [default INFO] Config from /home/tejas/stellar-private/node7/stellar-core.cfg
2026-06-23T01:01:07.463 [default INFO] Generated QUORUM_SET: {
   "t" : 9,
   "v" : [
      "node14",
      "node1",
      "node6",
      "node9",
      "node2",
      "node12",
      "node3",
      "node5",
      "node8",
     

Initializing database for node9...
Initializing database for node10...
Initializing database for node11...
Initializing database for node12...
Initializing database for node13...
Initializing database for node14...
Initializing database for node15...


2026-06-23T01:01:07.657 [default INFO] Config from /home/tejas/stellar-private/node13/stellar-core.cfg
2026-06-23T01:01:07.660 [default INFO] Generated QUORUM_SET: {
   "t" : 9,
   "v" : [
      "node14",
      "node1",
      "node6",
      "node9",
      "node2",
      "node12",
      "node3",
      "node5",
      "node8",
      "node7",
      "GBRMK",
      "node10",
      "node16",
      "node11",
      "node15",
      "node4"
   ]
}

2026-06-23T01:01:07.660 [default WARNING] Adjusted TARGET_PEER_CONNECTIONS to 336 due to insufficient MAX_ADDITIONAL_PEER_CONNECTIONS=1000
2026-06-23T01:01:07.660 [default INFO] Assigning calculated value of 7 to FAILURE_SAFETY
2026-06-23T01:01:07.689 [default INFO] Config from /home/tejas/stellar-private/node14/stellar-core.cfg
2026-06-23T01:01:07.692 [default INFO] Generated QUORUM_SET: {
   "t" : 9,
   "v" : [
      "GAJTT",
      "node1",
      "node6",
      "node9",
      "node2",
      "node12",
      "node3",
      "node5",
      "node8",
     

Initializing database for node16...
✅ 16-node private Stellar network setup complete!
Start the nodes with (substituting node number):
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node1/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node2/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node3/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node4/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node5/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node6/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node7/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node8/stellar-core.cfg &
/home/tejas/stell

# Collection vs Collection Rounds Experiment

In [50]:
import subprocess
import concurrent.futures
import posixpath
import shutil
import time
import re
from pathlib import Path


# ============================================================
# Helpers
# ============================================================

def run_shell(command):
    return subprocess.call(command, shell=True)


def run_parallel(func, iterable, max_workers):
    items = list(iterable)
    if not items:
        return []

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        return list(executor.map(func, items))


def run_command(command):
    print(f"Running: {command}")
    return subprocess.call(command, shell=True)


def fetch_existing_instances(project):
    fetch_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --format="value(name,zone)"
    '''

    output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
    instances = []

    for line in output.splitlines():
        if line.strip():
            name, inst_zone = line.split()
            instances.append((name, inst_zone))

    return instances


def delete_instance(project, instance):
    name, inst_zone = instance

    cmd = f'''
    gcloud compute instances delete {name} \
        --zone={inst_zone} \
        --project={project} \
        --quiet
    '''

    print(f"🗑️ Deleting {name} in {inst_zone}")
    return subprocess.call(cmd, shell=True)


def delete_all_tsm_instances(project):
    instances = fetch_existing_instances(project)

    print("\n➡ Existing instances to delete:")
    for name, inst_zone in instances:
        print(f"  - {name} ({inst_zone})")

    if instances:
        run_parallel(
            lambda instance: delete_instance(project, instance),
            instances,
            max_workers=min(32, len(instances)),
        )
        print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
    else:
        print("\n✔ No tsm-sc-* instances found.\n")


# ============================================================
# Experiment config
# ============================================================

# latencies: 50, 90, 150, 210
default_region = ["us-west1-b"]
# regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']
regions = ["us-west1-b", "us-west1-b", "us-west1-b", "us-west1-b"]

zone_no = 0

# System size for this failure / collection-window experiment.
NUM_NODES_LIST = [16]

# Use 1 extra 2-core machine as client machine.
# For num_nodes = N, client is tsm-sc-N.
n_clients = 1

# Collection-round sweep.
COLLECT_ATTEMPT_POINTS = [5]

# Source file to patch before each subrun.
OVERLAY_MANAGER_IMPL_PATH = Path("src/overlay/OverlayManagerImpl.cpp")

# Start clean only once, before the first run.
DELETE_BEFORE_FIRST_RUN = True

# Delete VMs after each MAX_COLLECT_ATTEMPTS subrun.
DELETE_UNUSED_AFTER_EACH_RUN = True

# Delete remaining VMs after final run.
# This is mostly redundant if DELETE_UNUSED_AFTER_EACH_RUN=True.
DELETE_ALL_AFTER_FINAL_RUN = True

# IMPORTANT:
# Code changes across collection-round settings, so we must compile every subrun.
COMPILE_ONLY_FIRST_RUN = False

MAX_NUM_NODES = max(NUM_NODES_LIST)

# Client experiment settings.
CLIENT_DURATION_SEC = 360
CLIENT_WAIT_AFTER_START_SEC = 210
CLIENT_TOTAL_REQUESTS = 100000000
SERVER_BATCH_SIZE_HINT = 100
SEND_INTERVAL_US = 0

# Fixed offered load.
LOAD_POINTS = [
    {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},
]


# ============================================================
# Local source patching
# ============================================================

def patch_max_collect_attempts(max_collect_attempts):
    text = OVERLAY_MANAGER_IMPL_PATH.read_text()

    pattern = r"static\s+constexpr\s+uint64_t\s+MAX_COLLECT_ATTEMPTS\s*=\s*\d+\s*;"
    replacement = (
        f"static constexpr uint64_t MAX_COLLECT_ATTEMPTS = {max_collect_attempts};"
    )

    new_text, count = re.subn(
        pattern,
        replacement,
        text,
        flags=re.MULTILINE,
    )

    if count != 1:
        raise RuntimeError(
            f"Expected to replace exactly one MAX_COLLECT_ATTEMPTS line in "
            f"{OVERLAY_MANAGER_IMPL_PATH}, but replaced {count}"
        )

    OVERLAY_MANAGER_IMPL_PATH.write_text(new_text)

    print(
        f"Updated {OVERLAY_MANAGER_IMPL_PATH}: "
        f"MAX_COLLECT_ATTEMPTS = {max_collect_attempts}"
    )


def push_collect_patch_to_git(max_collect_attempts):
    cmd = (
        f'git add {OVERLAY_MANAGER_IMPL_PATH}; '
        f'git commit -m "set collect attempts {max_collect_attempts}" || true; '
        f'git push'
    )

    return subprocess.call(cmd, shell=True)


# ============================================================
# Main experiment loop
# ============================================================

for run_idx, num_nodes in enumerate(NUM_NODES_LIST):
    for collect_idx, max_collect_attempts in enumerate(COLLECT_ATTEMPT_POINTS):

        project = "research-488322"
        zone = "us-west1-b"
        machine_type = "e2-standard-2"
        image_family = "tsm-sc-family"
        subnet = "default"
        gcp_username = "tejas"

        print("\n" + "#" * 100)
        print(
            f"Starting experiment for num_nodes={num_nodes}, "
            f"MAX_COLLECT_ATTEMPTS={max_collect_attempts}"
        )
        print("#" * 100)

        def get_zone_for_instance(i):
            # Use MAX_NUM_NODES so zone assignment stays stable.
            if i < int(MAX_NUM_NODES / 2):
                return default_region[0]
            else:
                return regions[zone_no]

        def parse_tsm_index(name):
            return int(name.rsplit("-", 1)[1])

        # -----------------------------------------------------------------
        # Delete existing tsm-sc-* instances only before the first subrun,
        # if enabled.
        # -----------------------------------------------------------------
        if DELETE_BEFORE_FIRST_RUN and run_idx == 0 and collect_idx == 0:
            print("\n🧹 Cleaning before first run...")
            delete_all_tsm_instances(project)

        # -----------------------------------------------------------------
        # Create only missing replica/client machines for this subrun.
        # -----------------------------------------------------------------
        existing_instances = fetch_existing_instances(project)
        existing_names = {name for name, _ in existing_instances}

        commands = []
        newly_created_indices = []

        # Create replica nodes only if they do not already exist.
        for i in range(num_nodes):
            instance_name = f"tsm-sc-{i:03}"

            if instance_name in existing_names:
                print(f"✔ Reusing existing replica {instance_name}")
                continue

            inst_zone = get_zone_for_instance(i)

            cmd = f'''
            gcloud compute instances create {instance_name} \
                --project={project} \
                --zone={inst_zone} \
                --machine-type={machine_type} \
                --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
                --can-ip-forward \
                --maintenance-policy=MIGRATE \
                --provisioning-model=STANDARD \
                --service-account=254510644191-compute@developer.gserviceaccount.com \
                --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
                --tags=http-server,https-server \
                --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
                --no-shielded-secure-boot \
                --shielded-vtpm \
                --shielded-integrity-monitoring \
                --labels=goog-ec-src=vm_add-gcloud \
                --reservation-affinity=any
            '''
            commands.append(cmd.strip())
            newly_created_indices.append(i)

        # Create client machine only if it does not already exist.
        for i in range(n_clients):
            client_idx = num_nodes + i
            instance_name = f"tsm-sc-{client_idx:03}"

            if instance_name in existing_names:
                print(f"✔ Reusing existing client {instance_name}")
                continue

            inst_zone = get_zone_for_instance(client_idx)

            cmd = f'''
            gcloud compute instances create {instance_name} \
                --project={project} \
                --zone={inst_zone} \
                --machine-type={machine_type} \
                --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
                --can-ip-forward \
                --maintenance-policy=MIGRATE \
                --provisioning-model=STANDARD \
                --service-account=254510644191-compute@developer.gserviceaccount.com \
                --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
                --tags=http-server,https-server \
                --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
                --no-shielded-secure-boot \
                --shielded-vtpm \
                --shielded-integrity-monitoring \
                --labels=goog-ec-src=vm_add-gcloud \
                --reservation-affinity=any
            '''
            commands.append(cmd.strip())
            newly_created_indices.append(client_idx)

        if commands:
            run_parallel(
                run_command,
                commands,
                max_workers=min(48, len(commands)),
            )

            print("All missing instances launched.")

            # Give GCP/SSH a little time after VM creation.
            time.sleep(30)
        else:
            print("✔ All required instances already exist; no VM creation needed.")

        # -----------------------------------------------------------------
        # Get sorted node and client IPs.
        # tsm_ips.txt must include only replica IPs, not clients.
        # -----------------------------------------------------------------
        ip_cmd = f'''
        gcloud compute instances list \
            --project={project} \
            --filter="name~'^tsm-sc-'" \
            --sort-by=name \
            --format="value(name,zone,networkInterfaces[0].networkIP)"
        '''

        ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

        instance_records = []

        for line in ip_output.splitlines():
            if line.strip():
                name, inst_zone, ip = line.split()
                idx = int(name.rsplit("-", 1)[1])
                instance_records.append((idx, name, inst_zone, ip))

        instance_records.sort()

        node_records = [r for r in instance_records if r[0] < num_nodes]
        client_records = [
            r for r in instance_records
            if num_nodes <= r[0] < num_nodes + n_clients
        ]

        if len(node_records) != num_nodes:
            raise RuntimeError(
                f"Expected {num_nodes} replica nodes, but found "
                f"{len(node_records)}: {node_records}"
            )

        if len(client_records) != n_clients:
            raise RuntimeError(
                f"Expected {n_clients} client nodes, but found "
                f"{len(client_records)}: {client_records}"
            )

        iplist = [r[3] for r in node_records]

        with open("tsm_ips.txt", "w") as f:
            for ip in iplist:
                f.write(ip + "\n")

        print("🎯 Node IPs:", iplist)
        print("🎯 Client instances:", client_records)

        node1_ip = iplist[0]
        print(f"Clients will connect to leader/node1 at: {node1_ip}")

        # -----------------------------------------------------------------
        # Remote helper functions.
        # -----------------------------------------------------------------

        def kill_stellar_private(i):
            inst_zone = get_zone_for_instance(i)

            remote_command = """\
sudo pkill -9 stellar-core || true; \
sudo pkill -9 shab_client || true; \
"""

            command = (
                f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
                f'--project "{project}" --command "{remote_command}"'
            )

            print(f"Executing: {command}")
            output = subprocess.call(command, shell=True)
            print(f"Return code for tsm-sc-{i:03}: {output}")
            return output

        def git_pull_stellar(i):
            inst_zone = get_zone_for_instance(i)

            command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
git pull"'''

            print(command)
            output = subprocess.call(command, shell=True)
            print(output)
            return output

        def compile_stellar(i):
            inst_zone = get_zone_for_instance(i)

            command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
g++ -O2 -std=c++17 -pthread \
-I/home/tejas/stellar-core/src \
/home/tejas/stellar-core/shab_client.cpp \
-o /home/tejas/stellar-core/shab_client; \
make -j4; \
cd; \
sudo rm -rf stellar-private"'''

            print(command)
            output = subprocess.call(command, shell=True)
            print(output)
            return output

        def clean_stellar_private(i):
            inst_zone = get_zone_for_instance(i)

            remote_command = """\
cd /home/tejas; \
sudo rm -rf stellar-private; \
"""

            command = (
                f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
                f'--project "{project}" --command "{remote_command}"'
            )

            print(f"Executing: {command}")
            output = subprocess.call(command, shell=True)
            print(f"Return code for tsm-sc-{i:03}: {output}")
            return output

        def copy_folder_to_instance(
            i,
            source_folder="/home/tejas/stellar-private",
            destination_path="/home/tejas/stellar-private",
        ):
            inst_zone = get_zone_for_instance(i)
            instance_name = f"tsm-sc-{i:03}"

            command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
--recurse "{source_folder}" "{instance_name}:{destination_path}"'''

            print(f"Executing command for {instance_name}: {command}")

            output = subprocess.call(command, shell=True)

            print(f"Command for {instance_name} finished with exit code: {output}")

            return (instance_name, output)

        def run_stellar_private(i):
            inst_zone = get_zone_for_instance(i)

            node_number = i + 1
            instance_name = f"tsm-sc-{i:03}"

            remote_command = f"""\
cd /home/tejas/stellar-private; \
nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
> node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
"""

            command = (
                f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
                f'--project "{project}" --command "{remote_command}"'
            )

            print(f"Executing: {command}")

            output = subprocess.call(command, shell=True)

            print(f"Return code for {instance_name}: {output}")
            return output

        def run_stellar_client(i, client_max_in_flight, client_threads):
            inst_zone = get_zone_for_instance(i)

            instance_name = f"tsm-sc-{i:03}"
            client_id = i - num_nodes

            remote_command = f"""\
cd /home/tejas/stellar-private; \
nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
{client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
{CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
> stellar-client-{client_id}.log 2>&1 < /dev/null & disown
"""

            command = (
                f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
                f'--project "{project}" --command "{remote_command}"'
            )

            print(f"Executing client command: {command}")

            output = subprocess.call(command, shell=True)

            print(f"Return code for {instance_name}: {output}")
            return output

        # -----------------------------------------------------------------
        # Generate stellar-private configs locally using only replica IPs.
        # -----------------------------------------------------------------
        stellar_private_path = Path("../stellar-private")
        if stellar_private_path.exists():
            shutil.rmtree(stellar_private_path)
        stellar_private_path.mkdir()

        subprocess.call(
            "cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh",
            shell=True,
        )

        subprocess.call(
            "cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; "
            "./gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh",
            shell=True,
        )

        # Enable custom message only on leader.
        line_to_add = "SEND_CUSTOM_MESSAGE=true"
        target_file = "../stellar-private/node1/stellar-core.cfg"

        subprocess.call(
            f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
            shell=True,
        )

        print("\n" + "*" * 100)
        print(f"Preparing subrun with MAX_COLLECT_ATTEMPTS={max_collect_attempts}")
        print("*" * 100)

        # Patch local source.
        patch_max_collect_attempts(max_collect_attempts)

        # Push patched source.
        push_collect_patch_to_git(max_collect_attempts)

        # Stop any old processes.
        results = run_parallel(
            kill_stellar_private,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

        # Pull patched code on active VMs.
        results = run_parallel(
            git_pull_stellar,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

        # Compile patched code on active VMs.
        results = run_parallel(
            compile_stellar,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

        # -----------------------------------------------------------------
        # Throughput/latency loop.
        #
        # IMPORTANT:
        # This is now inside the MAX_COLLECT_ATTEMPTS loop.
        # So every collect-attempt value gets its own actual run.
        # -----------------------------------------------------------------
        for load in LOAD_POINTS:
            active_clients = load["active_clients"]
            client_threads = load["client_threads"]
            client_max_in_flight = load["max_in_flight"]

            if active_clients > n_clients:
                raise RuntimeError(
                    f"active_clients={active_clients} exceeds n_clients={n_clients}"
                )

            total_client_threads = active_clients * client_threads
            aggregate_max_in_flight = (
                active_clients * client_threads * client_max_in_flight
            )

            run_label = (
                f"clients_{active_clients}_threads_{client_threads}_"
                f"inflight_{client_max_in_flight}_"
                f"total_threads_{total_client_threads}_"
                f"total_inflight_{aggregate_max_in_flight}"
            )

            print("\n" + "=" * 80)
            print(
                f"🚀 Starting run: num_nodes={num_nodes}, "
                f"MAX_COLLECT_ATTEMPTS={max_collect_attempts}, {run_label}"
            )
            print("=" * 80)

            # Clean remote stellar-private.
            results = run_parallel(
                clean_stellar_private,
                range(num_nodes + n_clients),
                max_workers=min(48, num_nodes + n_clients),
            )
            print(results)

            # Copy fresh stellar-private config to all active machines.
            results = run_parallel(
                copy_folder_to_instance,
                range(num_nodes + n_clients),
                max_workers=min(48, num_nodes + n_clients),
            )
            print(results)

            # Kill old processes on all active machines.
            results = run_parallel(
                kill_stellar_private,
                range(num_nodes + n_clients),
                max_workers=min(48, num_nodes + n_clients),
            )
            print(results)

            # Start consensus replicas.
            results = run_parallel(
                run_stellar_private,
                range(num_nodes),
                max_workers=min(48, num_nodes),
            )

            print(results)
            print("All Stellar nodes should be starting in the background.")

            # Give nodes time to authenticate and start the client listener.
            time.sleep(40)

            # Start only required client VMs for this load point.
            active_client_indices = [
                num_nodes + j for j in range(active_clients)
            ]

            results = run_parallel(
                lambda i: run_stellar_client(i, client_max_in_flight, client_threads),
                active_client_indices,
                max_workers=active_clients,
            )

            print(results)
            print(
                f"Started {active_clients} client VM(s), "
                f"each with {client_threads} client threads. "
                f"Total client threads = {total_client_threads}. "
                f"Aggregate max in-flight = {aggregate_max_in_flight}."
            )

            # Let the system run.
            time.sleep(CLIENT_WAIT_AFTER_START_SEC)

            # Stop all nodes and clients.
            results = run_parallel(
                kill_stellar_private,
                range(num_nodes + n_clients),
                max_workers=min(48, num_nodes + n_clients),
            )
            print(results)

            # -------------------------------------------------------------
            # Save logs.
            # -------------------------------------------------------------
            remote_base_folder = "/home/tejas/stellar-private"

            local_base_destination = (
                "/home/tejas/work/experiments/shabdiz/"
                + f"Collection_{max_collect_attempts}_"
                + f"nodes_{num_nodes}_{run_label}"
            )

            Path(local_base_destination).mkdir(parents=True, exist_ok=True)

            # Save run metadata.
            with open(Path(local_base_destination) / "run_config.txt", "w") as f:
                f.write(f"num_nodes={num_nodes}\n")
                f.write(f"max_collect_attempts={max_collect_attempts}\n")
                f.write(f"force_collect_after_sec=100\n")
                f.write(f"active_clients={active_clients}\n")
                f.write(f"client_threads_per_vm={client_threads}\n")
                f.write(f"total_client_threads={total_client_threads}\n")
                f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
                f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
                f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
                f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
                f.write(f"leader_ip={node1_ip}\n")
                f.write(f"machine_type={machine_type}\n")
                f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

            def copy_folder_from_instance(i):
                inst_zone = get_zone_for_instance(i)

                instance_name = f"tsm-sc-{i:03}"

                node_number = i + 1
                node_folder = f"node{node_number}"

                remote_source_path = posixpath.join(remote_base_folder, node_folder)

                local_destination_path = Path(local_base_destination) / instance_name
                local_destination_path.mkdir(parents=True, exist_ok=True)

                remote_source = f"{instance_name}:{remote_source_path}"

                command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
--recurse "{remote_source}" "{local_destination_path}"'''

                print(
                    f"Executing command to copy {node_folder} "
                    f"from {instance_name}: {command}"
                )

                output = subprocess.call(command, shell=True)

                print(f"Copy from {instance_name} finished with exit code: {output}")

                return (instance_name, output)

            # Copy all replica logs.
            # FIX: was range(3), which copied only nodes 0, 1, 2.
            node_copy_results = run_parallel(
                copy_folder_from_instance,
                range(3),
                max_workers=3,
            )

            def copy_client_log(i):
                inst_zone = get_zone_for_instance(i)

                instance_name = f"tsm-sc-{i:03}"
                client_id = i - num_nodes

                remote_source = (
                    f"{instance_name}:/home/tejas/stellar-private/"
                    f"stellar-client-{client_id}.log"
                )

                local_destination_path = Path(local_base_destination)
                local_destination_path.mkdir(parents=True, exist_ok=True)

                command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
"{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

                print(f"Copying client log from {instance_name}...")

                output = subprocess.call(command, shell=True)

                print(f"Copy finished with exit code: {output}")

                return (instance_name, output)

            client_copy_results = run_parallel(
                copy_client_log,
                active_client_indices,
                max_workers=active_clients,
            )

            print("\n--- Summary of Download Results ---")
            print("Node log copies:", node_copy_results)
            print("Client log copies:", client_copy_results)
            print(f"Saved run to: {local_base_destination}")

        # -----------------------------------------------------------------
        # Delete instances after this MAX_COLLECT_ATTEMPTS subrun.
        #
        # IMPORTANT:
        # This is inside the collect-attempt loop, so after Collection_50
        # finishes, VMs are deleted before Collection_300 starts.
        # -----------------------------------------------------------------
        if DELETE_UNUSED_AFTER_EACH_RUN:
            print(
                "\n🧹 Deleting VMs after completed subrun: "
                f"num_nodes={num_nodes}, MAX_COLLECT_ATTEMPTS={max_collect_attempts}"
            )
            delete_all_tsm_instances(project)


# ============================================================
# Final cleanup
# ============================================================

if DELETE_ALL_AFTER_FINAL_RUN and not DELETE_UNUSED_AFTER_EACH_RUN:
    print("\n🧹 Final cleanup after all runs...")
    delete_all_tsm_instances(project)


####################################################################################################
Starting experiment for num_nodes=16, MAX_COLLECT_ATTEMPTS=5
####################################################################################################

🧹 Cleaning before first run...



➡ Existing instances to delete:

✔ No tsm-sc-* instances found.



Running: gcloud compute instances create tsm-sc-000                 --project=research-488322                 --zone=us-west1-b                 --machine-type=e2-standard-2                 --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default                 --can-ip-forward                 --maintenance-policy=MIGRATE                 --provisioning-model=STANDARD                 --service-account=254510644191-compute@developer.gserviceaccount.com                 --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append                 --tags=http-server,https-server                 --create-disk=auto-delete=yes,boot=yes,image-family=tsm-sc-family,mode=rw,size=20,type=pd-balanced                 --no-shielded-secure-

Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-015].
Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-003].
 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert

 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert

Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-005].
Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-010].


NAME        ZONE        MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP    STATUS
tsm-sc-015  us-west1-b  e2-standard-2               10.138.0.17  136.66.39.157  RUNNING
NAME        ZONE        MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP     STATUS
tsm-sc-003  us-west1-b  e2-standard-2               10.138.0.5   136.66.221.231  RUNNING


...

ssh: connect to host 8.229.55.79 port 22: Connection refused

Recommendation: To check for possible causes of SSH connectivity issues and get
recommendations, rerun the ssh command with the --troubleshoot option.

gcloud compute ssh tsm-sc-001 --project=research-488322 --zone=us-west1-b --troubleshoot

Or, to investigate an IAP tunneling issue:

gcloud compute ssh tsm-sc-001 --project=research-488322 --zone=us-west1-b --troubleshoot --tunnel-through-iap

ERROR: (gcloud.compute.ssh) [/usr/bin/ssh] exited with return code [255].


Return code for tsm-sc-001: 255
Return code for tsm-sc-016: 0
Return code for tsm-sc-007: 0
Return code for tsm-sc-002: 0
Return code for tsm-sc-004: 0
Return code for tsm-sc-003: 0
Return code for tsm-sc-010: 0
Return code for tsm-sc-015: 0
Return code for tsm-sc-012: 0
Return code for tsm-sc-011: 0
Return code for tsm-sc-013: 0
Return code for tsm-sc-009: 0
Return code for tsm-sc-006: 0
Return code for tsm-sc-014: 0
Return code for tsm-sc-000: 0
Return code for tsm-sc-008: 0
Return code for tsm-sc-005: 0
[0, 255, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Executing command to copy node1 from tsm-sc-000: gcloud compute scp --zone "us-west1-b" --project "research-488322" --recurse "tsm-sc-000:/home/tejas/stellar-private/node1" "/home/tejas/work/experiments/shabdiz/Collection_5_nodes_16_clients_1_threads_2_inflight_150_total_threads_2_total_inflight_300/tsm-sc-000"
Executing command to copy node2 from tsm-sc-001: gcloud compute scp --zone "us-west1-b" --project "research-488322" --rec

Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-007].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-003].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-004].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-005].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-002].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-006].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-009].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-008].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-s


🧹 All tsm-sc-* instances deleted across all regions.



Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-001].


# Collection vs num nodes

In [48]:
import subprocess
import concurrent.futures
import posixpath
import shutil
import time
import re
from pathlib import Path


# ============================================================
# Helpers
# ============================================================

def run_shell(command):
    return subprocess.call(command, shell=True)


def run_parallel(func, iterable, max_workers):
    items = list(iterable)
    if not items:
        return []

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        return list(executor.map(func, items))


def run_command(command):
    print(f"Running: {command}")
    return subprocess.call(command, shell=True)


def fetch_existing_instances(project):
    fetch_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --format="value(name,zone)"
    '''

    output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
    instances = []

    for line in output.splitlines():
        if line.strip():
            name, inst_zone = line.split()
            instances.append((name, inst_zone))

    return instances


def parse_tsm_index(name):
    return int(name.rsplit("-", 1)[1])


def delete_instance(project, instance):
    name, inst_zone = instance

    cmd = f'''
    gcloud compute instances delete {name} \
        --zone={inst_zone} \
        --project={project} \
        --quiet
    '''

    print(f"🗑️ Deleting {name} in {inst_zone}")
    return subprocess.call(cmd, shell=True)


def delete_all_tsm_instances(project):
    instances = fetch_existing_instances(project)

    print("\n➡ Existing instances to delete:")
    for name, inst_zone in instances:
        print(f"  - {name} ({inst_zone})")

    if instances:
        run_parallel(
            lambda instance: delete_instance(project, instance),
            instances,
            max_workers=min(32, len(instances)),
        )
        print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
    else:
        print("\n✔ No tsm-sc-* instances found.\n")


def prune_tsm_instances_for_next_run(project, next_num_nodes, n_clients):
    """
    Keep only instances needed for the next system size.

    For next_num_nodes = N and n_clients = 1, we keep:
        replicas: tsm-sc-000 ... tsm-sc-(N-1)
        client:   tsm-sc-N

    Everything with index > N is deleted.
    """
    keep_indices = set(range(next_num_nodes + n_clients))
    instances = fetch_existing_instances(project)

    instances_to_delete = []

    for name, inst_zone in instances:
        idx = parse_tsm_index(name)
        if idx not in keep_indices:
            instances_to_delete.append((name, inst_zone))

    print(
        "\n➡ Pruning instances for next run: "
        f"next_num_nodes={next_num_nodes}, keeping indices "
        f"0 through {next_num_nodes + n_clients - 1}"
    )

    if instances_to_delete:
        print("Instances to delete:")
        for name, inst_zone in instances_to_delete:
            print(f"  - {name} ({inst_zone})")

        run_parallel(
            lambda instance: delete_instance(project, instance),
            instances_to_delete,
            max_workers=min(32, len(instances_to_delete)),
        )

        print("\n🧹 Non-needed instances deleted.\n")
    else:
        print("\n✔ No non-needed instances to delete.\n")


# ============================================================
# Experiment config
# ============================================================

# latencies: 50, 90, 150, 210
default_region = ["us-west1-b"]
# regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']
regions = ["us-west1-b", "us-west1-b", "us-west1-b", "us-west1-b"]

zone_no = 0

project = "research-488322"
zone = "us-west1-b"
machine_type = "e2-standard-2"
image_family = "tsm-sc-family"
subnet = "default"
gcp_username = "tejas"

# Vary system size.
# Descending order is important because we can create 32 once,
# compile once, and then delete unnecessary higher-index nodes.
NUM_NODES_LIST = [4]

# Use 1 extra 2-core machine as client machine.
# For num_nodes = N, client is tsm-sc-N.
n_clients = 1

# Fixed collection attempts.
MAX_COLLECT_ATTEMPTS = 100

# Source file to patch before the first subrun.
OVERLAY_MANAGER_IMPL_PATH = Path("src/overlay/OverlayManagerImpl.cpp")

# Start clean only once, before the first run.
DELETE_BEFORE_FIRST_RUN = False

# After each subrun, delete nodes not needed for the next smaller run.
PRUNE_UNUSED_AFTER_EACH_RUN = True

# Delete remaining VMs after the final run.
DELETE_ALL_AFTER_FINAL_RUN = True

# Code does not change across system sizes now, so compile only first subrun.
COMPILE_ONLY_FIRST_RUN = True

MAX_NUM_NODES = max(NUM_NODES_LIST)

# Client experiment settings.
CLIENT_DURATION_SEC = 360
CLIENT_WAIT_AFTER_START_SEC = 210
CLIENT_TOTAL_REQUESTS = 100000000
SERVER_BATCH_SIZE_HINT = 100
SEND_INTERVAL_US = 0

# Fixed offered load.
LOAD_POINTS = [
    {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},
]


# ============================================================
# Local source patching
# ============================================================

def patch_max_collect_attempts(max_collect_attempts):
    text = OVERLAY_MANAGER_IMPL_PATH.read_text()

    pattern = r"static\s+constexpr\s+uint64_t\s+MAX_COLLECT_ATTEMPTS\s*=\s*\d+\s*;"
    replacement = (
        f"static constexpr uint64_t MAX_COLLECT_ATTEMPTS = {max_collect_attempts};"
    )

    new_text, count = re.subn(
        pattern,
        replacement,
        text,
        flags=re.MULTILINE,
    )

    if count != 1:
        raise RuntimeError(
            f"Expected to replace exactly one MAX_COLLECT_ATTEMPTS line in "
            f"{OVERLAY_MANAGER_IMPL_PATH}, but replaced {count}"
        )

    OVERLAY_MANAGER_IMPL_PATH.write_text(new_text)

    print(
        f"Updated {OVERLAY_MANAGER_IMPL_PATH}: "
        f"MAX_COLLECT_ATTEMPTS = {max_collect_attempts}"
    )


def push_collect_patch_to_git(max_collect_attempts):
    cmd = (
        f'git add {OVERLAY_MANAGER_IMPL_PATH}; '
        f'git commit -m "set collect attempts {max_collect_attempts}" || true; '
        f'git push'
    )

    return subprocess.call(cmd, shell=True)


# ============================================================
# Main experiment loop
# ============================================================

for run_idx, num_nodes in enumerate(NUM_NODES_LIST):

    print("\n" + "#" * 100)
    print(
        f"Starting experiment for num_nodes={num_nodes}, "
        f"MAX_COLLECT_ATTEMPTS={MAX_COLLECT_ATTEMPTS}"
    )
    print("#" * 100)

    def get_zone_for_instance(i):
        # Use MAX_NUM_NODES so zone assignment stays stable.
        if i < int(MAX_NUM_NODES / 2):
            return default_region[0]
        else:
            return regions[zone_no]

    # -----------------------------------------------------------------
    # Delete existing tsm-sc-* instances only before the first subrun.
    # -----------------------------------------------------------------
    if DELETE_BEFORE_FIRST_RUN and run_idx == 0:
        print("\n🧹 Cleaning before first run...")
        delete_all_tsm_instances(project)

    # -----------------------------------------------------------------
    # Create only missing replica/client machines for this system size.
    # -----------------------------------------------------------------
    existing_instances = fetch_existing_instances(project)
    existing_names = {name for name, _ in existing_instances}

    commands = []
    newly_created_indices = []

    # Create replica nodes only if they do not already exist.
    for i in range(num_nodes):
        instance_name = f"tsm-sc-{i:03}"

        if instance_name in existing_names:
            print(f"✔ Reusing existing replica {instance_name}")
            continue

        inst_zone = get_zone_for_instance(i)

        cmd = f'''
        gcloud compute instances create {instance_name} \
            --project={project} \
            --zone={inst_zone} \
            --machine-type={machine_type} \
            --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
            --can-ip-forward \
            --maintenance-policy=MIGRATE \
            --provisioning-model=STANDARD \
            --service-account=254510644191-compute@developer.gserviceaccount.com \
            --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
            --tags=http-server,https-server \
            --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
            --no-shielded-secure-boot \
            --shielded-vtpm \
            --shielded-integrity-monitoring \
            --labels=goog-ec-src=vm_add-gcloud \
            --reservation-affinity=any
        '''
        commands.append(cmd.strip())
        newly_created_indices.append(i)

    # Create client machine only if it does not already exist.
    for i in range(n_clients):
        client_idx = num_nodes + i
        instance_name = f"tsm-sc-{client_idx:03}"

        if instance_name in existing_names:
            print(f"✔ Reusing existing client {instance_name}")
            continue

        inst_zone = get_zone_for_instance(client_idx)

        cmd = f'''
        gcloud compute instances create {instance_name} \
            --project={project} \
            --zone={inst_zone} \
            --machine-type={machine_type} \
            --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
            --can-ip-forward \
            --maintenance-policy=MIGRATE \
            --provisioning-model=STANDARD \
            --service-account=254510644191-compute@developer.gserviceaccount.com \
            --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
            --tags=http-server,https-server \
            --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
            --no-shielded-secure-boot \
            --shielded-vtpm \
            --shielded-integrity-monitoring \
            --labels=goog-ec-src=vm_add-gcloud \
            --reservation-affinity=any
        '''
        commands.append(cmd.strip())
        newly_created_indices.append(client_idx)

    if commands:
        run_parallel(
            run_command,
            commands,
            max_workers=min(48, len(commands)),
        )

        print("All missing instances launched.")

        # Give GCP/SSH a little time after VM creation.
        time.sleep(30)
    else:
        print("✔ All required instances already exist; no VM creation needed.")

    # -----------------------------------------------------------------
    # Get sorted node and client IPs.
    # tsm_ips.txt must include only replica IPs, not clients.
    # -----------------------------------------------------------------
    ip_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --sort-by=name \
        --format="value(name,zone,networkInterfaces[0].networkIP)"
    '''

    ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

    instance_records = []

    for line in ip_output.splitlines():
        if line.strip():
            name, inst_zone, ip = line.split()
            idx = int(name.rsplit("-", 1)[1])
            instance_records.append((idx, name, inst_zone, ip))

    instance_records.sort()

    node_records = [r for r in instance_records if r[0] < num_nodes]
    client_records = [
        r for r in instance_records
        if num_nodes <= r[0] < num_nodes + n_clients
    ]

    if len(node_records) != num_nodes:
        raise RuntimeError(
            f"Expected {num_nodes} replica nodes, but found "
            f"{len(node_records)}: {node_records}"
        )

    if len(client_records) != n_clients:
        raise RuntimeError(
            f"Expected {n_clients} client nodes, but found "
            f"{len(client_records)}: {client_records}"
        )

    iplist = [r[3] for r in node_records]

    with open("tsm_ips.txt", "w") as f:
        for ip in iplist:
            f.write(ip + "\n")

    print("🎯 Node IPs:", iplist)
    print("🎯 Client instances:", client_records)

    node1_ip = iplist[0]
    print(f"Clients will connect to leader/node1 at: {node1_ip}")

    # -----------------------------------------------------------------
    # Remote helper functions.
    # -----------------------------------------------------------------

    def kill_stellar_private(i):
        inst_zone = get_zone_for_instance(i)

        remote_command = """\
sudo pkill -9 stellar-core || true; \
sudo pkill -9 shab_client || true; \
"""

        command = (
            f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
            f'--project "{project}" --command "{remote_command}"'
        )

        print(f"Executing: {command}")
        output = subprocess.call(command, shell=True)
        print(f"Return code for tsm-sc-{i:03}: {output}")
        return output

    def git_pull_stellar(i):
        inst_zone = get_zone_for_instance(i)

        command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
git pull"'''

        print(command)
        output = subprocess.call(command, shell=True)
        print(output)
        return output

    def compile_stellar(i):
        inst_zone = get_zone_for_instance(i)

        command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
g++ -O2 -std=c++17 -pthread \
-I/home/tejas/stellar-core/src \
/home/tejas/stellar-core/shab_client.cpp \
-o /home/tejas/stellar-core/shab_client; \
make -j4; \
cd; \
sudo rm -rf stellar-private"'''

        print(command)
        output = subprocess.call(command, shell=True)
        print(output)
        return output

    def clean_stellar_private(i):
        inst_zone = get_zone_for_instance(i)

        remote_command = """\
cd /home/tejas; \
sudo rm -rf stellar-private; \
"""

        command = (
            f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
            f'--project "{project}" --command "{remote_command}"'
        )

        print(f"Executing: {command}")
        output = subprocess.call(command, shell=True)
        print(f"Return code for tsm-sc-{i:03}: {output}")
        return output

    def copy_folder_to_instance(
        i,
        source_folder="/home/tejas/stellar-private",
        destination_path="/home/tejas/stellar-private",
    ):
        inst_zone = get_zone_for_instance(i)
        instance_name = f"tsm-sc-{i:03}"

        command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
--recurse "{source_folder}" "{instance_name}:{destination_path}"'''

        print(f"Executing command for {instance_name}: {command}")

        output = subprocess.call(command, shell=True)

        print(f"Command for {instance_name} finished with exit code: {output}")

        return (instance_name, output)

    def run_stellar_private(i):
        inst_zone = get_zone_for_instance(i)

        node_number = i + 1
        instance_name = f"tsm-sc-{i:03}"

        remote_command = f"""\
cd /home/tejas/stellar-private; \
nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
> node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
"""

        command = (
            f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
            f'--project "{project}" --command "{remote_command}"'
        )

        print(f"Executing: {command}")

        output = subprocess.call(command, shell=True)

        print(f"Return code for {instance_name}: {output}")
        return output

    def run_stellar_client(i, client_max_in_flight, client_threads):
        inst_zone = get_zone_for_instance(i)

        instance_name = f"tsm-sc-{i:03}"
        client_id = i - num_nodes

        remote_command = f"""\
cd /home/tejas/stellar-private; \
nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
{client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
{CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
> stellar-client-{client_id}.log 2>&1 < /dev/null & disown
"""

        command = (
            f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
            f'--project "{project}" --command "{remote_command}"'
        )

        print(f"Executing client command: {command}")

        output = subprocess.call(command, shell=True)

        print(f"Return code for {instance_name}: {output}")
        return output

    # -----------------------------------------------------------------
    # Generate stellar-private configs locally using only replica IPs.
    # -----------------------------------------------------------------
    stellar_private_path = Path("../stellar-private")
    if stellar_private_path.exists():
        shutil.rmtree(stellar_private_path)
    stellar_private_path.mkdir()

    subprocess.call(
        "cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh",
        shell=True,
    )

    subprocess.call(
        "cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; "
        "./gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh",
        shell=True,
    )

    # Enable custom message only on leader.
    line_to_add = "SEND_CUSTOM_MESSAGE=true"
    target_file = "../stellar-private/node1/stellar-core.cfg"

    subprocess.call(
        f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
        shell=True,
    )

    print("\n" + "*" * 100)
    print(
        f"Preparing subrun with num_nodes={num_nodes}, "
        f"MAX_COLLECT_ATTEMPTS={MAX_COLLECT_ATTEMPTS}"
    )
    print("*" * 100)

    # -----------------------------------------------------------------
    # Patch/push/compile only on the first subrun.
    #
    # Because MAX_COLLECT_ATTEMPTS is fixed at 50 and system sizes are
    # run in descending order, all later nodes are reused from the first
    # 32-node deployment and already have compiled code.
    # -----------------------------------------------------------------
    if run_idx == 0 or not COMPILE_ONLY_FIRST_RUN:
        # Patch local source.
        patch_max_collect_attempts(MAX_COLLECT_ATTEMPTS)

        # Push patched source.
        push_collect_patch_to_git(MAX_COLLECT_ATTEMPTS)

        # Stop any old processes.
        results = run_parallel(
            kill_stellar_private,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

        # Pull patched code on active VMs.
        results = run_parallel(
            git_pull_stellar,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

        # Compile patched code on active VMs.
        results = run_parallel(
            compile_stellar,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)
    else:
        print(
            "\n✔ Skipping git pull and compile because "
            "COMPILE_ONLY_FIRST_RUN=True and this is not the first subrun."
        )

        # Still kill any stale processes on reused machines.
        results = run_parallel(
            kill_stellar_private,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

    # -----------------------------------------------------------------
    # Throughput/latency loop.
    # -----------------------------------------------------------------
    for load in LOAD_POINTS:
        active_clients = load["active_clients"]
        client_threads = load["client_threads"]
        client_max_in_flight = load["max_in_flight"]

        if active_clients > n_clients:
            raise RuntimeError(
                f"active_clients={active_clients} exceeds n_clients={n_clients}"
            )

        total_client_threads = active_clients * client_threads
        aggregate_max_in_flight = (
            active_clients * client_threads * client_max_in_flight
        )

        run_label = (
            f"clients_{active_clients}_threads_{client_threads}_"
            f"inflight_{client_max_in_flight}_"
            f"total_threads_{total_client_threads}_"
            f"total_inflight_{aggregate_max_in_flight}"
        )

        print("\n" + "=" * 80)
        print(
            f"🚀 Starting run: num_nodes={num_nodes}, "
            f"MAX_COLLECT_ATTEMPTS={MAX_COLLECT_ATTEMPTS}, {run_label}"
        )
        print("=" * 80)

        # Clean remote stellar-private.
        results = run_parallel(
            clean_stellar_private,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

        # Copy fresh stellar-private config to all active machines.
        results = run_parallel(
            copy_folder_to_instance,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

        # Kill old processes on all active machines.
        results = run_parallel(
            kill_stellar_private,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

        # Start consensus replicas.
        results = run_parallel(
            run_stellar_private,
            range(num_nodes),
            max_workers=min(48, num_nodes),
        )

        print(results)
        print("All Stellar nodes should be starting in the background.")

        # Give nodes time to authenticate and start the client listener.
        time.sleep(60)

        # Start only required client VMs for this load point.
        active_client_indices = [
            num_nodes + j for j in range(active_clients)
        ]

        results = run_parallel(
            lambda i: run_stellar_client(i, client_max_in_flight, client_threads),
            active_client_indices,
            max_workers=active_clients,
        )

        print(results)
        print(
            f"Started {active_clients} client VM(s), "
            f"each with {client_threads} client threads. "
            f"Total client threads = {total_client_threads}. "
            f"Aggregate max in-flight = {aggregate_max_in_flight}."
        )

        # Let the system run.
        time.sleep(CLIENT_WAIT_AFTER_START_SEC)

        # Stop all nodes and clients.
        results = run_parallel(
            kill_stellar_private,
            range(num_nodes + n_clients),
            max_workers=min(48, num_nodes + n_clients),
        )
        print(results)

        # -------------------------------------------------------------
        # Save logs.
        # -------------------------------------------------------------
        remote_base_folder = "/home/tejas/stellar-private"

        local_base_destination = (
            "/home/tejas/work/experiments/shabdiz/"
            + f"CollectionVSNumNodes_{MAX_COLLECT_ATTEMPTS}_"
            + f"nodes_{num_nodes}_{run_label}"
        )

        Path(local_base_destination).mkdir(parents=True, exist_ok=True)

        # Save run metadata.
        with open(Path(local_base_destination) / "run_config.txt", "w") as f:
            f.write(f"num_nodes={num_nodes}\n")
            f.write(f"max_collect_attempts={MAX_COLLECT_ATTEMPTS}\n")
            f.write(f"force_collect_after_sec=100\n")
            f.write(f"active_clients={active_clients}\n")
            f.write(f"client_threads_per_vm={client_threads}\n")
            f.write(f"total_client_threads={total_client_threads}\n")
            f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
            f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
            f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
            f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
            f.write(f"leader_ip={node1_ip}\n")
            f.write(f"machine_type={machine_type}\n")
            f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

        def copy_folder_from_instance(i):
            inst_zone = get_zone_for_instance(i)

            instance_name = f"tsm-sc-{i:03}"

            node_number = i + 1
            node_folder = f"node{node_number}"

            remote_source_path = posixpath.join(remote_base_folder, node_folder)

            local_destination_path = Path(local_base_destination) / instance_name
            local_destination_path.mkdir(parents=True, exist_ok=True)

            remote_source = f"{instance_name}:{remote_source_path}"

            command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
--recurse "{remote_source}" "{local_destination_path}"'''

            print(
                f"Executing command to copy {node_folder} "
                f"from {instance_name}: {command}"
            )

            output = subprocess.call(command, shell=True)

            print(f"Copy from {instance_name} finished with exit code: {output}")

            return (instance_name, output)

        # Copy only first 3 replica logs.
        node_copy_results = run_parallel(
            copy_folder_from_instance,
            range(3),
            max_workers=3,
        )

        def copy_client_log(i):
            inst_zone = get_zone_for_instance(i)

            instance_name = f"tsm-sc-{i:03}"
            client_id = i - num_nodes

            remote_source = (
                f"{instance_name}:/home/tejas/stellar-private/"
                f"stellar-client-{client_id}.log"
            )

            local_destination_path = Path(local_base_destination)
            local_destination_path.mkdir(parents=True, exist_ok=True)

            command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
"{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

            print(f"Copying client log from {instance_name}...")

            output = subprocess.call(command, shell=True)

            print(f"Copy finished with exit code: {output}")

            return (instance_name, output)

        client_copy_results = run_parallel(
            copy_client_log,
            active_client_indices,
            max_workers=active_clients,
        )

        print("\n--- Summary of Download Results ---")
        print("Node log copies:", node_copy_results)
        print("Client log copies:", client_copy_results)
        print(f"Saved run to: {local_base_destination}")

    # -----------------------------------------------------------------
    # After each system-size subrun, delete nodes not needed for the next.
    # -----------------------------------------------------------------
    is_last_run = run_idx == len(NUM_NODES_LIST) - 1

    if PRUNE_UNUSED_AFTER_EACH_RUN and not is_last_run:
        next_num_nodes = NUM_NODES_LIST[run_idx + 1]
        prune_tsm_instances_for_next_run(
            project=project,
            next_num_nodes=next_num_nodes,
            n_clients=n_clients,
        )

    elif is_last_run and DELETE_ALL_AFTER_FINAL_RUN:
        print("\n🧹 Final run completed. Deleting all remaining VMs...")
        delete_all_tsm_instances(project)


####################################################################################################
Starting experiment for num_nodes=4, MAX_COLLECT_ATTEMPTS=100
####################################################################################################


Running: gcloud compute instances create tsm-sc-000             --project=research-488322             --zone=us-west1-b             --machine-type=e2-standard-2             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default             --can-ip-forward             --maintenance-policy=MIGRATE             --provisioning-model=STANDARD             --service-account=254510644191-compute@developer.gserviceaccount.com             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append             --tags=http-server,https-server             --create-disk=auto-delete=yes,boot=yes,image-family=tsm-sc-family,mode=rw,size=20,type=pd-balanced             --no-shielded-secure-boot             --shielded-vtpm             --s

Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-003].
Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-004].
 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert



NAME        ZONE        MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP  STATUS
tsm-sc-003  us-west1-b  e2-standard-2               10.138.0.7   8.229.55.79  RUNNING


 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert



...

At global scope:
cc1plus: note: unrecognized command-line option ‘-Wno-unknown-warning-option’ may have been intended to silence earlier diagnostics
cc1plus: note: unrecognized command-line option ‘-Wno-unused-local-typedef’ may have been intended to silence earlier diagnostics
cc1plus: note: unrecognized command-line option ‘-Wno-unused-command-line-argument’ may have been intended to silence earlier diagnostics


make[3]: Leaving directory '/home/tejas/stellar-core/src'
make[2]: Leaving directory '/home/tejas/stellar-core/src'
make[2]: Entering directory '/home/tejas/stellar-core'
make[2]: Leaving directory '/home/tejas/stellar-core'
make[1]: Leaving directory '/home/tejas/stellar-core'
0
/bin/bash ../libtool  --tag=CXX   --mode=link g++ -std=c++17  -g -O2 -fno-omit-frame-pointer  -pthread -DFMT_HEADER_ONLY=1 -Wall -Wno-unused-command-line-argument -Wno-unused-local-typedef -Wno-unknown-warning-option -Werror=unused-result   -o stellar-core main/StellarCoreVersion.o main/XDRFilesSha256.o bucket/BucketApplicator.o bucket/BucketBase.o bucket/BucketIndexUtils.o bucket/BucketInputIterator.o bucket/BucketListBase.o bucket/BucketListSnapshotBase.o bucket/BucketManager.o bucket/BucketMergeMap.o bucket/BucketOutputIterator.o bucket/BucketSnapshot.o bucket/BucketSnapshotManager.o bucket/BucketUtils.o bucket/DiskIndex.o bucket/FutureBucket.o bucket/HotArchiveBucket.o bucket/HotArchiveBucketIndex.o bucket

Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-004].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-002].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-001].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-000].



🧹 All tsm-sc-* instances deleted across all regions.



Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-003].


# Collection vs Network Latency

In [ ]:
import subprocess
import concurrent.futures
import posixpath
import shutil
import time
import re
from pathlib import Path

import os
# ============================================================
# Helpers
# ============================================================

def run_shell(command):
    return subprocess.call(command, shell=True)


def run_parallel(func, iterable, max_workers):
    items = list(iterable)
    if not items:
        return []

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        return list(executor.map(func, items))


def run_command(command):
    print(f"Running: {command}")
    return subprocess.call(command, shell=True)


def fetch_existing_instances(project):
    fetch_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --format="value(name,zone)"
    '''

    output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
    instances = []

    for line in output.splitlines():
        if line.strip():
            name, inst_zone = line.split()
            instances.append((name, inst_zone))

    return instances


def delete_instance(project, instance):
    name, inst_zone = instance

    cmd = f'''
    gcloud compute instances delete {name} \
        --zone={inst_zone} \
        --project={project} \
        --quiet
    '''

    print(f"🗑️ Deleting {name} in {inst_zone}")
    return subprocess.call(cmd, shell=True)


def delete_all_tsm_instances(project):
    instances = fetch_existing_instances(project)

    print("\n➡ Existing instances to delete:")
    for name, inst_zone in instances:
        print(f"  - {name} ({inst_zone})")

    if instances:
        run_parallel(
            lambda instance: delete_instance(project, instance),
            instances,
            max_workers=min(32, len(instances)),
        )
        print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
    else:
        print("\n✔ No tsm-sc-* instances found.\n")


# ============================================================
# Experiment config
# ============================================================

project = "research-488322"
machine_type = "e2-standard-2"
image_family = "tsm-sc-family"
subnet = "default"
gcp_username = "tejas"

# Fixed system size.
NUM_NODES = 16

# One client VM.
# Client index will be tsm-sc-016.
n_clients = 1

# Fixed collection attempts.
COLLECT_ATTEMPT_POINTS = [5]

# First 8 replicas are always here.
BASE_REPLICA_ZONE = "us-west1-b"

# Client is always here.
CLIENT_ZONE = "us-west1-b"

# Replicas 8..15 sweep over these remote zones.
REMOTE_REPLICA_ZONES = [
    "asia-northeast1-b",
    "europe-west3-c",
    "asia-south1-c",
    "northamerica-northeast2-a"
    "us-west1-b"
]

# Source file to patch before each subrun.
OVERLAY_MANAGER_IMPL_PATH = Path("src/overlay/OverlayManagerImpl.cpp")

# Local stellar-private path.
LOCAL_STELLAR_PRIVATE_PATH = Path("../stellar-private").resolve()

# Delete before first run.
DELETE_BEFORE_FIRST_RUN = True

# Delete all VMs after every subrun.
DELETE_AFTER_EACH_SUBRUN = True

# Since VMs are deleted after every subrun, compile every subrun.
COMPILE_EVERY_SUBRUN = True

# Copy only these replica logs.
# Change to [0, 8, 15] if you want one local and two remote-node logs.
COPY_NODE_INDICES = [0, 1, 2]

# Client experiment settings.
CLIENT_DURATION_SEC = 360
CLIENT_WAIT_AFTER_START_SEC = 210
CLIENT_TOTAL_REQUESTS = 100000000
SERVER_BATCH_SIZE_HINT = 100
SEND_INTERVAL_US = 0

# Fixed offered load.
LOAD_POINTS = [
    {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},
]


# ============================================================
# Local source patching
# ============================================================

def patch_max_collect_attempts(max_collect_attempts):
    text = OVERLAY_MANAGER_IMPL_PATH.read_text()

    pattern = r"static\s+constexpr\s+uint64_t\s+MAX_COLLECT_ATTEMPTS\s*=\s*\d+\s*;"
    replacement = (
        f"static constexpr uint64_t MAX_COLLECT_ATTEMPTS = {max_collect_attempts};"
    )

    new_text, count = re.subn(
        pattern,
        replacement,
        text,
        flags=re.MULTILINE,
    )

    if count != 1:
        raise RuntimeError(
            f"Expected to replace exactly one MAX_COLLECT_ATTEMPTS line in "
            f"{OVERLAY_MANAGER_IMPL_PATH}, but replaced {count}"
        )

    OVERLAY_MANAGER_IMPL_PATH.write_text(new_text)

    print(
        f"Updated {OVERLAY_MANAGER_IMPL_PATH}: "
        f"MAX_COLLECT_ATTEMPTS = {max_collect_attempts}"
    )


def push_collect_patch_to_git(max_collect_attempts):
    cmd = (
        f'git add .; '
        f'git commit -m "set collect attempts {max_collect_attempts}" || true; '
        f'git push'
    )

    return subprocess.call(cmd, shell=True)


# ============================================================
# Zone assignment
# ============================================================

def get_zone_for_instance(i, remote_replica_zone):
    """
    Replica layout for NUM_NODES = 16:
        tsm-sc-000 ... tsm-sc-007 -> us-west1-b
        tsm-sc-008 ... tsm-sc-015 -> remote_replica_zone
        tsm-sc-016                -> us-west1-b client
    """
    if i < 8:
        return BASE_REPLICA_ZONE

    if i < NUM_NODES:
        return remote_replica_zone

    return CLIENT_ZONE


# ============================================================
# GCP instance creation
# ============================================================

def create_instance_command(instance_name, inst_zone):
    return f'''
    gcloud compute instances create {instance_name} \
        --project={project} \
        --zone={inst_zone} \
        --machine-type={machine_type} \
        --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
        --can-ip-forward \
        --maintenance-policy=MIGRATE \
        --provisioning-model=STANDARD \
        --service-account=254510644191-compute@developer.gserviceaccount.com \
        --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
        --tags=http-server,https-server \
        --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
        --no-shielded-secure-boot \
        --shielded-vtpm \
        --shielded-integrity-monitoring \
        --labels=goog-ec-src=vm_add-gcloud \
        --reservation-affinity=any
    '''.strip()


# ============================================================
# Main experiment loop
# ============================================================
os.system('git add .; git commit -m "update"; git push')
subrun_idx = 0

for remote_replica_zone in REMOTE_REPLICA_ZONES:
    for max_collect_attempts in COLLECT_ATTEMPT_POINTS:

        subrun_idx += 1

        print("\n" + "#" * 100)
        print(
            f"Starting network-latency subrun {subrun_idx}: "
            f"num_nodes={NUM_NODES}, "
            f"MAX_COLLECT_ATTEMPTS={max_collect_attempts}, "
            f"first_8_zone={BASE_REPLICA_ZONE}, "
            f"next_8_zone={remote_replica_zone}, "
            f"client_zone={CLIENT_ZONE}"
        )
        print("#" * 100)

        if DELETE_BEFORE_FIRST_RUN and subrun_idx == 1:
            print("\n🧹 Cleaning before first run...")
            delete_all_tsm_instances(project)

        try:
            # ------------------------------------------------------------
            # Create all replica/client machines for this subrun.
            # Since we delete after every subrun, these are usually fresh.
            # ------------------------------------------------------------
            existing_instances = fetch_existing_instances(project)
            existing_names = {name for name, _ in existing_instances}

            commands = []

            # Create replica nodes.
            for i in range(NUM_NODES):
                instance_name = f"tsm-sc-{i:03}"

                if instance_name in existing_names:
                    print(f"✔ Reusing existing replica {instance_name}")
                    continue

                inst_zone = get_zone_for_instance(i, remote_replica_zone)
                commands.append(create_instance_command(instance_name, inst_zone))

            # Create client node.
            for i in range(n_clients):
                client_idx = NUM_NODES + i
                instance_name = f"tsm-sc-{client_idx:03}"

                if instance_name in existing_names:
                    print(f"✔ Reusing existing client {instance_name}")
                    continue

                inst_zone = get_zone_for_instance(client_idx, remote_replica_zone)
                commands.append(create_instance_command(instance_name, inst_zone))

            if commands:
                run_parallel(
                    run_command,
                    commands,
                    max_workers=min(48, len(commands)),
                )

                print("All missing instances launched.")
                time.sleep(30)
            else:
                print("✔ All required instances already exist; no VM creation needed.")

            # ------------------------------------------------------------
            # Get sorted replica and client IPs.
            # tsm_ips.txt must include only replica IPs, not clients.
            # ------------------------------------------------------------
            ip_cmd = f'''
            gcloud compute instances list \
                --project={project} \
                --filter="name~'^tsm-sc-'" \
                --sort-by=name \
                --format="value(name,zone,networkInterfaces[0].networkIP)"
            '''

            ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

            instance_records = []

            for line in ip_output.splitlines():
                if line.strip():
                    name, inst_zone, ip = line.split()
                    idx = int(name.rsplit("-", 1)[1])
                    instance_records.append((idx, name, inst_zone, ip))

            instance_records.sort()

            node_records = [r for r in instance_records if r[0] < NUM_NODES]
            client_records = [
                r for r in instance_records
                if NUM_NODES <= r[0] < NUM_NODES + n_clients
            ]

            if len(node_records) != NUM_NODES:
                raise RuntimeError(
                    f"Expected {NUM_NODES} replica nodes, but found "
                    f"{len(node_records)}: {node_records}"
                )

            if len(client_records) != n_clients:
                raise RuntimeError(
                    f"Expected {n_clients} client nodes, but found "
                    f"{len(client_records)}: {client_records}"
                )

            iplist = [r[3] for r in node_records]

            with open("tsm_ips.txt", "w") as f:
                for ip in iplist:
                    f.write(ip + "\n")

            print("🎯 Node IPs:", iplist)
            print("🎯 Client instances:", client_records)

            node1_ip = iplist[0]
            print(f"Clients will connect to leader/node1 at: {node1_ip}")

            # ------------------------------------------------------------
            # Remote helper functions.
            # ------------------------------------------------------------

            def kill_stellar_private(i):
                inst_zone = get_zone_for_instance(i, remote_replica_zone)

                remote_command = """\
sudo pkill -9 stellar-core || true; \
sudo pkill -9 shab_client || true; \
"""

                command = (
                    f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
                    f'--project "{project}" --command "{remote_command}"'
                )

                print(f"Executing: {command}")
                output = subprocess.call(command, shell=True)
                print(f"Return code for tsm-sc-{i:03}: {output}")
                return output

            def git_pull_stellar(i):
                inst_zone = get_zone_for_instance(i, remote_replica_zone)

                command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
git pull"'''

                print(command)
                output = subprocess.call(command, shell=True)
                print(output)
                return output

            def compile_stellar(i):
                inst_zone = get_zone_for_instance(i, remote_replica_zone)

                command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
g++ -O2 -std=c++17 -pthread \
-I/home/tejas/stellar-core/src \
/home/tejas/stellar-core/shab_client.cpp \
-o /home/tejas/stellar-core/shab_client; \
make -j4; \
cd; \
sudo rm -rf stellar-private"'''

                print(command)
                output = subprocess.call(command, shell=True)
                print(output)
                return output

            def clean_stellar_private(i):
                inst_zone = get_zone_for_instance(i, remote_replica_zone)

                remote_command = """\
cd /home/tejas; \
sudo rm -rf stellar-private; \
"""

                command = (
                    f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
                    f'--project "{project}" --command "{remote_command}"'
                )

                print(f"Executing: {command}")
                output = subprocess.call(command, shell=True)
                print(f"Return code for tsm-sc-{i:03}: {output}")
                return output

            def copy_folder_to_instance(
                i,
                source_folder=None,
                destination_path="/home/tejas/stellar-private",
            ):
                inst_zone = get_zone_for_instance(i, remote_replica_zone)
                instance_name = f"tsm-sc-{i:03}"

                if source_folder is None:
                    source_folder = str(LOCAL_STELLAR_PRIVATE_PATH)

                command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
--recurse "{source_folder}" "{instance_name}:{destination_path}"'''

                print(f"Executing command for {instance_name}: {command}")

                output = subprocess.call(command, shell=True)

                print(f"Command for {instance_name} finished with exit code: {output}")

                return (instance_name, output)

            def run_stellar_private(i):
                inst_zone = get_zone_for_instance(i, remote_replica_zone)

                node_number = i + 1
                instance_name = f"tsm-sc-{i:03}"

                remote_command = f"""\
cd /home/tejas/stellar-private; \
nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
> node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
"""

                command = (
                    f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
                    f'--project "{project}" --command "{remote_command}"'
                )

                print(f"Executing: {command}")

                output = subprocess.call(command, shell=True)

                print(f"Return code for {instance_name}: {output}")
                return output

            def run_stellar_client(i, client_max_in_flight, client_threads):
                inst_zone = get_zone_for_instance(i, remote_replica_zone)

                instance_name = f"tsm-sc-{i:03}"
                client_id = i - NUM_NODES

                remote_command = f"""\
cd /home/tejas/stellar-private; \
nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
{client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
{CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
> stellar-client-{client_id}.log 2>&1 < /dev/null & disown
"""

                command = (
                    f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
                    f'--project "{project}" --command "{remote_command}"'
                )

                print(f"Executing client command: {command}")

                output = subprocess.call(command, shell=True)

                print(f"Return code for {instance_name}: {output}")
                return output

            # ------------------------------------------------------------
            # Generate stellar-private configs locally using only replica IPs.
            # This must run every subrun because IPs/zones change.
            # ------------------------------------------------------------
            if LOCAL_STELLAR_PRIVATE_PATH.exists():
                shutil.rmtree(LOCAL_STELLAR_PRIVATE_PATH)
            LOCAL_STELLAR_PRIVATE_PATH.mkdir(parents=True, exist_ok=True)

            subprocess.call(
                f"cp gcp_setup_stellar_private.sh "
                f"{LOCAL_STELLAR_PRIVATE_PATH}/gcp_setup_stellar_private.sh",
                shell=True,
            )

            subprocess.call(
                f"cd {LOCAL_STELLAR_PRIVATE_PATH}; "
                f"chmod +x gcp_setup_stellar_private.sh; "
                f"./gcp_setup_stellar_private.sh start; "
                f"./gcp_setup_stellar_private.sh",
                shell=True,
            )

            # Enable custom message only on leader.
            line_to_add = "SEND_CUSTOM_MESSAGE=true"
            target_file = LOCAL_STELLAR_PRIVATE_PATH / "node1" / "stellar-core.cfg"

            subprocess.call(
                f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
                shell=True,
            )

            print("\n" + "*" * 100)
            print(
                f"Preparing subrun with "
                f"MAX_COLLECT_ATTEMPTS={max_collect_attempts}, "
                f"remote_replica_zone={remote_replica_zone}"
            )
            print("*" * 100)

            # ------------------------------------------------------------
            # Patch, push, pull, and compile every subrun.
            # ------------------------------------------------------------
            patch_max_collect_attempts(max_collect_attempts)
            push_collect_patch_to_git(max_collect_attempts)

            results = run_parallel(
                kill_stellar_private,
                range(NUM_NODES + n_clients),
                max_workers=min(48, NUM_NODES + n_clients),
            )
            print(results)

            results = run_parallel(
                git_pull_stellar,
                range(NUM_NODES + n_clients),
                max_workers=min(48, NUM_NODES + n_clients),
            )
            print(results)

            results = run_parallel(
                compile_stellar,
                range(NUM_NODES + n_clients),
                max_workers=min(48, NUM_NODES + n_clients),
            )
            print(results)

            # ------------------------------------------------------------
            # Throughput/latency loop.
            # ------------------------------------------------------------
            for load in LOAD_POINTS:
                active_clients = load["active_clients"]
                client_threads = load["client_threads"]
                client_max_in_flight = load["max_in_flight"]

                if active_clients > n_clients:
                    raise RuntimeError(
                        f"active_clients={active_clients} exceeds n_clients={n_clients}"
                    )

                total_client_threads = active_clients * client_threads
                aggregate_max_in_flight = (
                    active_clients * client_threads * client_max_in_flight
                )

                run_label = (
                    f"clients_{active_clients}_threads_{client_threads}_"
                    f"inflight_{client_max_in_flight}_"
                    f"total_threads_{total_client_threads}_"
                    f"total_inflight_{aggregate_max_in_flight}"
                )

                print("\n" + "=" * 80)
                print(
                    f"🚀 Starting run: "
                    f"num_nodes={NUM_NODES}, "
                    f"MAX_COLLECT_ATTEMPTS={max_collect_attempts}, "
                    f"remote_replica_zone={remote_replica_zone}, "
                    f"{run_label}"
                )
                print("=" * 80)

                # Clean remote stellar-private.
                results = run_parallel(
                    clean_stellar_private,
                    range(NUM_NODES + n_clients),
                    max_workers=min(48, NUM_NODES + n_clients),
                )
                print(results)

                # Copy fresh stellar-private config to all active machines.
                results = run_parallel(
                    copy_folder_to_instance,
                    range(NUM_NODES + n_clients),
                    max_workers=min(48, NUM_NODES + n_clients),
                )
                print(results)

                # Kill old processes on all active machines.
                results = run_parallel(
                    kill_stellar_private,
                    range(NUM_NODES + n_clients),
                    max_workers=min(48, NUM_NODES + n_clients),
                )
                print(results)

                # Start consensus replicas.
                results = run_parallel(
                    run_stellar_private,
                    range(NUM_NODES),
                    max_workers=min(48, NUM_NODES),
                )

                print(results)
                print("All Stellar nodes should be starting in the background.")

                # Give nodes time to authenticate and start the client listener.
                time.sleep(40)

                # Start only required client VMs for this load point.
                active_client_indices = [
                    NUM_NODES + j for j in range(active_clients)
                ]

                results = run_parallel(
                    lambda i: run_stellar_client(
                        i,
                        client_max_in_flight,
                        client_threads,
                    ),
                    active_client_indices,
                    max_workers=active_clients,
                )

                print(results)
                print(
                    f"Started {active_clients} client VM(s), "
                    f"each with {client_threads} client threads. "
                    f"Total client threads = {total_client_threads}. "
                    f"Aggregate max in-flight = {aggregate_max_in_flight}."
                )

                # Let the system run.
                time.sleep(CLIENT_WAIT_AFTER_START_SEC)

                # Stop all nodes and clients.
                results = run_parallel(
                    kill_stellar_private,
                    range(NUM_NODES + n_clients),
                    max_workers=min(48, NUM_NODES + n_clients),
                )
                print(results)

                # --------------------------------------------------------
                # Save logs.
                # --------------------------------------------------------
                remote_base_folder = "/home/tejas/stellar-private"

                local_base_destination = (
                    "/home/tejas/work/experiments/shabdiz/"
                    + f"CollectionVSNetworkLatency_{max_collect_attempts}_"
                    + f"remote_{remote_replica_zone}_"
                    + f"nodes_{NUM_NODES}_{run_label}"
                )

                Path(local_base_destination).mkdir(parents=True, exist_ok=True)

                with open(Path(local_base_destination) / "run_config.txt", "w") as f:
                    f.write(f"experiment=CollectionVSNetworkLatency\n")
                    f.write(f"num_nodes={NUM_NODES}\n")
                    f.write(f"max_collect_attempts={max_collect_attempts}\n")
                    f.write(f"force_collect_after_sec=100\n")
                    f.write(f"base_replica_zone={BASE_REPLICA_ZONE}\n")
                    f.write(f"remote_replica_zone={remote_replica_zone}\n")
                    f.write(f"client_zone={CLIENT_ZONE}\n")
                    f.write(f"replicas_0_to_7_zone={BASE_REPLICA_ZONE}\n")
                    f.write(f"replicas_8_to_15_zone={remote_replica_zone}\n")
                    f.write(f"client_instance=tsm-sc-{NUM_NODES:03}\n")
                    f.write(f"active_clients={active_clients}\n")
                    f.write(f"client_threads_per_vm={client_threads}\n")
                    f.write(f"total_client_threads={total_client_threads}\n")
                    f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
                    f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
                    f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
                    f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
                    f.write(f"leader_ip={node1_ip}\n")
                    f.write(f"machine_type={machine_type}\n")
                    f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

                def copy_folder_from_instance(i):
                    inst_zone = get_zone_for_instance(i, remote_replica_zone)

                    instance_name = f"tsm-sc-{i:03}"

                    node_number = i + 1
                    node_folder = f"node{node_number}"

                    remote_source_path = posixpath.join(remote_base_folder, node_folder)

                    local_destination_path = Path(local_base_destination) / instance_name
                    local_destination_path.mkdir(parents=True, exist_ok=True)

                    remote_source = f"{instance_name}:{remote_source_path}"

                    command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
--recurse "{remote_source}" "{local_destination_path}"'''

                    print(
                        f"Executing command to copy {node_folder} "
                        f"from {instance_name}: {command}"
                    )

                    output = subprocess.call(command, shell=True)

                    print(f"Copy from {instance_name} finished with exit code: {output}")

                    return (instance_name, output)

                node_copy_results = run_parallel(
                    copy_folder_from_instance,
                    COPY_NODE_INDICES,
                    max_workers=min(3, len(COPY_NODE_INDICES)),
                )

                def copy_client_log(i):
                    inst_zone = get_zone_for_instance(i, remote_replica_zone)

                    instance_name = f"tsm-sc-{i:03}"
                    client_id = i - NUM_NODES

                    remote_source = (
                        f"{instance_name}:/home/tejas/stellar-private/"
                        f"stellar-client-{client_id}.log"
                    )

                    local_destination_path = Path(local_base_destination)
                    local_destination_path.mkdir(parents=True, exist_ok=True)

                    command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
"{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

                    print(f"Copying client log from {instance_name}...")

                    output = subprocess.call(command, shell=True)

                    print(f"Copy finished with exit code: {output}")

                    return (instance_name, output)

                client_copy_results = run_parallel(
                    copy_client_log,
                    active_client_indices,
                    max_workers=active_clients,
                )

                print("\n--- Summary of Download Results ---")
                print("Node log copies:", node_copy_results)
                print("Client log copies:", client_copy_results)
                print(f"Saved run to: {local_base_destination}")

        finally:
            if DELETE_AFTER_EACH_SUBRUN:
                print(
                    "\n🧹 Deleting VMs after completed/attempted subrun: "
                    f"remote_replica_zone={remote_replica_zone}, "
                    f"MAX_COLLECT_ATTEMPTS={max_collect_attempts}"
                )
                delete_all_tsm_instances(project)

[main 4b1be41] update
 2 files changed, 4685 insertions(+), 440 deletions(-)


To github.com:tejas-shivanand-mane/stellar-core.git
   6058f4b..4b1be41  main -> main



####################################################################################################
Starting network-latency subrun 1: num_nodes=16, MAX_COLLECT_ATTEMPTS=5, first_8_zone=us-west1-b, next_8_zone=asia-northeast1-b, client_zone=us-west1-b
####################################################################################################

🧹 Cleaning before first run...



➡ Existing instances to delete:

✔ No tsm-sc-* instances found.



Running: gcloud compute instances create tsm-sc-000         --project=research-488322         --zone=us-west1-b         --machine-type=e2-standard-2         --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default         --can-ip-forward         --maintenance-policy=MIGRATE         --provisioning-model=STANDARD         --service-account=254510644191-compute@developer.gserviceaccount.com         --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append         --tags=http-server,https-server         --create-disk=auto-delete=yes,boot=yes,image-family=tsm-sc-family,mode=rw,size=20,type=pd-balanced         --no-shielded-secure-boot         --shielded-vtpm         --shielded-integrity-monitoring         --labels=goog-ec-sr

...

NAME        ZONE        MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP     STATUS
tsm-sc-004  us-west1-b  e2-standard-2               10.138.0.2   136.66.221.231  RUNNING
NAME        ZONE        MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP    STATUS
tsm-sc-001  us-west1-b  e2-standard-2               10.138.0.19  136.66.39.157  RUNNING


 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert



NAME        ZONE        MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP     STATUS
tsm-sc-005  us-west1-b  e2-standard-2               10.138.0.9   136.66.207.242  RUNNING


Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-002].
 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert



NAME        ZONE        MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP      STATUS
tsm-sc-002  us-west1-b  e2-standard-2               10.138.0.18  136.109.221.202  RUNNING


In [54]:

    # Fetch all tsm-sc-* instances across ALL zones
    fetch_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --format="value(name,zone)"
    '''
    
    output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
    instances = []
    
    for line in output.splitlines():
        if line.strip():
            name, zone = line.split()
            instances.append((name, zone))
    
    print("\n➡ Existing instances to delete:")
    for name, zone in instances:
        print(f"  - {name} ({zone})")
    
    def delete_instance(name, zone):
        cmd = f'''
        gcloud compute instances delete {name} \
            --zone={zone} \
            --project={project} \
            --quiet
        '''
        print(f"🗑️ Deleting {name} in {zone}")
        return subprocess.call(cmd, shell=True)
    
    if instances:
        with concurrent.futures.ThreadPoolExecutor(max_workers=32) as executor:
            futures = [
                executor.submit(delete_instance, name, zone)
                for name, zone in instances
            ]
            concurrent.futures.wait(futures)
    
        print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
    else:
        print("\n✔ No tsm-sc-* instances found.\n")


➡ Existing instances to delete:
  - tsm-sc-000 (us-west1-b)
  - tsm-sc-001 (us-west1-b)
  - tsm-sc-002 (us-west1-b)
  - tsm-sc-003 (us-west1-b)
  - tsm-sc-004 (us-west1-b)
  - tsm-sc-005 (us-west1-b)
  - tsm-sc-006 (us-west1-b)
  - tsm-sc-007 (us-west1-b)
  - tsm-sc-016 (us-west1-b)
  - tsm-sc-008 (us-east5-a)
  - tsm-sc-011 (us-east5-a)
  - tsm-sc-012 (us-east5-a)
🗑️ Deleting tsm-sc-000 in us-west1-b
🗑️ Deleting tsm-sc-001 in us-west1-b
🗑️ Deleting tsm-sc-002 in us-west1-b
🗑️ Deleting tsm-sc-003 in us-west1-b
🗑️ Deleting tsm-sc-004 in us-west1-b
🗑️ Deleting tsm-sc-005 in us-west1-b
🗑️ Deleting tsm-sc-006 in us-west1-b
🗑️ Deleting tsm-sc-007 in us-west1-b
🗑️ Deleting tsm-sc-016 in us-west1-b
🗑️ Deleting tsm-sc-008 in us-east5-a
🗑️ Deleting tsm-sc-011 in us-east5-a
🗑️ Deleting tsm-sc-012 in us-east5-a


Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-east5-a/instances/tsm-sc-008].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-007].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-016].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-005].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-004].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-002].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-001].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-000].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-s


🧹 All tsm-sc-* instances deleted across all regions.



Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-east5-a/instances/tsm-sc-012].


In [23]:
#             # Clean remote stellar-private.
#             results = run_parallel(
#                 clean_stellar_private,
#                 range(num_nodes + n_clients),
#                 max_workers=min(48, num_nodes + n_clients),
#             )
#             print(results)

#             # Copy fresh stellar-private config to all active machines.
#             results = run_parallel(
#                 copy_folder_to_instance,
#                 range(num_nodes + n_clients),
#                 max_workers=min(48, num_nodes + n_clients),
#             )
#             print(results)

#             # Kill old processes on all active machines.
#             results = run_parallel(
#                 kill_stellar_private,
#                 range(num_nodes + n_clients),
#                 max_workers=min(48, num_nodes + n_clients),
#             )
#             print(results)

#             # Start consensus replicas.
#             results = run_parallel(
#                 run_stellar_private,
#                 range(num_nodes),
#                 max_workers=min(48, num_nodes),
#             )

#             print(results)
#             print("All Stellar nodes should be starting in the background.")

#             # Give nodes time to authenticate and start the client listener.
#             time.sleep(60)

#             # Start only required client VMs for this load point.
#             active_client_indices = [
#                 num_nodes + j for j in range(active_clients)
#             ]

#             results = run_parallel(
#                 lambda i: run_stellar_client(i, client_max_in_flight, client_threads),
#                 active_client_indices,
#                 max_workers=active_clients,
#             )

#             print(results)
#             print(
#                 f"Started {active_clients} client VM(s), "
#                 f"each with {client_threads} client threads. "
#                 f"Total client threads = {total_client_threads}. "
#                 f"Aggregate max in-flight = {aggregate_max_in_flight}."
#             )

#             # Let the system run before forcing failures.
#             time.sleep(CLIENT_WAIT_AFTER_START_SEC)


#             # Stop all nodes and clients.
#             results = run_parallel(
#                 kill_stellar_private,
#                 range(num_nodes + n_clients),
#                 max_workers=min(48, num_nodes + n_clients),
#             )
#             print(results)

#             # -----------------------------------------------------------------
#             # Save logs.
#             # -----------------------------------------------------------------
#             remote_base_folder = "/home/tejas/stellar-private"

#             local_base_destination = (
#                 "/home/tejas/work/experiments/shabdiz/"
#                 + f"Collection_{max_collect_attempts}_"
#                 + f"nodes_{num_nodes}_{run_label}"
#             )

#             Path(local_base_destination).mkdir(parents=True, exist_ok=True)

#             # Save run metadata.
#             with open(Path(local_base_destination) / "run_config.txt", "w") as f:
#                 f.write(f"num_nodes={num_nodes}\n")
#                 f.write(f"max_collect_attempts={max_collect_attempts}\n")
#                 f.write(f"force_collect_after_sec=100\n")
#                 f.write(f"active_clients={active_clients}\n")
#                 f.write(f"client_threads_per_vm={client_threads}\n")
#                 f.write(f"total_client_threads={total_client_threads}\n")
#                 f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
#                 f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
#                 f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
#                 f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
#                 f.write(f"leader_ip={node1_ip}\n")
#                 f.write(f"machine_type={machine_type}\n")
#                 f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

#             def copy_folder_from_instance(i):
#                 inst_zone = get_zone_for_instance(i)

#                 instance_name = f"tsm-sc-{i:03}"

#                 node_number = i + 1
#                 node_folder = f"node{node_number}"

#                 remote_source_path = posixpath.join(remote_base_folder, node_folder)

#                 local_destination_path = Path(local_base_destination) / instance_name
#                 local_destination_path.mkdir(parents=True, exist_ok=True)

#                 remote_source = f"{instance_name}:{remote_source_path}"

#                 command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{remote_source}" "{local_destination_path}"'''

#                 print(
#                     f"Executing command to copy {node_folder} "
#                     f"from {instance_name}: {command}"
#                 )

#                 output = subprocess.call(command, shell=True)

#                 print(f"Copy from {instance_name} finished with exit code: {output}")

#                 return (instance_name, output)

#             # Copy all replica logs.
#             node_copy_results = run_parallel(
#                 copy_folder_from_instance,
#                 range(3),
#                 max_workers=min(48, max(1, num_nodes)),
#             )

#             def copy_client_log(i):
#                 inst_zone = get_zone_for_instance(i)

#                 instance_name = f"tsm-sc-{i:03}"
#                 client_id = i - num_nodes

#                 remote_source = (
#                     f"{instance_name}:/home/tejas/stellar-private/"
#                     f"stellar-client-{client_id}.log"
#                 )

#                 local_destination_path = Path(local_base_destination)
#                 local_destination_path.mkdir(parents=True, exist_ok=True)

#                 command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

#                 print(f"Copying client log from {instance_name}...")

#                 output = subprocess.call(command, shell=True)

#                 print(f"Copy finished with exit code: {output}")

#                 return (instance_name, output)

#             client_copy_results = run_parallel(
#                 copy_client_log,
#                 active_client_indices,
#                 max_workers=active_clients,
#             )

#             print("\n--- Summary of Download Results ---")
#             print("Node log copies:", node_copy_results)
#             print("Client log copies:", client_copy_results)
#             print(f"Saved run to: {local_base_destination}")

Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-000" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-001" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-002" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-003" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-004" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-005" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-006" --project "research-48

In [28]:

# import subprocess
# import concurrent.futures
# import posixpath
# import shutil
# import time
# import re
# from pathlib import Path


# def run_shell(command):
#     return subprocess.call(command, shell=True)


# def run_parallel(func, iterable, max_workers):
#     items = list(iterable)
#     if not items:
#         return []
#     with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
#         return list(executor.map(func, items))


# # =============================================================================
# # Fixed read-ratio experiment configuration
# # =============================================================================

# # Fixed system size for performance-vs-read-ratio experiment.
# NUM_NODES = 8

# # Use 1 extra 2-core machine as client machine.
# # For num_nodes = 8, client is tsm-sc-008.
# n_clients = 1

# # Read-ratio workloads. These must already exist in CustomYCSBWorkload.h:
# # WORKLOAD_R0, WORKLOAD_R25, WORKLOAD_R50, WORKLOAD_R75, WORKLOAD_R100
# READ_RATIO_WORKLOADS = [
#     # {"read_ratio": 0,   "workload": "WORKLOAD_R0"},
#     {"read_ratio": 10,   "workload": "WORKLOAD_R10"},
#     # {"read_ratio": 25,  "workload": "WORKLOAD_R25"},
#     # {"read_ratio": 50,  "workload": "WORKLOAD_R50"},
#     # {"read_ratio": 75,  "workload": "WORKLOAD_R75"},
#     # {"read_ratio": 20,  "workload": "WORKLOAD_R20"},
#     {"read_ratio": 40,  "workload": "WORKLOAD_R40"},
#     # {"read_ratio": 60,  "workload": "WORKLOAD_R60"},
#     # {"read_ratio": 80,  "workload": "WORKLOAD_R80"},
#     # {"read_ratio": 100, "workload": "WORKLOAD_R100"},
# ]

# # The file containing:
# #   static YCSBWorkload currentWorkload = WORKLOAD_R50;
# YCSB_HEADER_LOCAL = Path("src/overlay/CustomYCSBWorkload.h")

# # Start clean before the first read-ratio run.
# DELETE_BEFORE_FIRST_RUN = True

# # Delete all VMs after the final read-ratio run.
# DELETE_ALL_AFTER_FINAL_RUN = True

# # Compile stellar-core only in the first read-ratio subrun.
# # The client is compiled every subrun because CustomYCSBWorkload.h changes.
# COMPILE_STELLAR_CORE_ONLY_FIRST_RUN = True

# # Client experiment settings.
# CLIENT_DURATION_SEC = 360
# CLIENT_WAIT_AFTER_START_SEC = 210
# CLIENT_TOTAL_REQUESTS = 100000000
# SERVER_BATCH_SIZE_HINT = 100
# SEND_INTERVAL_US = 0

# # Same fixed client load as your previous experiment.
# LOAD_POINTS = [
#     {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},   # total inflight 300
# ]

# # GCP settings.
# project = "research-488322"
# zone = "us-central1-c"
# machine_type = "e2-standard-2"
# image_family = "tsm-sc-family"  # your custom image
# subnet = "default"
# gcp_username = "tejas"

# # latencies: 50, 90 150, 210
# default_region = ["us-central1-a"]
# # regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# # regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']
# regions = ["us-central1-a", "us-central1-a", "us-central1-a", "us-central1-a"]

# zone_no = 0


# def get_zone_for_instance(i):
#     # Keep the same placement rule as your previous script.
#     if i < int(NUM_NODES / 2):
#         return default_region[0]
#     else:
#         return regions[zone_no]


# def set_local_ycsb_workload(workload_name):
#     """
#     Updates this line in src/overlay/CustomYCSBWorkload.h:

#         static YCSBWorkload currentWorkload = WORKLOAD_R50;

#     to the requested workload.
#     """
#     if not YCSB_HEADER_LOCAL.exists():
#         raise FileNotFoundError(
#             f"Could not find {YCSB_HEADER_LOCAL}. "
#             "Run this script from the stellar-core repo root."
#         )

#     text = YCSB_HEADER_LOCAL.read_text()

#     new_text, count = re.subn(
#         r"static\s+YCSBWorkload\s+currentWorkload\s*=\s*WORKLOAD_[A-Za-z0-9_]+\s*;",
#         f"static YCSBWorkload currentWorkload = {workload_name};",
#         text,
#         count=1,
#     )

#     if count != 1:
#         raise RuntimeError(
#             "Could not find exactly one currentWorkload line in "
#             f"{YCSB_HEADER_LOCAL}"
#         )

#     YCSB_HEADER_LOCAL.write_text(new_text)
#     print(f"✅ Updated {YCSB_HEADER_LOCAL}: currentWorkload = {workload_name}")


# def fetch_existing_instances():
#     fetch_cmd = f"""
#     gcloud compute instances list \
#         --project={project} \
#         --filter="name~'^tsm-sc-'" \
#         --format="value(name,zone)"
#     """

#     output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
#     instances = []

#     for line in output.splitlines():
#         if line.strip():
#             name, inst_zone = line.split()
#             instances.append((name, inst_zone))

#     return instances


# def delete_instance(instance):
#     name, inst_zone = instance

#     cmd = f"""
#     gcloud compute instances delete {name} \
#         --zone={inst_zone} \
#         --project={project} \
#         --quiet
#     """

#     print(f"🗑️ Deleting {name} in {inst_zone}")
#     return subprocess.call(cmd, shell=True)


# def parse_tsm_index(name):
#     return int(name.rsplit("-", 1)[1])


# def run_command(command):
#     print(f"Running: {command}")
#     return subprocess.call(command, shell=True)


# def kill_stellar_private(i):
#     inst_zone = get_zone_for_instance(i)

#     remote_command = f"""\
# cd /home/tejas/stellar-private 2>/dev/null || true; \
# sudo pkill -9 stellar-core || true; sudo pkill -9 shab_client || true; \
# """

#     command = (
#         f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#         f'--project "{project}" --command "{remote_command}"'
#     )

#     print(f"Executing: {command}")
#     output = subprocess.call(command, shell=True)
#     print(f"Return code for tsm-sc-{i:03}: {output}")
#     return output


# def clean_stellar_private(i):
#     inst_zone = get_zone_for_instance(i)

#     remote_command = f"""\
# cd /home/tejas; \
# sudo rm -rf stellar-private; \
# """

#     command = (
#         f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#         f'--project "{project}" --command "{remote_command}"'
#     )

#     print(f"Executing: {command}")
#     output = subprocess.call(command, shell=True)
#     print(f"Return code for tsm-sc-{i:03}: {output}")
#     return output


# def copy_folder_to_instance(
#     i,
#     source_folder="/home/tejas/stellar-private",
#     destination_path="/home/tejas/stellar-private",
# ):
#     inst_zone = get_zone_for_instance(i)
#     instance_name = f"tsm-sc-{i:03}"

#     command = f"""gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{source_folder}" "{instance_name}:{destination_path}" """

#     print(f"Executing command for {instance_name}: {command}")

#     output = subprocess.call(command, shell=True)

#     print(f"Command for {instance_name} finished with exit code: {output}")

#     return (instance_name, output)


# def run_stellar_private(i):
#     inst_zone = get_zone_for_instance(i)

#     node_number = i + 1
#     instance_name = f"tsm-sc-{i:03}"

#     remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# """

#     command = (
#         f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#         f'--project "{project}" --command "{remote_command}"'
#     )

#     print(f"Executing: {command}")

#     output = subprocess.call(command, shell=True)

#     print(f"Return code for {instance_name}: {output}")
#     return output


# def git_pull_stellar(i):
#     inst_zone = get_zone_for_instance(i)

#     command = f"""gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# git pull\""""

#     print(command)
#     output = subprocess.call(command, shell=True)
#     print(output)
#     return output


# def compile_stellar_and_client(i):
#     """
#     First subrun only.
#     This compiles both shab_client and stellar-core.
#     """
#     inst_zone = get_zone_for_instance(i)

#     command = f"""gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# g++ -O2 -std=c++17 -pthread \
# -I/home/tejas/stellar-core/src \
# /home/tejas/stellar-core/shab_client.cpp \
# -o /home/tejas/stellar-core/shab_client; \
# make -j16; \
# cd; \
# sudo rm -rf stellar-private\""""

#     print(command)
#     output = subprocess.call(command, shell=True)
#     print(output)
#     return output


# def copy_ycsb_header_to_instance(i):
#     """
#     Later subruns only.
#     Copy changed CustomYCSBWorkload.h to the VM source tree.
#     This keeps the source file in sync. We only recompile the client binary after this.
#     """
#     inst_zone = get_zone_for_instance(i)
#     instance_name = f"tsm-sc-{i:03}"

#     remote_path = (
#         f"{instance_name}:/home/tejas/stellar-core/src/overlay/"
#         f"{YCSB_HEADER_LOCAL.name}"
#     )

#     command = f"""gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{YCSB_HEADER_LOCAL}" "{remote_path}" """

#     print(f"Copying workload header to {instance_name}: {command}")
#     output = subprocess.call(command, shell=True)
#     print(f"Header copy to {instance_name} finished with exit code: {output}")
#     return (instance_name, output)


# def compile_client_only(i):
#     """
#     Every subrun after the first.
#     This recompiles only shab_client, not stellar-core.
#     """
#     inst_zone = get_zone_for_instance(i)

#     command = f"""gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd /home/tejas/stellar-core; \
# g++ -O2 -std=c++17 -pthread \
# -I/home/tejas/stellar-core/src \
# /home/tejas/stellar-core/shab_client.cpp \
# -o /home/tejas/stellar-core/shab_client\""""

#     print(command)
#     output = subprocess.call(command, shell=True)
#     print(output)
#     return output


# # =============================================================================
# # VM setup
# # =============================================================================

# if DELETE_BEFORE_FIRST_RUN:
#     instances = fetch_existing_instances()

#     print("\n➡ Existing instances to delete before first run:")
#     for name, inst_zone in instances:
#         print(f"  - {name} ({inst_zone})")

#     if instances:
#         run_parallel(
#             delete_instance,
#             instances,
#             max_workers=min(32, len(instances)),
#         )
#         print("\n🧹 All existing tsm-sc-* instances deleted before first run.\n")
#     else:
#         print("\n✔ No existing tsm-sc-* instances found before first run.\n")


# # Create required replica/client machines.
# existing_instances = fetch_existing_instances()
# existing_names = {name for name, _ in existing_instances}

# commands = []

# # Replicas: tsm-sc-000 ... tsm-sc-007
# for i in range(NUM_NODES):
#     instance_name = f"tsm-sc-{i:03}"

#     if instance_name in existing_names:
#         print(f"✔ Reusing existing replica {instance_name}")
#         continue

#     inst_zone = get_zone_for_instance(i)

#     cmd = f"""
#     gcloud compute instances create {instance_name} \
#         --project={project} \
#         --zone={inst_zone} \
#         --machine-type={machine_type} \
#         --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#         --can-ip-forward \
#         --maintenance-policy=MIGRATE \
#         --provisioning-model=STANDARD \
#         --service-account=254510644191-compute@developer.gserviceaccount.com \
#         --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#         --tags=http-server,https-server \
#         --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#         --no-shielded-secure-boot \
#         --shielded-vtpm \
#         --shielded-integrity-monitoring \
#         --labels=goog-ec-src=vm_add-gcloud \
#         --reservation-affinity=any
#     """
#     commands.append(cmd.strip())

# # Client: tsm-sc-008
# for i in range(n_clients):
#     client_idx = NUM_NODES + i
#     instance_name = f"tsm-sc-{client_idx:03}"

#     if instance_name in existing_names:
#         print(f"✔ Reusing existing client {instance_name}")
#         continue

#     inst_zone = get_zone_for_instance(client_idx)

#     cmd = f"""
#     gcloud compute instances create {instance_name} \
#         --project={project} \
#         --zone={inst_zone} \
#         --machine-type={machine_type} \
#         --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#         --can-ip-forward \
#         --maintenance-policy=MIGRATE \
#         --provisioning-model=STANDARD \
#         --service-account=254510644191-compute@developer.gserviceaccount.com \
#         --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#         --tags=http-server,https-server \
#         --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#         --no-shielded-secure-boot \
#         --shielded-vtpm \
#         --shielded-integrity-monitoring \
#         --labels=goog-ec-src=vm_add-gcloud \
#         --reservation-affinity=any
#     """
#     commands.append(cmd.strip())

# if commands:
#     run_parallel(
#         run_command,
#         commands,
#         max_workers=min(48, len(commands)),
#     )

#     print("All missing instances launched.")

#     # Give GCP/SSH a little time after VM creation.
#     time.sleep(30)
# else:
#     print("✔ All required instances already exist; no VM creation needed.")


# # Get sorted node and client IPs.
# ip_cmd = f"""
# gcloud compute instances list \
#     --project={project} \
#     --filter="name~'^tsm-sc-'" \
#     --sort-by=name \
#     --format="value(name,zone,networkInterfaces[0].networkIP)"
# """

# ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

# instance_records = []

# for line in ip_output.splitlines():
#     if line.strip():
#         name, inst_zone, ip = line.split()
#         idx = int(name.rsplit("-", 1)[1])
#         instance_records.append((idx, name, inst_zone, ip))

# instance_records.sort()

# node_records = [r for r in instance_records if r[0] < NUM_NODES]
# client_records = [r for r in instance_records if NUM_NODES <= r[0] < NUM_NODES + n_clients]

# if len(node_records) != NUM_NODES:
#     raise RuntimeError(
#         f"Expected {NUM_NODES} replica nodes, but found {len(node_records)}: {node_records}"
#     )

# if len(client_records) != n_clients:
#     raise RuntimeError(
#         f"Expected {n_clients} client nodes, but found {len(client_records)}: {client_records}"
#     )

# iplist = [r[3] for r in node_records]

# with open("tsm_ips.txt", "w") as f:
#     for ip in iplist:
#         f.write(ip + "\n")

# print("🎯 Node IPs:", iplist)
# print("🎯 Client instances:", client_records)

# node1_ip = iplist[0]
# print(f"Clients will connect to leader/node1 at: {node1_ip}")

# all_instance_indices = list(range(NUM_NODES + n_clients))


# # =============================================================================
# # Read-ratio experiment loop
# # =============================================================================

# for workload_idx, workload_cfg in enumerate(READ_RATIO_WORKLOADS):
#     read_ratio = workload_cfg["read_ratio"]
#     workload_name = workload_cfg["workload"]

#     print("\n" + "#" * 100)
#     print(
#         f"Starting read-ratio experiment: "
#         f"num_nodes={NUM_NODES}, read_ratio={read_ratio}%, workload={workload_name}"
#     )
#     print("#" * 100)

#     # -------------------------------------------------------------------------
#     # Change CustomYCSBWorkload.h for this subrun.
#     # -------------------------------------------------------------------------
#     set_local_ycsb_workload(workload_name)

#     # -------------------------------------------------------------------------
#     # First subrun:
#     #   - commit/push changed workload
#     #   - git pull on every VM
#     #   - compile shab_client + stellar-core on every VM
#     #
#     # Later subruns:
#     #   - copy changed header to VMs
#     #   - compile only shab_client on client VM(s)
#     # -------------------------------------------------------------------------
#     first_subrun = (workload_idx == 0)

#     if first_subrun:
#         subprocess.call(
#             f'git add .; git commit -m "read ratio workload {workload_name}" || true; git push',
#             shell=True,
#         )

#         # Optional local build, preserved from your earlier script.
#         subprocess.call("make -j8", shell=True)

#         results = run_parallel(
#             kill_stellar_private,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )

#         results = run_parallel(
#             git_pull_stellar,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )
#         print(results)

#         results = run_parallel(
#             compile_stellar_and_client,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )
#         print(results)
#     else:
#         # No git pull and no stellar-core compilation after the first subrun.
#         # The server binary is unchanged. The client workload changes through
#         # CustomYCSBWorkload.h, so update source and recompile shab_client.
#         results = run_parallel(
#             copy_ycsb_header_to_instance,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )
#         print(results)

#     # -------------------------------------------------------------------------
#     # Generate stellar-private configs locally using only replica IPs.
#     # -------------------------------------------------------------------------
#     stellar_private_path = Path("../stellar-private")
#     if stellar_private_path.exists():
#         shutil.rmtree(stellar_private_path)
#     stellar_private_path.mkdir()

#     subprocess.call(
#         "cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh",
#         shell=True,
#     )

#     subprocess.call(
#         "cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; "
#         "./gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh",
#         shell=True,
#     )

#     # Enable custom message only on leader.
#     line_to_add = "SEND_CUSTOM_MESSAGE=true"
#     target_file = "../stellar-private/node1/stellar-core.cfg"

#     subprocess.call(
#         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#         shell=True,
#     )

#     # -------------------------------------------------------------------------
#     # Fixed client load loop.
#     # -------------------------------------------------------------------------
#     for load in LOAD_POINTS:
#         active_clients = load["active_clients"]
#         client_threads = load["client_threads"]
#         client_max_in_flight = load["max_in_flight"]

#         if active_clients > n_clients:
#             raise RuntimeError(
#                 f"active_clients={active_clients} exceeds n_clients={n_clients}"
#             )

#         total_client_threads = active_clients * client_threads
#         aggregate_max_in_flight = (
#             active_clients * client_threads * client_max_in_flight
#         )

#         active_client_indices = [
#             NUM_NODES + j for j in range(active_clients)
#         ]

#         # Compile the client every subrun. On the first subrun it was already
#         # compiled by compile_stellar_and_client(), but compiling it again is
#         # cheap and guarantees the workload header is reflected in shab_client.
#         results = run_parallel(
#             compile_client_only,
#             active_client_indices,
#             max_workers=active_clients,
#         )
#         print(results)

#         run_label = (
#             f"read_ratio_{read_ratio}_"
#             f"{workload_name}_"
#             f"nodes_{NUM_NODES}_"
#             f"clients_{active_clients}_threads_{client_threads}_"
#             f"inflight_{client_max_in_flight}_"
#             f"total_threads_{total_client_threads}_"
#             f"total_inflight_{aggregate_max_in_flight}"
#         )

#         print("\n" + "=" * 80)
#         print(
#             f"🚀 Starting read-ratio run: "
#             f"num_nodes={NUM_NODES}, read_ratio={read_ratio}%, "
#             f"workload={workload_name}, {run_label}"
#         )
#         print("=" * 80)

#         results = run_parallel(
#             clean_stellar_private,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )

#         results = run_parallel(
#             copy_folder_to_instance,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )

#         # Kill old processes on all replica and client machines.
#         results = run_parallel(
#             kill_stellar_private,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )

#         # Start consensus replicas.
#         results = run_parallel(
#             run_stellar_private,
#             range(NUM_NODES),
#             max_workers=min(48, NUM_NODES),
#         )

#         print(results)
#         print("All Stellar nodes should be starting in the background.")

#         # Give nodes time to authenticate and start the client listener.
#         time.sleep(80)

#         def run_stellar_client(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - NUM_NODES

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
# {client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
# {CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
# > stellar-client-{client_id}.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing client command: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         results = run_parallel(
#             run_stellar_client,
#             active_client_indices,
#             max_workers=active_clients,
#         )

#         print(results)
#         print(
#             f"Started {active_clients} client VM(s), "
#             f"each with {client_threads} client threads. "
#             f"Total client threads = {total_client_threads}. "
#             f"Aggregate max in-flight = {aggregate_max_in_flight}."
#         )

#         # Wait for the duration run to produce stable per-second client logs.
#         # The clients may wait forever on final partial write batches, so we kill them after this.
#         time.sleep(CLIENT_WAIT_AFTER_START_SEC)

#         # Stop all nodes and clients.
#         results = run_parallel(
#             kill_stellar_private,
#             all_instance_indices,
#             max_workers=min(48, len(all_instance_indices)),
#         )

#         remote_base_folder = "/home/tejas/stellar-private"

#         local_base_destination = (
#             "/home/tejas/work/experiments/shabdiz/"
#             + f"SHABDIZ_vs_read_ratio_{run_label}"
#         )

#         Path(local_base_destination).mkdir(parents=True, exist_ok=True)

#         # Save run metadata.
#         with open(Path(local_base_destination) / "run_config.txt", "w") as f:
#             f.write(f"num_nodes={NUM_NODES}\n")
#             f.write(f"read_ratio_percent={read_ratio}\n")
#             f.write(f"workload={workload_name}\n")
#             f.write(f"active_clients={active_clients}\n")
#             f.write(f"client_threads_per_vm={client_threads}\n")
#             f.write(f"total_client_threads={total_client_threads}\n")
#             f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
#             f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
#             f.write(f"client_wait_after_start_sec={CLIENT_WAIT_AFTER_START_SEC}\n")
#             f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
#             f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
#             f.write(f"leader_ip={node1_ip}\n")
#             f.write(f"machine_type={machine_type}\n")
#             f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")
#             f.write(f"compile_stellar_core_this_subrun={first_subrun}\n")
#             f.write("client_compiled_this_subrun=true\n")

#         def copy_folder_from_instance(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"

#             node_number = i + 1
#             node_folder = f"node{node_number}"

#             remote_source_path = posixpath.join(remote_base_folder, node_folder)

#             local_destination_path = Path(local_base_destination) / instance_name
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             remote_source = f"{instance_name}:{remote_source_path}"

#             command = f"""gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{remote_source}" "{local_destination_path}" """

#             print(f"Executing command to copy {node_folder} from {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy from {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         # Copy only a few node logs to reduce time.
#         # Change range(min(3, NUM_NODES)) to range(NUM_NODES) if you want all logs.
#         node_copy_results = run_parallel(
#             copy_folder_from_instance,
#             range(min(3, NUM_NODES)),
#             max_workers=min(48, max(1, min(3, NUM_NODES))),
#         )

#         def copy_client_log(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - NUM_NODES

#             remote_source = (
#                 f"{instance_name}:/home/tejas/stellar-private/"
#                 f"stellar-client-{client_id}.log"
#             )

#             local_destination_path = Path(local_base_destination)
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             command = f"""gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{remote_source}" "{local_destination_path}/client_{client_id}.log" """

#             print(f"Copying client log from {instance_name}...")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy finished with exit code: {output}")

#             return (instance_name, output)

#         client_copy_results = run_parallel(
#             copy_client_log,
#             active_client_indices,
#             max_workers=active_clients,
#         )

#         print("\n--- Summary of Download Results ---")
#         print("Node log copies:", node_copy_results)
#         print("Client log copies:", client_copy_results)
#         print(f"Saved run to: {local_base_destination}")


# # =============================================================================
# # Cleanup after all read-ratio subruns
# # =============================================================================

# # if DELETE_ALL_AFTER_FINAL_RUN:
# #     instances = fetch_existing_instances()

# #     print("\n➡ All read-ratio runs complete. Deleting all remaining tsm-sc-* VMs:")
# #     for name, inst_zone in instances:
# #         print(f"  - {name} ({inst_zone})")

# #     if instances:
# #         run_parallel(
# #             delete_instance,
# #             instances,
# #             max_workers=min(32, len(instances)),
# #         )
# #         print(f"\n🧹 Deleted {len(instances)} instance(s).\n")
# #     else:
# #         print("\n✔ No instances to delete.\n")
# # else:
# #     print("\n✔ All read-ratio runs complete. Leaving VMs running.")



➡ Existing instances to delete before first run:

✔ No existing tsm-sc-* instances found before first run.



Running: gcloud compute instances create tsm-sc-000         --project=research-488322         --zone=us-central1-a         --machine-type=e2-standard-2         --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default         --can-ip-forward         --maintenance-policy=MIGRATE         --provisioning-model=STANDARD         --service-account=254510644191-compute@developer.gserviceaccount.com         --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append         --tags=http-server,https-server         --create-disk=auto-delete=yes,boot=yes,image-family=tsm-sc-family,mode=rw,size=20,type=pd-balanced         --no-shielded-secure-boot         --shielded-vtpm         --shielded-integrity-monitoring         --labels=goog-ec

Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-000].
 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert



NAME        ZONE           MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP    STATUS
tsm-sc-000  us-central1-a  e2-standard-2               10.128.0.68  34.70.225.218  RUNNING


Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-001].
Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-006].
Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-008].
 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert



...

Detected 8 nodes based on tsm_ips.txt.
Initializing database for node1...
Initializing database for node2...
Initializing database for node3...
Initializing database for node4...
Initializing database for node5...
Initializing database for node6...


2026-06-24T13:08:19.870 [default INFO] Config from /home/tejas/stellar-private/node1/stellar-core.cfg
2026-06-24T13:08:19.872 [default INFO] Generated QUORUM_SET: {
   "t" : 5,
   "v" : [
      "node4",
      "node7",
      "node6",
      "node2",
      "node5",
      "node8",
      "GDB7U",
      "node3"
   ]
}

2026-06-24T13:08:19.872 [default WARNING] Adjusted TARGET_PEER_CONNECTIONS to 336 due to insufficient MAX_ADDITIONAL_PEER_CONNECTIONS=1000
2026-06-24T13:08:19.872 [default INFO] Assigning calculated value of 3 to FAILURE_SAFETY
2026-06-24T13:08:19.919 [default INFO] Config from /home/tejas/stellar-private/node2/stellar-core.cfg
2026-06-24T13:08:19.921 [default INFO] Generated QUORUM_SET: {
   "t" : 5,
   "v" : [
      "node4",
      "node7",
      "node6",
      "GC4TM",
      "node5",
      "node8",
      "node1",
      "node3"
   ]
}

2026-06-24T13:08:19.921 [default WARNING] Adjusted TARGET_PEER_CONNECTIONS to 336 due to insufficient MAX_ADDITIONAL_PEER_CONNECTIONS=1000
202

Initializing database for node7...
Initializing database for node8...
✅ 8-node private Stellar network setup complete!
Start the nodes with (substituting node number):
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node1/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node2/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node3/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node4/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node5/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node6/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node7/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node8/ste

2026-06-24T13:08:20.097 [default INFO] Config from /home/tejas/stellar-private/node7/stellar-core.cfg
2026-06-24T13:08:20.099 [default INFO] Generated QUORUM_SET: {
   "t" : 5,
   "v" : [
      "node4",
      "GCDAF",
      "node6",
      "node2",
      "node5",
      "node8",
      "node1",
      "node3"
   ]
}

2026-06-24T13:08:20.099 [default WARNING] Adjusted TARGET_PEER_CONNECTIONS to 336 due to insufficient MAX_ADDITIONAL_PEER_CONNECTIONS=1000
2026-06-24T13:08:20.099 [default INFO] Assigning calculated value of 3 to FAILURE_SAFETY
2026-06-24T13:08:20.130 [default INFO] Config from /home/tejas/stellar-private/node8/stellar-core.cfg
2026-06-24T13:08:20.133 [default INFO] Generated QUORUM_SET: {
   "t" : 5,
   "v" : [
      "node4",
      "node7",
      "node6",
      "node2",
      "node5",
      "GC73B",
      "node1",
      "node3"
   ]
}

2026-06-24T13:08:20.133 [default WARNING] Adjusted TARGET_PEER_CONNECTIONS to 336 due to insufficient MAX_ADDITIONAL_PEER_CONNECTIONS=1000
202

0
[0]

🚀 Starting read-ratio run: num_nodes=8, read_ratio=40%, workload=WORKLOAD_R40, read_ratio_40_WORKLOAD_R40_nodes_8_clients_1_threads_2_inflight_150_total_threads_2_total_inflight_300
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-000" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-001" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-002" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-003" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Executing: gcloud compute ssh --zone "us-central1-a" "tsm-sc-004" --project "research-488322" --command "cd /home/tejas; sudo rm -rf stellar-private; "
Executing: gcloud compute ssh --zone "us-central1-a

#  Memory Exp

In [ ]:
# import subprocess
# import concurrent.futures
# import posixpath
# import shutil
# import time
# from pathlib import Path


# def run_shell(command):
#     return subprocess.call(command, shell=True)


# def run_parallel(func, iterable, max_workers):
#     items = list(iterable)
#     if not items:
#         return []
#     with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
#         return list(executor.map(func, items))


# # latencies: 50, 90 150, 210
# default_region = ['us-central1-a']
# # regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# # regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']

# regions = ['us-central1-a', 'us-central1-a', 'us-central1-a', 'us-central1-a']


# # Regions

# zone_no = 0

# # Run throughput/latency vs num_nodes for these system sizes.
# NUM_NODES_LIST = [4]

# # Use 1 extra 2-core machine as client machine.
# # For num_nodes = N, client is tsm-sc-N.
# n_clients = 1

# # Start clean only once, before the largest run.
# # After that, keep the lower-index VMs and delete only the extra higher-index VMs.
# DELETE_BEFORE_FIRST_RUN = False

# # Since NUM_NODES_LIST goes 48 -> 32 -> 16 -> 8 -> 4, delete only the VMs
# # that will not be needed by the next smaller run.
# DELETE_UNUSED_AFTER_EACH_RUN = False

# # Delete the remaining 4 replica VMs + 1 client VM after the final run.
# DELETE_ALL_AFTER_FINAL_RUN = False

# # Code is unchanged across node-count runs, so compile only on the first run.
# COMPILE_ONLY_FIRST_RUN = True

# MAX_NUM_NODES = max(NUM_NODES_LIST)

# # Client experiment settings.
# CLIENT_DURATION_SEC = 360
# CLIENT_WAIT_AFTER_START_SEC = 160
# CLIENT_TOTAL_REQUESTS = 100000000
# SERVER_BATCH_SIZE_HINT = 100
# SEND_INTERVAL_US = 0

# # Fixed load point for throughput-vs-num_nodes and latency-vs-num_nodes.
# # Aggregate max in-flight = active_clients * client_threads * max_in_flight = 200.
# LOAD_POINTS = [
#     {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},   # total inflight 300
# ]


# for run_idx, num_nodes in enumerate(NUM_NODES_LIST):
# # for zone_no in  [0,1,2,3, 4]:

#     project = "research-488322"
#     zone = "us-central1-a"
#     machine_type = "e2-standard-2"
#     image_family = "tsm-sc-family"  # your custom image
#     subnet = "default"
#     gcp_username = "tejas"

#     print("\n" + "#" * 100)
#     print(f"Starting experiment for num_nodes={num_nodes}")
#     print("#" * 100)

#     def get_zone_for_instance(i):
#         # IMPORTANT: use MAX_NUM_NODES instead of current num_nodes.
#         # We reuse VMs while moving 48 -> 32 -> 16 -> 8 -> 4, so the zone for
#         # tsm-sc-016, tsm-sc-032, etc. must stay the same across runs.
#         if i < int(MAX_NUM_NODES / 2):
#             return default_region[0]
#         else:
#             return regions[zone_no]

#     def fetch_existing_instances():
#         fetch_cmd = f'''
#         gcloud compute instances list \
#             --project={project} \
#             --filter="name~'^tsm-sc-'" \
#             --format="value(name,zone)"
#         '''

#         output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
#         instances = []

#         for line in output.splitlines():
#             if line.strip():
#                 name, inst_zone = line.split()
#                 instances.append((name, inst_zone))

#         return instances

#     def delete_instance(instance):
#         name, inst_zone = instance

#         cmd = f'''
#         gcloud compute instances delete {name} \
#             --zone={inst_zone} \
#             --project={project} \
#             --quiet
#         '''

#         print(f"🗑️ Deleting {name} in {inst_zone}")
#         return subprocess.call(cmd, shell=True)

#     def parse_tsm_index(name):
#         return int(name.rsplit("-", 1)[1])

#     # -------------------------------------------------------------------------
#     # Delete existing tsm-sc-* instances only before the first/largest run.
#     # Later runs reuse tsm-sc-000 ... tsm-sc-(next_num_nodes).
#     # -------------------------------------------------------------------------
#     if DELETE_BEFORE_FIRST_RUN and run_idx == 0:
#         instances = fetch_existing_instances()

#         print("\n➡ Existing instances to delete before first run:")
#         for name, inst_zone in instances:
#             print(f"  - {name} ({inst_zone})")

#         if instances:
#             run_parallel(
#                 delete_instance,
#                 instances,
#                 max_workers=min(32, len(instances))
#             )
#             print("\n🧹 All existing tsm-sc-* instances deleted before first run.\n")
#         else:
#             print("\n✔ No existing tsm-sc-* instances found before first run.\n")

#     # -------------------------------------------------------------------------
#     # Create only the missing replica/client machines for this num_nodes run.
#     # When moving 48 -> 32 -> 16 -> 8 -> 4, the lower-index machines are reused.
#     # Replicas: tsm-sc-000 ... tsm-sc-(num_nodes-1)
#     # Client:   tsm-sc-num_nodes
#     # -------------------------------------------------------------------------
#     existing_instances = fetch_existing_instances()
#     existing_names = {name for name, _ in existing_instances}

#     commands = []
#     newly_created_indices = []

#     # Create replica nodes only if they do not already exist.
#     for i in range(num_nodes):
#         instance_name = f"tsm-sc-{i:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing replica {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(i)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(i)

#     # Create client machine only if it does not already exist.
#     for i in range(n_clients):
#         client_idx = num_nodes + i
#         instance_name = f"tsm-sc-{client_idx:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing client {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(client_idx)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(client_idx)

#     def run_command(command):
#         print(f"Running: {command}")
#         return subprocess.call(command, shell=True)

#     if commands:
#         run_parallel(
#             run_command,
#             commands,
#             max_workers=min(48, len(commands))
#         )

#         print("All missing instances launched.")

#         # Give GCP/SSH a little time after VM creation.
#         time.sleep(30)
#     else:
#         print("✔ All required instances already exist; no VM creation needed.")

#     # -------------------------------------------------------------------------
#     # Get sorted node and client IPs.
#     # IMPORTANT: tsm_ips.txt must include only server/replica IPs.
#     # Client IPs must not be included.
#     # -------------------------------------------------------------------------
#     ip_cmd = f'''
#     gcloud compute instances list \
#         --project={project} \
#         --filter="name~'^tsm-sc-'" \
#         --sort-by=name \
#         --format="value(name,zone,networkInterfaces[0].networkIP)"
#     '''

#     ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

#     instance_records = []

#     for line in ip_output.splitlines():
#         if line.strip():
#             name, inst_zone, ip = line.split()
#             idx = int(name.rsplit("-", 1)[1])
#             instance_records.append((idx, name, inst_zone, ip))

#     instance_records.sort()

#     node_records = [r for r in instance_records if r[0] < num_nodes]
#     client_records = [r for r in instance_records if num_nodes <= r[0] < num_nodes + n_clients]

#     if len(node_records) != num_nodes:
#         raise RuntimeError(
#             f"Expected {num_nodes} replica nodes, but found {len(node_records)}: {node_records}"
#         )

#     if len(client_records) != n_clients:
#         raise RuntimeError(
#             f"Expected {n_clients} client nodes, but found {len(client_records)}: {client_records}"
#         )

#     iplist = [r[3] for r in node_records]

#     with open("tsm_ips.txt", "w") as f:
#         for ip in iplist:
#             f.write(ip + "\n")

#     print("🎯 Node IPs:", iplist)
#     print("🎯 Client instances:", client_records)

#     node1_ip = iplist[0]
#     print(f"Clients will connect to leader/node1 at: {node1_ip}")

#     # -------------------------------------------------------------------------
#     # Push/pull/compile only for the first run.
#     # Later runs reuse the same lower-index VMs, and the code has not changed.
#     # -------------------------------------------------------------------------
#     def kill_stellar_private(i):
#         inst_zone = get_zone_for_instance(i)

#         remote_command = f"""\
# cd /home/tejas/stellar-private; \
# sudo pkill -9 stellar-core; sudo pkill -9 shab_client; \
# """

#         command = (
#             f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#             f'--project "{project}" --command "{remote_command}"'
#         )

#         print(f"Executing: {command}")
#         output = subprocess.call(command, shell=True)
#         print(f"Return code for tsm-sc-{i:03}: {output}")
#         return output

#     do_compile_this_run = True#(not COMPILE_ONLY_FIRST_RUN) or (run_idx == 0)

# #     if do_compile_this_run:
# #         subprocess.call('git add .; git commit -m "testing"; git push', shell=True)

# #         n_collection = 100
# #         subprocess.call('make -j8', shell=True)

# #         results = run_parallel(
# #             kill_stellar_private,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )

# #         def git_pull_stellar(i):
# #             inst_zone = get_zone_for_instance(i)

# #             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# # cd stellar-core; \
# # git pull"'''

# #             print(command)
# #             output = subprocess.call(command, shell=True)
# #             print(output)
# #             return output

# #         results = run_parallel(
# #             git_pull_stellar,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )
# #         print(results)

# #         # ---------------------------------------------------------------------
# #         # Compile only on the first/largest run.
# #         # Since later runs reuse a subset of these VMs, no recompilation is needed.
# #         # ---------------------------------------------------------------------
# #         def compile_stellar(i):
# #             inst_zone = get_zone_for_instance(i)

# #             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# # cd stellar-core; \
# # g++ -O2 -std=c++17 -pthread \
# # -I/home/tejas/stellar-core/src \
# # /home/tejas/stellar-core/shab_client.cpp \
# # -o /home/tejas/stellar-core/shab_client; \
# # make -j16; \
# # cd; \
# # sudo rm -rf stellar-private"'''

# #             print(command)
# #             output = subprocess.call(command, shell=True)
# #             print(output)
# #             return output

# #         results = run_parallel(
# #             compile_stellar,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )


        
# #         print(results)
# #     else:
# #         print(f"⏭️ Skipping git pull and compilation for num_nodes={num_nodes}; code already compiled on first run.")

# #     # -------------------------------------------------------------------------
# #     # Generate stellar-private configs locally using only replica IPs.
# #     # -------------------------------------------------------------------------
# #     stellar_private_path = Path('../stellar-private')
# #     if stellar_private_path.exists():
# #         shutil.rmtree(stellar_private_path)
# #     stellar_private_path.mkdir()

# #     subprocess.call(
# #         'cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh',
# #         shell=True
# #     )

# #     subprocess.call(
# #         'cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; '
# #         './gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh',
# #         shell=True
# #     )

# #     # Enable custom message only on leader.
# #     line_to_add = "SEND_CUSTOM_MESSAGE=true"
# #     target_file = "../stellar-private/node1/stellar-core.cfg"

# #     subprocess.call(
# #         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
# #         shell=True
# #     )

# #     # Optional memory profiling on node2.
# #     if num_nodes >= 2:
# #         target_file = "../stellar-private/node2/stellar-core.cfg"
# #         line_to_add = "MEMORY_PROF=true"

# #         subprocess.call(
# #             f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
# #             shell=True
# #         )

# #     #     print(f"The line '{line_to_add}' has been prepended to {target_file}.")

# #     # -------------------------------------------------------------------------
# #     # Throughput/latency experiment loop.
# #     # Fixed offered load for scalability:
# #     # active_clients=1, client_threads=2, max_in_flight=100.
# #     # Aggregate max in-flight = 200.
# #     # -------------------------------------------------------------------------
# #     for load in LOAD_POINTS:
# #         active_clients = load["active_clients"]
# #         client_threads = load["client_threads"]
# #         client_max_in_flight = load["max_in_flight"]

# #         if active_clients > n_clients:
# #             raise RuntimeError(
# #                 f"active_clients={active_clients} exceeds n_clients={n_clients}"
# #             )

# #         total_client_threads = active_clients * client_threads
# #         aggregate_max_in_flight = active_clients * client_threads * client_max_in_flight

# #         run_label = (
# #             f"clients_{active_clients}_threads_{client_threads}_"
# #             f"inflight_{client_max_in_flight}_"
# #             f"total_threads_{total_client_threads}_"
# #             f"total_inflight_{aggregate_max_in_flight}"
# #         )

# #         print("\n" + "=" * 80)
# #         print(f"🚀 Starting throughput/latency run: num_nodes={num_nodes}, {run_label}")
# #         print("=" * 80)

# #         def clean_stellar_private(i):
# #             inst_zone = get_zone_for_instance(i)

# #             remote_command = f"""\
# # cd /home/tejas; \
# # sudo rm -rf stellar-private; \
# # """

# #             command = (
# #                 f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
# #                 f'--project "{project}" --command "{remote_command}"'
# #             )

# #             print(f"Executing: {command}")
# #             output = subprocess.call(command, shell=True)
# #             print(f"Return code for tsm-sc-{i:03}: {output}")
# #             return output

# #         results = run_parallel(
# #             clean_stellar_private,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )

# #         def copy_folder_to_instance(
# #             i,
# #             source_folder="/home/tejas/stellar-private",
# #             destination_path="/home/tejas/stellar-private"
# #         ):
# #             inst_zone = get_zone_for_instance(i)
# #             instance_name = f"tsm-sc-{i:03}"

# #             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# # --recurse "{source_folder}" "{instance_name}:{destination_path}"'''

# #             print(f"Executing command for {instance_name}: {command}")

# #             output = subprocess.call(command, shell=True)

# #             print(f"Command for {instance_name} finished with exit code: {output}")

# #             return (instance_name, output)

# #         results = run_parallel(
# #             copy_folder_to_instance,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )

# #         def run_stellar_private(i):
# #             inst_zone = get_zone_for_instance(i)

# #             node_number = i + 1
# #             instance_name = f"tsm-sc-{i:03}"

# #             remote_command = f"""\
# # cd /home/tejas/stellar-private; \
# # nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# # > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# # """

# #             command = (
# #                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
# #                 f'--project "{project}" --command "{remote_command}"'
# #             )

# #             print(f"Executing: {command}")

# #             output = subprocess.call(command, shell=True)

# #             print(f"Return code for {instance_name}: {output}")
# #             return output

# #         # Kill old processes on all replica and client machines.
# #         results = run_parallel(
# #             kill_stellar_private,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )

# #         # Start consensus replicas.
# #         results = run_parallel(
# #             run_stellar_private,
# #             range(num_nodes),
# #             max_workers=min(48, num_nodes)
# #         )

# #         print(results)
# #         print("All Stellar nodes should be starting in the background.")

# #         # Give nodes time to authenticate and start the client listener.
# #         time.sleep(80)

# #         def run_stellar_client(i):
# #             inst_zone = get_zone_for_instance(i)

# #             instance_name = f"tsm-sc-{i:03}"
# #             client_id = i - num_nodes

# #             remote_command = f"""\
# # cd /home/tejas/stellar-private; \
# # nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
# # {client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
# # {CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
# # > stellar-client-{client_id}.log 2>&1 < /dev/null & disown
# # """

# #             command = (
# #                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
# #                 f'--project "{project}" --command "{remote_command}"'
# #             )

# #             print(f"Executing client command: {command}")

# #             output = subprocess.call(command, shell=True)

# #             print(f"Return code for {instance_name}: {output}")
# #             return output

# #         # Start only the required number of client VMs for this load point.
# #         active_client_indices = [
# #             num_nodes + j for j in range(active_clients)
# #         ]

# #         results = run_parallel(
# #             run_stellar_client,
# #             active_client_indices,
# #             max_workers=active_clients
# #         )

# #         print(results)
# #         print(
# #             f"Started {active_clients} client VM(s), "
# #             f"each with {client_threads} client threads. "
# #             f"Total client threads = {total_client_threads}. "
# #             f"Aggregate max in-flight = {aggregate_max_in_flight}."
# #         )

# #         # Wait for the duration run to produce stable per-second client logs.
# #         # The clients may wait forever on final partial batches, so we kill them after this.
# #         time.sleep(CLIENT_WAIT_AFTER_START_SEC)
        

# #         # Stop all nodes and clients.
# #         results = run_parallel(
# #             kill_stellar_private,
# #             range(num_nodes + n_clients),
# #             max_workers=min(48, num_nodes + n_clients)
# #         )

# #         remote_base_folder = "/home/tejas/stellar-private"

# #         local_base_destination = (
# #             "/home/tejas/work/experiments/shabdiz/"
# #             + f"memory_no_cleanup_{num_nodes}_{run_label}"
# #         )

# #         Path(local_base_destination).mkdir(parents=True, exist_ok=True)

# #         # Save run metadata.
# #         with open(Path(local_base_destination) / "run_config.txt", "w") as f:
# #             f.write(f"num_nodes={num_nodes}\n")
# #             f.write(f"active_clients={active_clients}\n")
# #             f.write(f"client_threads_per_vm={client_threads}\n")
# #             f.write(f"total_client_threads={total_client_threads}\n")
# #             f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
# #             f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
# #             f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
# #             f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
# #             f.write(f"leader_ip={node1_ip}\n")
# #             f.write(f"machine_type={machine_type}\n")
# #             f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

# #         def copy_folder_from_instance(i):
# #             inst_zone = get_zone_for_instance(i)

# #             instance_name = f"tsm-sc-{i:03}"

# #             node_number = i + 1
# #             node_folder = f"node{node_number}"

# #             remote_source_path = posixpath.join(remote_base_folder, node_folder)

# #             local_destination_path = Path(local_base_destination) / instance_name
# #             local_destination_path.mkdir(parents=True, exist_ok=True)

# #             remote_source = f"{instance_name}:{remote_source_path}"

# #             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# # --recurse "{remote_source}" "{local_destination_path}"'''

# #             print(f"Executing command to copy {node_folder} from {instance_name}: {command}")

# #             output = subprocess.call(command, shell=True)

# #             print(f"Copy from {instance_name} finished with exit code: {output}")

# #             return (instance_name, output)

# #         # Copy only a few node logs to reduce time.
# #         # Change range(min(3, num_nodes)) to range(num_nodes) if you want all logs.
# #         node_copy_results = run_parallel(
# #             copy_folder_from_instance,
# #             range((num_nodes)),
# #             max_workers=min(48, max(1, min(3, num_nodes)))
# #         )

# #         def copy_client_log(i):
# #             inst_zone = get_zone_for_instance(i)

# #             instance_name = f"tsm-sc-{i:03}"
# #             client_id = i - num_nodes

# #             remote_source = (
# #                 f"{instance_name}:/home/tejas/stellar-private/"
# #                 f"stellar-client-{client_id}.log"
# #             )

# #             local_destination_path = Path(local_base_destination)
# #             local_destination_path.mkdir(parents=True, exist_ok=True)

# #             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# # "{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

# #             print(f"Copying client log from {instance_name}...")

# #             output = subprocess.call(command, shell=True)

# #             print(f"Copy finished with exit code: {output}")

# #             return (instance_name, output)

# #         client_copy_results = run_parallel(
# #             copy_client_log,
# #             active_client_indices,
# #             max_workers=active_clients
# #         )

# #         print("\n--- Summary of Download Results ---")
# #         print("Node log copies:", node_copy_results)
# #         print("Client log copies:", client_copy_results)
# #         print(f"Saved run to: {local_base_destination}")



#     do_compile_this_run = True#(not COMPILE_ONLY_FIRST_RUN) or (run_idx == 0)

#     if do_compile_this_run:
#         subprocess.call('git add .; git commit -m "testing"; git push', shell=True)

#         n_collection = 100
#         subprocess.call('make -j8', shell=True)

#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def git_pull_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# git pull"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         results = run_parallel(
#             git_pull_stellar,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )
#         print(results)

#         # ---------------------------------------------------------------------
#         # Compile only on the first/largest run.
#         # Since later runs reuse a subset of these VMs, no recompilation is needed.
#         # ---------------------------------------------------------------------
#         def compile_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# g++ -O2 -std=c++17 -pthread \
# -I/home/tejas/stellar-core/src \
# /home/tejas/stellar-core/shab_client.cpp \
# -o /home/tejas/stellar-core/shab_client; \
# make -j2; \
# cd; \
# sudo rm -rf stellar-private"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         results = run_parallel(
#             compile_stellar,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )


        
#         print(results)
#     else:
#         print(f"⏭️ Skipping git pull and compilation for num_nodes={num_nodes}; code already compiled on first run.")

#     # -------------------------------------------------------------------------
#     # Generate stellar-private configs locally using only replica IPs.
#     # -------------------------------------------------------------------------
#     stellar_private_path = Path('../stellar-private')
#     if stellar_private_path.exists():
#         shutil.rmtree(stellar_private_path)
#     stellar_private_path.mkdir()

#     subprocess.call(
#         'cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     subprocess.call(
#         'cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; '
#         './gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     # Enable custom message only on leader.
#     line_to_add = "SEND_CUSTOM_MESSAGE=true"
#     target_file = "../stellar-private/node1/stellar-core.cfg"

#     subprocess.call(
#         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#         shell=True
#     )

#     # Optional memory profiling on node2.
#     if num_nodes >= 2:
#         target_file = "../stellar-private/node2/stellar-core.cfg"
#         line_to_add = "MEMORY_PROF=true"

#         subprocess.call(
#             f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#             shell=True
#         )

#     #     print(f"The line '{line_to_add}' has been prepended to {target_file}.")

#     # -------------------------------------------------------------------------
#     # Throughput/latency experiment loop.
#     # Fixed offered load for scalability:
#     # active_clients=1, client_threads=2, max_in_flight=100.
#     # Aggregate max in-flight = 200.
#     # -------------------------------------------------------------------------
#     for load in LOAD_POINTS:
#         active_clients = load["active_clients"]
#         client_threads = load["client_threads"]
#         client_max_in_flight = load["max_in_flight"]

#         if active_clients > n_clients:
#             raise RuntimeError(
#                 f"active_clients={active_clients} exceeds n_clients={n_clients}"
#             )

#         total_client_threads = active_clients * client_threads
#         aggregate_max_in_flight = active_clients * client_threads * client_max_in_flight

#         run_label = (
#             f"clients_{active_clients}_threads_{client_threads}_"
#             f"inflight_{client_max_in_flight}_"
#             f"total_threads_{total_client_threads}_"
#             f"total_inflight_{aggregate_max_in_flight}"
#         )

#         print("\n" + "=" * 80)
#         print(f"🚀 Starting throughput/latency run: num_nodes={num_nodes}, {run_label}")
#         print("=" * 80)

#         def clean_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             remote_command = f"""\
# cd /home/tejas; \
# sudo rm -rf stellar-private; \
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")
#             output = subprocess.call(command, shell=True)
#             print(f"Return code for tsm-sc-{i:03}: {output}")
#             return output

#         results = run_parallel(
#             clean_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def copy_folder_to_instance(
#             i,
#             source_folder="/home/tejas/stellar-private",
#             destination_path="/home/tejas/stellar-private"
#         ):
#             inst_zone = get_zone_for_instance(i)
#             instance_name = f"tsm-sc-{i:03}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{source_folder}" "{instance_name}:{destination_path}"'''

#             print(f"Executing command for {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Command for {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         results = run_parallel(
#             copy_folder_to_instance,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def run_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             node_number = i + 1
#             instance_name = f"tsm-sc-{i:03}"

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Kill old processes on all replica and client machines.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         # Start consensus replicas.
#         results = run_parallel(
#             run_stellar_private,
#             range(num_nodes),
#             max_workers=min(48, num_nodes)
#         )

#         print(results)
#         print("All Stellar nodes should be starting in the background.")

#         # Give nodes time to authenticate and start the client listener.
#         time.sleep(80)

#         def run_stellar_client(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
# {client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
# {CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
# > stellar-client-{client_id}.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing client command: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Start only the required number of client VMs for this load point.
#         active_client_indices = [
#             num_nodes + j for j in range(active_clients)
#         ]

#         results = run_parallel(
#             run_stellar_client,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print(results)
#         print(
#             f"Started {active_clients} client VM(s), "
#             f"each with {client_threads} client threads. "
#             f"Total client threads = {total_client_threads}. "
#             f"Aggregate max in-flight = {aggregate_max_in_flight}."
#         )

#         # Wait for the duration run to produce stable per-second client logs.
#         # The clients may wait forever on final partial batches, so we kill them after this.
#         time.sleep(CLIENT_WAIT_AFTER_START_SEC)
        

#         # Stop all nodes and clients.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         remote_base_folder = "/home/tejas/stellar-private"

#         local_base_destination = (
#             "/home/tejas/work/experiments/shabdiz/"
#             + f"memory_withv3_cleanup_{num_nodes}_{run_label}"
#         )

#         Path(local_base_destination).mkdir(parents=True, exist_ok=True)

#         # Save run metadata.
#         with open(Path(local_base_destination) / "run_config.txt", "w") as f:
#             f.write(f"num_nodes={num_nodes}\n")
#             f.write(f"active_clients={active_clients}\n")
#             f.write(f"client_threads_per_vm={client_threads}\n")
#             f.write(f"total_client_threads={total_client_threads}\n")
#             f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
#             f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
#             f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
#             f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
#             f.write(f"leader_ip={node1_ip}\n")
#             f.write(f"machine_type={machine_type}\n")
#             f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

#         def copy_folder_from_instance(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"

#             node_number = i + 1
#             node_folder = f"node{node_number}"

#             remote_source_path = posixpath.join(remote_base_folder, node_folder)

#             local_destination_path = Path(local_base_destination) / instance_name
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             remote_source = f"{instance_name}:{remote_source_path}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{remote_source}" "{local_destination_path}"'''

#             print(f"Executing command to copy {node_folder} from {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy from {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         # Copy only a few node logs to reduce time.
#         # Change range(min(3, num_nodes)) to range(num_nodes) if you want all logs.
#         node_copy_results = run_parallel(
#             copy_folder_from_instance,
#             range((num_nodes)),
#             max_workers=min(48, max(1, min(3, num_nodes)))
#         )

#         def copy_client_log(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_source = (
#                 f"{instance_name}:/home/tejas/stellar-private/"
#                 f"stellar-client-{client_id}.log"
#             )

#             local_destination_path = Path(local_base_destination)
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

#             print(f"Copying client log from {instance_name}...")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy finished with exit code: {output}")

#             return (instance_name, output)

#         client_copy_results = run_parallel(
#             copy_client_log,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print("\n--- Summary of Download Results ---")
#         print("Node log copies:", node_copy_results)
#         print("Client log copies:", client_copy_results)
#         print(f"Saved run to: {local_base_destination}")

# PBFT Test

In [1]:
# import subprocess
# import concurrent.futures
# import posixpath
# import shutil
# import time
# from pathlib import Path


# def run_shell(command):
#     return subprocess.call(command, shell=True)


# def run_parallel(func, iterable, max_workers):
#     items = list(iterable)
#     if not items:
#         return []
#     with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
#         return list(executor.map(func, items))


# # latencies: 50, 90 150, 210
# default_region = ['us-central1-a']
# # regions = ['us-west1-b', 'us-east5-c', 'asia-northeast1-b', 'europe-west3-c', 'asia-south1-c']
# # regions = ['us-west1-b','asia-northeast1-b', 'asia-south1-c']

# regions = ['us-central1-a', 'us-central1-a', 'us-central1-a', 'us-central1-a']


# # Regions

# zone_no = 0

# # Run throughput/latency vs num_nodes for these system sizes.
# NUM_NODES_LIST = [4]

# # Use 1 extra 2-core machine as client machine.
# # For num_nodes = N, client is tsm-sc-N.
# n_clients = 1

# # Start clean only once, before the largest run.
# # After that, keep the lower-index VMs and delete only the extra higher-index VMs.
# DELETE_BEFORE_FIRST_RUN = False

# # Since NUM_NODES_LIST goes 48 -> 32 -> 16 -> 8 -> 4, delete only the VMs
# # that will not be needed by the next smaller run.
# DELETE_UNUSED_AFTER_EACH_RUN = False

# # Delete the remaining 4 replica VMs + 1 client VM after the final run.
# DELETE_ALL_AFTER_FINAL_RUN = False

# # Code is unchanged across node-count runs, so compile only on the first run.
# COMPILE_ONLY_FIRST_RUN = True

# MAX_NUM_NODES = max(NUM_NODES_LIST)

# # Client experiment settings.
# CLIENT_DURATION_SEC = 360
# CLIENT_WAIT_AFTER_START_SEC = 160
# CLIENT_TOTAL_REQUESTS = 100000000
# SERVER_BATCH_SIZE_HINT = 100
# SEND_INTERVAL_US = 0

# # Fixed load point for throughput-vs-num_nodes and latency-vs-num_nodes.
# # Aggregate max in-flight = active_clients * client_threads * max_in_flight = 200.
# LOAD_POINTS = [
#     {"active_clients": 1, "client_threads": 2, "max_in_flight": 150},   # total inflight 300
# ]


# for run_idx, num_nodes in enumerate(NUM_NODES_LIST):
# # for zone_no in  [0,1,2,3, 4]:

#     project = "research-488322"
#     zone = "us-central1-a"
#     machine_type = "e2-standard-2"
#     image_family = "tsm-sc-family"  # your custom image
#     subnet = "default"
#     gcp_username = "tejas"

#     print("\n" + "#" * 100)
#     print(f"Starting experiment for num_nodes={num_nodes}")
#     print("#" * 100)

#     def get_zone_for_instance(i):
#         # IMPORTANT: use MAX_NUM_NODES instead of current num_nodes.
#         # We reuse VMs while moving 48 -> 32 -> 16 -> 8 -> 4, so the zone for
#         # tsm-sc-016, tsm-sc-032, etc. must stay the same across runs.
#         if i < int(MAX_NUM_NODES / 2):
#             return default_region[0]
#         else:
#             return regions[zone_no]

#     def fetch_existing_instances():
#         fetch_cmd = f'''
#         gcloud compute instances list \
#             --project={project} \
#             --filter="name~'^tsm-sc-'" \
#             --format="value(name,zone)"
#         '''

#         output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
#         instances = []

#         for line in output.splitlines():
#             if line.strip():
#                 name, inst_zone = line.split()
#                 instances.append((name, inst_zone))

#         return instances

#     def delete_instance(instance):
#         name, inst_zone = instance

#         cmd = f'''
#         gcloud compute instances delete {name} \
#             --zone={inst_zone} \
#             --project={project} \
#             --quiet
#         '''

#         print(f"🗑️ Deleting {name} in {inst_zone}")
#         return subprocess.call(cmd, shell=True)

#     def parse_tsm_index(name):
#         return int(name.rsplit("-", 1)[1])

#     # -------------------------------------------------------------------------
#     # Delete existing tsm-sc-* instances only before the first/largest run.
#     # Later runs reuse tsm-sc-000 ... tsm-sc-(next_num_nodes).
#     # -------------------------------------------------------------------------
#     if DELETE_BEFORE_FIRST_RUN and run_idx == 0:
#         instances = fetch_existing_instances()

#         print("\n➡ Existing instances to delete before first run:")
#         for name, inst_zone in instances:
#             print(f"  - {name} ({inst_zone})")

#         if instances:
#             run_parallel(
#                 delete_instance,
#                 instances,
#                 max_workers=min(32, len(instances))
#             )
#             print("\n🧹 All existing tsm-sc-* instances deleted before first run.\n")
#         else:
#             print("\n✔ No existing tsm-sc-* instances found before first run.\n")

#     # -------------------------------------------------------------------------
#     # Create only the missing replica/client machines for this num_nodes run.
#     # When moving 48 -> 32 -> 16 -> 8 -> 4, the lower-index machines are reused.
#     # Replicas: tsm-sc-000 ... tsm-sc-(num_nodes-1)
#     # Client:   tsm-sc-num_nodes
#     # -------------------------------------------------------------------------
#     existing_instances = fetch_existing_instances()
#     existing_names = {name for name, _ in existing_instances}

#     commands = []
#     newly_created_indices = []

#     # Create replica nodes only if they do not already exist.
#     for i in range(num_nodes):
#         instance_name = f"tsm-sc-{i:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing replica {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(i)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(i)

#     # Create client machine only if it does not already exist.
#     for i in range(n_clients):
#         client_idx = num_nodes + i
#         instance_name = f"tsm-sc-{client_idx:03}"

#         if instance_name in existing_names:
#             print(f"✔ Reusing existing client {instance_name}")
#             continue

#         inst_zone = get_zone_for_instance(client_idx)

#         cmd = f'''
#         gcloud compute instances create {instance_name} \
#             --project={project} \
#             --zone={inst_zone} \
#             --machine-type={machine_type} \
#             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
#             --can-ip-forward \
#             --maintenance-policy=MIGRATE \
#             --provisioning-model=STANDARD \
#             --service-account=254510644191-compute@developer.gserviceaccount.com \
#             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#             --tags=http-server,https-server \
#             --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
#             --no-shielded-secure-boot \
#             --shielded-vtpm \
#             --shielded-integrity-monitoring \
#             --labels=goog-ec-src=vm_add-gcloud \
#             --reservation-affinity=any
#         '''
#         commands.append(cmd.strip())
#         newly_created_indices.append(client_idx)

#     def run_command(command):
#         print(f"Running: {command}")
#         return subprocess.call(command, shell=True)

#     if commands:
#         run_parallel(
#             run_command,
#             commands,
#             max_workers=min(48, len(commands))
#         )

#         print("All missing instances launched.")

#         # Give GCP/SSH a little time after VM creation.
#         time.sleep(30)
#     else:
#         print("✔ All required instances already exist; no VM creation needed.")

#     # -------------------------------------------------------------------------
#     # Get sorted node and client IPs.
#     # IMPORTANT: tsm_ips.txt must include only server/replica IPs.
#     # Client IPs must not be included.
#     # -------------------------------------------------------------------------
#     ip_cmd = f'''
#     gcloud compute instances list \
#         --project={project} \
#         --filter="name~'^tsm-sc-'" \
#         --sort-by=name \
#         --format="value(name,zone,networkInterfaces[0].networkIP)"
#     '''

#     ip_output = subprocess.check_output(ip_cmd, shell=True).decode().strip()

#     instance_records = []

#     for line in ip_output.splitlines():
#         if line.strip():
#             name, inst_zone, ip = line.split()
#             idx = int(name.rsplit("-", 1)[1])
#             instance_records.append((idx, name, inst_zone, ip))

#     instance_records.sort()

#     node_records = [r for r in instance_records if r[0] < num_nodes]
#     client_records = [r for r in instance_records if num_nodes <= r[0] < num_nodes + n_clients]

#     if len(node_records) != num_nodes:
#         raise RuntimeError(
#             f"Expected {num_nodes} replica nodes, but found {len(node_records)}: {node_records}"
#         )

#     if len(client_records) != n_clients:
#         raise RuntimeError(
#             f"Expected {n_clients} client nodes, but found {len(client_records)}: {client_records}"
#         )

#     iplist = [r[3] for r in node_records]

#     with open("tsm_ips.txt", "w") as f:
#         for ip in iplist:
#             f.write(ip + "\n")

#     print("🎯 Node IPs:", iplist)
#     print("🎯 Client instances:", client_records)

#     node1_ip = iplist[0]
#     print(f"Clients will connect to leader/node1 at: {node1_ip}")

#     # -------------------------------------------------------------------------
#     # Push/pull/compile only for the first run.
#     # Later runs reuse the same lower-index VMs, and the code has not changed.
#     # -------------------------------------------------------------------------
#     def kill_stellar_private(i):
#         inst_zone = get_zone_for_instance(i)

#         remote_command = f"""\
# cd /home/tejas/stellar-private; \
# sudo pkill -9 stellar-core; sudo pkill -9 shab_client; \
# """

#         command = (
#             f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#             f'--project "{project}" --command "{remote_command}"'
#         )

#         print(f"Executing: {command}")
#         output = subprocess.call(command, shell=True)
#         print(f"Return code for tsm-sc-{i:03}: {output}")
#         return output




#     do_compile_this_run = True#(not COMPILE_ONLY_FIRST_RUN) or (run_idx == 0)

#     if do_compile_this_run:
#         subprocess.call('git add .; git commit -m "testing"; git push', shell=True)

#         n_collection = 100
#         subprocess.call('make -j8', shell=True)

#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def git_pull_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# git pull"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         results = run_parallel(
#             git_pull_stellar,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )
#         print(results)

#         # ---------------------------------------------------------------------
#         # Compile only on the first/largest run.
#         # Since later runs reuse a subset of these VMs, no recompilation is needed.
#         # ---------------------------------------------------------------------
#         def compile_stellar(i):
#             inst_zone = get_zone_for_instance(i)

#             command = f'''gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd stellar-core; \
# g++ -O2 -std=c++17 -pthread \
# -I/home/tejas/stellar-core/src \
# /home/tejas/stellar-core/shab_client.cpp \
# -o /home/tejas/stellar-core/shab_client; \
# make -j2; \
# cd; \
# sudo rm -rf stellar-private"'''

#             print(command)
#             output = subprocess.call(command, shell=True)
#             print(output)
#             return output

#         results = run_parallel(
#             compile_stellar,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )
        
#         print(results)
#     else:
#         print(f"⏭️ Skipping git pull and compilation for num_nodes={num_nodes}; code already compiled on first run.")

#     # -------------------------------------------------------------------------
#     # Generate stellar-private configs locally using only replica IPs.
#     # -------------------------------------------------------------------------
#     stellar_private_path = Path('../stellar-private')
#     if stellar_private_path.exists():
#         shutil.rmtree(stellar_private_path)
#     stellar_private_path.mkdir()

#     subprocess.call(
#         'cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     subprocess.call(
#         'cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; '
#         './gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh',
#         shell=True
#     )

#     # Enable custom message only on leader.
#     line_to_add = "SEND_CUSTOM_MESSAGE=true"
#     target_file = "../stellar-private/node1/stellar-core.cfg"

#     subprocess.call(
#         f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}',
#         shell=True
#     )


#     # -------------------------------------------------------------------------
#     # Throughput/latency experiment loop.
#     # Fixed offered load for scalability:
#     # active_clients=1, client_threads=2, max_in_flight=100.
#     # Aggregate max in-flight = 200.
#     # -------------------------------------------------------------------------
#     for load in LOAD_POINTS:
#         active_clients = load["active_clients"]
#         client_threads = load["client_threads"]
#         client_max_in_flight = load["max_in_flight"]

#         if active_clients > n_clients:
#             raise RuntimeError(
#                 f"active_clients={active_clients} exceeds n_clients={n_clients}"
#             )

#         total_client_threads = active_clients * client_threads
#         aggregate_max_in_flight = active_clients * client_threads * client_max_in_flight

#         run_label = (
#             f"clients_{active_clients}_threads_{client_threads}_"
#             f"inflight_{client_max_in_flight}_"
#             f"total_threads_{total_client_threads}_"
#             f"total_inflight_{aggregate_max_in_flight}"
#         )

#         print("\n" + "=" * 80)
#         print(f"🚀 Starting throughput/latency run: num_nodes={num_nodes}, {run_label}")
#         print("=" * 80)

#         def clean_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             remote_command = f"""\
# cd /home/tejas; \
# sudo rm -rf stellar-private; \
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "tsm-sc-{i:03}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")
#             output = subprocess.call(command, shell=True)
#             print(f"Return code for tsm-sc-{i:03}: {output}")
#             return output



#         results = run_parallel(
#             clean_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def copy_folder_to_instance(
#             i,
#             source_folder="/home/tejas/stellar-private",
#             destination_path="/home/tejas/stellar-private"
#         ):
#             inst_zone = get_zone_for_instance(i)
#             instance_name = f"tsm-sc-{i:03}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{source_folder}" "{instance_name}:{destination_path}"'''

#             print(f"Executing command for {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Command for {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         results = run_parallel(
#             copy_folder_to_instance,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         def run_stellar_private(i):
#             inst_zone = get_zone_for_instance(i)

#             node_number = i + 1
#             instance_name = f"tsm-sc-{i:03}"

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Kill old processes on all replica and client machines.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         # Start consensus replicas.
#         results = run_parallel(
#             run_stellar_private,
#             range(num_nodes),
#             max_workers=min(48, num_nodes)
#         )

#         print(results)
#         print("All Stellar nodes should be starting in the background.")

#         # Give nodes time to authenticate and start the client listener.
#         time.sleep(40)

#         def run_stellar_client(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/shab_client {node1_ip} 12000 \
# {client_max_in_flight} {CLIENT_TOTAL_REQUESTS} {SERVER_BATCH_SIZE_HINT} \
# {CLIENT_DURATION_SEC} {SEND_INTERVAL_US} {client_threads} \
# > stellar-client-{client_id}.log 2>&1 < /dev/null & disown
# """

#             command = (
#                 f'gcloud compute ssh --zone "{inst_zone}" "{instance_name}" '
#                 f'--project "{project}" --command "{remote_command}"'
#             )

#             print(f"Executing client command: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Return code for {instance_name}: {output}")
#             return output

#         # Start only the required number of client VMs for this load point.
#         active_client_indices = [
#             num_nodes + j for j in range(active_clients)
#         ]

#         results = run_parallel(
#             run_stellar_client,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print(results)
#         print(
#             f"Started {active_clients} client VM(s), "
#             f"each with {client_threads} client threads. "
#             f"Total client threads = {total_client_threads}. "
#             f"Aggregate max in-flight = {aggregate_max_in_flight}."
#         )

#         # Wait for the duration run to produce stable per-second client logs.
#         # The clients may wait forever on final partial batches, so we kill them after this.
#         time.sleep(CLIENT_WAIT_AFTER_START_SEC)
        

#         # Stop all nodes and clients.
#         results = run_parallel(
#             kill_stellar_private,
#             range(num_nodes + n_clients),
#             max_workers=min(48, num_nodes + n_clients)
#         )

#         remote_base_folder = "/home/tejas/stellar-private"

#         local_base_destination = (
#             "/home/tejas/work/experiments/shabdiz/"
#             + f"SCP_test_{num_nodes}_{run_label}"
#         )

#         Path(local_base_destination).mkdir(parents=True, exist_ok=True)

#         # Save run metadata.
#         with open(Path(local_base_destination) / "run_config.txt", "w") as f:
#             f.write(f"num_nodes={num_nodes}\n")
#             f.write(f"active_clients={active_clients}\n")
#             f.write(f"client_threads_per_vm={client_threads}\n")
#             f.write(f"total_client_threads={total_client_threads}\n")
#             f.write(f"client_max_in_flight_per_thread={client_max_in_flight}\n")
#             f.write(f"client_duration_sec={CLIENT_DURATION_SEC}\n")
#             f.write(f"send_interval_us={SEND_INTERVAL_US}\n")
#             f.write(f"server_batch_size_hint={SERVER_BATCH_SIZE_HINT}\n")
#             f.write(f"leader_ip={node1_ip}\n")
#             f.write(f"machine_type={machine_type}\n")
#             f.write(f"aggregate_max_in_flight={aggregate_max_in_flight}\n")

#         def copy_folder_from_instance(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"

#             node_number = i + 1
#             node_folder = f"node{node_number}"

#             remote_source_path = posixpath.join(remote_base_folder, node_folder)

#             local_destination_path = Path(local_base_destination) / instance_name
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             remote_source = f"{instance_name}:{remote_source_path}"

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# --recurse "{remote_source}" "{local_destination_path}"'''

#             print(f"Executing command to copy {node_folder} from {instance_name}: {command}")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy from {instance_name} finished with exit code: {output}")

#             return (instance_name, output)

#         # Copy only a few node logs to reduce time.
#         # Change range(min(3, num_nodes)) to range(num_nodes) if you want all logs.
#         node_copy_results = run_parallel(
#             copy_folder_from_instance,
#             range((num_nodes)),
#             max_workers=min(48, max(1, min(3, num_nodes)))
#         )

#         def copy_client_log(i):
#             inst_zone = get_zone_for_instance(i)

#             instance_name = f"tsm-sc-{i:03}"
#             client_id = i - num_nodes

#             remote_source = (
#                 f"{instance_name}:/home/tejas/stellar-private/"
#                 f"stellar-client-{client_id}.log"
#             )

#             local_destination_path = Path(local_base_destination)
#             local_destination_path.mkdir(parents=True, exist_ok=True)

#             command = f'''gcloud compute scp --zone "{inst_zone}" --project "{project}" \
# "{remote_source}" "{local_destination_path}/client_{client_id}.log"'''

#             print(f"Copying client log from {instance_name}...")

#             output = subprocess.call(command, shell=True)

#             print(f"Copy finished with exit code: {output}")

#             return (instance_name, output)

#         client_copy_results = run_parallel(
#             copy_client_log,
#             active_client_indices,
#             max_workers=active_clients
#         )

#         print("\n--- Summary of Download Results ---")
#         print("Node log copies:", node_copy_results)
#         print("Client log copies:", client_copy_results)
#         print(f"Saved run to: {local_base_destination}")


####################################################################################################
Starting experiment for num_nodes=4
####################################################################################################


Running: gcloud compute instances create tsm-sc-000             --project=research-488322             --zone=us-central1-a             --machine-type=e2-standard-2             --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default             --can-ip-forward             --maintenance-policy=MIGRATE             --provisioning-model=STANDARD             --service-account=254510644191-compute@developer.gserviceaccount.com             --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append             --tags=http-server,https-server             --create-disk=auto-delete=yes,boot=yes,image-family=tsm-sc-family,mode=rw,size=20,type=pd-balanced             --no-shielded-secure-boot             --shielded-vtpm             

Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-004].
Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-001].
 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert

 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert



NAME        ZONE           MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP    STATUS
tsm-sc-004  us-central1-a  e2-standard-2               10.128.0.11  34.134.66.176  RUNNING
NAME        ZONE           MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP    STATUS
tsm-sc-001  us-central1-a  e2-standard-2               10.128.0.16  34.68.215.172  RUNNING


Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-a/instances/tsm-sc-003].
 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert



NAME        ZONE           MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP     STATUS
tsm-sc-003  us-central1-a  e2-standard-2               10.128.0.29  35.188.171.175  RUNNING




Command killed by keyboard interrupt



Command killed by keyboard interrupt



KeyboardInterrupt: 

# SCP

In [36]:
# import subprocess
# import concurrent.futures
# import shutil
# import time
# import re
# from pathlib import Path
# from datetime import datetime


# # =============================================================================
# # Basic helpers
# # =============================================================================

# def run_shell(command, check=False):
#     print(f"\n$ {command}")
#     rc = subprocess.call(command, shell=True)
#     if check and rc != 0:
#         raise RuntimeError(f"Command failed with code {rc}: {command}")
#     return rc


# def run_shell_output(command):
#     print(f"\n$ {command}")
#     return subprocess.check_output(command, shell=True).decode()


# def run_parallel(func, iterable, max_workers):
#     items = list(iterable)
#     if not items:
#         return []
#     with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
#         return list(executor.map(func, items))


# # =============================================================================
# # Experiment config
# # =============================================================================

# PROJECT = "research-488322"
# ZONE = "us-central1-b"
# MACHINE_TYPE = "e2-standard-2"
# IMAGE_FAMILY = "tsm-sc-family"
# SUBNET = "default"
# GCP_USERNAME = "tejas"

# # Run in decreasing order so we can delete unused nodes after each subrun.
# NUM_NODES_LIST = [48, 32, 16, 8, 4]
# MAX_NUM_NODES = max(NUM_NODES_LIST)

# N_CLIENTS = 0

# # You said NUM_TEST_ACCOUNTS is already set to 100 in C++.
# NUM_TEST_ACCOUNTS = 100

# # This is only used for summary labels. It is not patched by this script.
# SCP_OPS_PER_TX = 100

# RUN_DURATION_SEC = 210
# MEASURE_START_SEC = 120
# MEASURE_END_SEC = 180

# LOCAL_PRIVATE_DIR = Path("../stellar-private")
# RESULTS_DIR = Path("/home/tejas/work/experiments/stellar_native")
# RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# DELETE_BEFORE_FIRST_RUN = False

# # Final cleanup deletes whatever remains after the 4-node run.
# DELETE_ALL_AFTER_FINAL_RUN = True

# # Compile only during the first subrun.
# COMPILE_ONLY_FIRST_SUBRUN = True


# # =============================================================================
# # GCP helpers
# # =============================================================================

# def instance_name(i):
#     return f"tsm-sc-{i:03}"


# def get_zone_for_instance(i):
#     return ZONE


# def fetch_existing_instances():
#     cmd = f"""
#     gcloud compute instances list \
#         --project={PROJECT} \
#         --filter="name~'^tsm-sc-'" \
#         --format="value(name,zone)"
#     """
#     output = subprocess.check_output(cmd, shell=True).decode().strip()
#     instances = []

#     for line in output.splitlines():
#         if line.strip():
#             name, inst_zone = line.split()
#             instances.append((name, inst_zone))

#     return instances


# def delete_instance(instance):
#     name, inst_zone = instance
#     cmd = f"""
#     gcloud compute instances delete {name} \
#         --zone={inst_zone} \
#         --project={PROJECT} \
#         --quiet
#     """
#     print(f"Deleting {name} in {inst_zone}")
#     return subprocess.call(cmd, shell=True)


# def delete_instance_by_index(i):
#     name = instance_name(i)
#     inst_zone = get_zone_for_instance(i)

#     cmd = f"""
#     gcloud compute instances delete {name} \
#         --zone={inst_zone} \
#         --project={PROJECT} \
#         --quiet
#     """
#     print(f"Deleting unused node {name}")
#     return subprocess.call(cmd, shell=True)


# def delete_nodes_not_needed_next(current_num_nodes, next_num_nodes):
#     """
#     Because NUM_NODES_LIST is descending:
#       after 48-node run, delete 32..47
#       after 32-node run, delete 16..31
#       after 16-node run, delete 8..15
#       after 8-node run, delete 4..7
#     """
#     if next_num_nodes is None:
#         return

#     to_delete = list(range(next_num_nodes, current_num_nodes))

#     if not to_delete:
#         return

#     print(
#         f"Deleting nodes no longer needed for future subruns: "
#         f"{instance_name(to_delete[0])} to {instance_name(to_delete[-1])}"
#     )

#     run_parallel(
#         delete_instance_by_index,
#         to_delete,
#         max_workers=min(32, len(to_delete)),
#     )


# def create_instance(i):
#     name = instance_name(i)
#     inst_zone = get_zone_for_instance(i)

#     cmd = f"""
#     gcloud compute instances create {name} \
#         --project={PROJECT} \
#         --zone={inst_zone} \
#         --machine-type={MACHINE_TYPE} \
#         --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={SUBNET} \
#         --can-ip-forward \
#         --maintenance-policy=MIGRATE \
#         --provisioning-model=STANDARD \
#         --service-account=254510644191-compute@developer.gserviceaccount.com \
#         --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
#         --tags=http-server,https-server \
#         --create-disk=auto-delete=yes,boot=yes,image-family={IMAGE_FAMILY},mode=rw,size=20,type=pd-balanced \
#         --no-shielded-secure-boot \
#         --shielded-vtpm \
#         --shielded-integrity-monitoring \
#         --labels=goog-ec-src=vm_add-gcloud \
#         --reservation-affinity=any
#     """
#     print(f"Creating {name}")
#     return subprocess.call(cmd, shell=True)


# def ensure_instances():
#     existing = fetch_existing_instances()
#     existing_names = {name for name, _ in existing}

#     needed_indices = list(range(MAX_NUM_NODES + N_CLIENTS))
#     missing = [i for i in needed_indices if instance_name(i) not in existing_names]

#     if not missing:
#         print("All required instances already exist.")
#         return

#     print("Missing instances:", [instance_name(i) for i in missing])
#     run_parallel(create_instance, missing, max_workers=min(32, len(missing)))

#     print("Waiting after VM creation...")
#     time.sleep(30)


# def get_node_records(num_nodes):
#     cmd = f"""
#     gcloud compute instances list \
#         --project={PROJECT} \
#         --filter="name~'^tsm-sc-'" \
#         --sort-by=name \
#         --format="value(name,zone,networkInterfaces[0].networkIP)"
#     """
#     output = subprocess.check_output(cmd, shell=True).decode().strip()

#     records = []
#     for line in output.splitlines():
#         if line.strip():
#             name, inst_zone, ip = line.split()
#             idx = int(name.rsplit("-", 1)[1])
#             records.append((idx, name, inst_zone, ip))

#     records.sort()
#     node_records = [r for r in records if r[0] < num_nodes]

#     if len(node_records) != num_nodes:
#         raise RuntimeError(
#             f"Expected {num_nodes} nodes, found {len(node_records)}: {node_records}"
#         )

#     return node_records


# def kill_stellar(i):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)

#     remote_command = r"""\
# cd /home/tejas; \
# sudo pkill -9 stellar-core || true; \
# sudo pkill -9 shab_client || true; \
# """

#     cmd = (
#         f'gcloud compute ssh --zone "{inst_zone}" "{name}" '
#         f'--project "{PROJECT}" --command "{remote_command}"'
#     )
#     return subprocess.call(cmd, shell=True)


# def git_pull_node(i):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)

#     remote_command = r"""\
# cd /home/tejas/stellar-core; \
# git pull
# """

#     cmd = (
#         f'gcloud compute ssh --zone "{inst_zone}" "{name}" '
#         f'--project "{PROJECT}" --command "{remote_command}"'
#     )
#     return subprocess.call(cmd, shell=True)


# def compile_node(i):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)

#     remote_command = r"""\
# cd /home/tejas/stellar-core; \
# make -j2; \
# cd /home/tejas; \
# sudo rm -rf stellar-private
# """

#     cmd = (
#         f'gcloud compute ssh --zone "{inst_zone}" "{name}" '
#         f'--project "{PROJECT}" --command "{remote_command}"'
#     )
#     return subprocess.call(cmd, shell=True)


# def clean_remote_private(i):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)

#     remote_command = r"""\
# cd /home/tejas; \
# sudo rm -rf stellar-private
# """

#     cmd = (
#         f'gcloud compute ssh --zone "{inst_zone}" "{name}" '
#         f'--project "{PROJECT}" --command "{remote_command}"'
#     )
#     return subprocess.call(cmd, shell=True)


# def copy_private_to_node(i):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)

#     cmd = f"""gcloud compute scp --zone "{inst_zone}" --project "{PROJECT}" \
# --recurse "/home/tejas/stellar-private" "{name}:/home/tejas/stellar-private"
# """
#     return subprocess.call(cmd, shell=True)


# def run_stellar_node(i):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)
#     node_number = i + 1

#     remote_command = f"""\
# cd /home/tejas/stellar-private; \
# nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
# > node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
# """

#     cmd = (
#         f'gcloud compute ssh --zone "{inst_zone}" "{name}" '
#         f'--project "{PROJECT}" --command "{remote_command}"'
#     )
#     return subprocess.call(cmd, shell=True)


# def collect_node_log(i, local_run_dir):
#     inst_zone = get_zone_for_instance(i)
#     name = instance_name(i)
#     node_number = i + 1

#     local_run_dir.mkdir(parents=True, exist_ok=True)
#     local_path = local_run_dir / f"{name}-node{node_number}-stellar-core.log"

#     cmd = f"""gcloud compute scp --zone "{inst_zone}" --project "{PROJECT}" \
# "{name}:/home/tejas/stellar-private/node{node_number}/stellar-core.log" \
# "{local_path}"
# """
#     return subprocess.call(cmd, shell=True)


# # =============================================================================
# # Config generation
# # =============================================================================

# def write_tsm_ips(node_ips):
#     with open("tsm_ips.txt", "w") as f:
#         for ip in node_ips:
#             f.write(ip + "\n")


# def generate_stellar_private_configs(num_nodes):
#     if LOCAL_PRIVATE_DIR.exists():
#         shutil.rmtree(LOCAL_PRIVATE_DIR)

#     LOCAL_PRIVATE_DIR.mkdir(parents=True)

#     run_shell(
#         "cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh",
#         check=True,
#     )

#     run_shell(
#         "cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; "
#         "./gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh",
#         check=True,
#     )

#     # Only node1 injects native Stellar workload.
#     # ENABLE_SCP_TRACKING is already true in C++.
#     for node_number in range(1, num_nodes + 1):
#         cfg = LOCAL_PRIVATE_DIR / f"node{node_number}" / "stellar-core.cfg"

#         if not cfg.exists():
#             raise FileNotFoundError(f"Missing config: {cfg}")

#         lines = cfg.read_text().splitlines()
#         lines = [
#             line for line in lines
#             if not line.strip().startswith("SEND_CUSTOM_MESSAGE=")
#         ]

#         if node_number == 1:
#             lines.insert(0, "SEND_CUSTOM_MESSAGE=true")
#         else:
#             lines.insert(0, "SEND_CUSTOM_MESSAGE=false")

#         cfg.write_text("\n".join(lines) + "\n")


# # =============================================================================
# # Log parsing
# # =============================================================================

# def parse_ts(line):
#     # Example:
#     # 2026-06-30T00:09:11.842 GDQOM [Ledger INFO] Got consensus...
#     m = re.match(r"^(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(?:\.\d+)?)", line)
#     if not m:
#         return None

#     try:
#         return datetime.fromisoformat(m.group(1))
#     except ValueError:
#         return None


# def parse_committed_ops(log_path, start_sec=MEASURE_START_SEC, end_sec=MEASURE_END_SEC):
#     entries = []

#     with open(log_path, "r", errors="ignore") as f:
#         for line in f:
#             if "Got consensus:" not in line:
#                 continue

#             ts = parse_ts(line)
#             if ts is None:
#                 continue

#             m_ops = re.search(r"ops=(\d+)", line)
#             m_txs = re.search(r"txs=(\d+)", line)

#             if not m_ops:
#                 continue

#             ops = int(m_ops.group(1))
#             txs = int(m_txs.group(1)) if m_txs else 0

#             entries.append((ts, txs, ops, line.rstrip()))

#     if not entries:
#         return {
#             "ops": 0,
#             "txs": 0,
#             "throughput_ops_s": 0.0,
#             "avg_ops_per_ledger": 0.0,
#             "avg_txs_per_ledger": 0.0,
#             "min_ops_per_ledger": 0,
#             "max_ops_per_ledger": 0,
#             "min_txs_per_ledger": 0,
#             "max_txs_per_ledger": 0,
#             "num_ledgers": 0,
#             "first_ts": None,
#             "last_ts": None,
#         }

#     t0 = entries[0][0]
#     selected = []

#     for ts, txs, ops, line in entries:
#         elapsed = (ts - t0).total_seconds()
#         if start_sec <= elapsed <= end_sec:
#             selected.append((ts, txs, ops, line))

#     total_ops = sum(ops for _, _, ops, _ in selected)
#     total_txs = sum(txs for _, txs, _, _ in selected)
#     num_ledgers = len(selected)
#     window = max(1.0, end_sec - start_sec)

#     ops_values = [ops for _, _, ops, _ in selected]
#     txs_values = [txs for _, txs, _, _ in selected]

#     return {
#         "ops": total_ops,
#         "txs": total_txs,
#         "throughput_ops_s": total_ops / window,
#         "avg_ops_per_ledger": total_ops / num_ledgers if num_ledgers else 0.0,
#         "avg_txs_per_ledger": total_txs / num_ledgers if num_ledgers else 0.0,
#         "min_ops_per_ledger": min(ops_values) if ops_values else 0,
#         "max_ops_per_ledger": max(ops_values) if ops_values else 0,
#         "min_txs_per_ledger": min(txs_values) if txs_values else 0,
#         "max_txs_per_ledger": max(txs_values) if txs_values else 0,
#         "num_ledgers": num_ledgers,
#         "first_ts": selected[0][0].isoformat() if selected else None,
#         "last_ts": selected[-1][0].isoformat() if selected else None,
#     }


# def write_summary(run_dir, num_nodes, leader_log):
#     summary = parse_committed_ops(leader_log)

#     summary_path = run_dir / "summary.txt"

#     with open(summary_path, "w") as f:
#         f.write(f"num_nodes={num_nodes}\n")
#         f.write(f"num_accounts={NUM_TEST_ACCOUNTS}\n")
#         f.write(f"ops_per_tx={SCP_OPS_PER_TX}\n")
#         f.write(f"measurement_window={MEASURE_START_SEC}-{MEASURE_END_SEC}s\n")
#         f.write(f"committed_ops={summary['ops']}\n")
#         f.write(f"committed_txs={summary['txs']}\n")
#         f.write(f"throughput_ops_s={summary['throughput_ops_s']:.2f}\n")
#         f.write(f"avg_ops_per_ledger={summary['avg_ops_per_ledger']:.2f}\n")
#         f.write(f"avg_txs_per_ledger={summary['avg_txs_per_ledger']:.2f}\n")
#         f.write(f"min_ops_per_ledger={summary['min_ops_per_ledger']}\n")
#         f.write(f"max_ops_per_ledger={summary['max_ops_per_ledger']}\n")
#         f.write(f"min_txs_per_ledger={summary['min_txs_per_ledger']}\n")
#         f.write(f"max_txs_per_ledger={summary['max_txs_per_ledger']}\n")
#         f.write(f"num_ledgers={summary['num_ledgers']}\n")
#         f.write(f"first_ts={summary['first_ts']}\n")
#         f.write(f"last_ts={summary['last_ts']}\n")

#     print("\n" + "=" * 80)
#     print(f"RESULT num_nodes={num_nodes}")
#     print(f"committed_ops={summary['ops']}")
#     print(f"committed_txs={summary['txs']}")
#     print(f"throughput_ops_s={summary['throughput_ops_s']:.2f}")
#     print(f"avg_ops_per_ledger={summary['avg_ops_per_ledger']:.2f}")
#     print(f"avg_txs_per_ledger={summary['avg_txs_per_ledger']:.2f}")
#     print(f"min_ops_per_ledger={summary['min_ops_per_ledger']}")
#     print(f"max_ops_per_ledger={summary['max_ops_per_ledger']}")
#     print(f"num_ledgers={summary['num_ledgers']}")
#     print(f"summary={summary_path}")
#     print("=" * 80 + "\n")

#     return summary


# import os
# os.system('git add .; git commit -m "update"; git push')

# # =============================================================================
# # Main experiment
# # =============================================================================

# if DELETE_BEFORE_FIRST_RUN:
#     instances = fetch_existing_instances()
#     if instances:
#         run_parallel(delete_instance, instances, max_workers=min(32, len(instances)))

# ensure_instances()

# max_node_records = get_node_records(MAX_NUM_NODES)

# print("All available node records:")
# for r in max_node_records:
#     print(r)

# all_results = []

# os.system('git add .; git commit -m "update"; git push')

# for run_idx, num_nodes in enumerate(NUM_NODES_LIST):
#     run_name = f"stellar_native_nodes_{num_nodes}_accounts_{NUM_TEST_ACCOUNTS}"
#     run_dir = RESULTS_DIR / run_name
#     run_dir.mkdir(parents=True, exist_ok=True)

#     print("\n" + "#" * 100)
#     print(
#         f"Starting Stellar native run: "
#         f"nodes={num_nodes}, accounts={NUM_TEST_ACCOUNTS}, opsPerTx={SCP_OPS_PER_TX}"
#     )
#     print("#" * 100)

#     active_indices = list(range(num_nodes))
#     active_node_records = get_node_records(num_nodes)
#     active_node_ips = [r[3] for r in active_node_records]

#     print("Active node records:")
#     for r in active_node_records:
#         print(r)

#     print("Active node IPs:", active_node_ips)

#     # Kill all currently-existing nodes before each subrun.
#     # For first run this is 48; after deletion it becomes smaller.
#     existing_before_run = fetch_existing_instances()
#     existing_indices = []
#     for name, _ in existing_before_run:
#         if name.startswith("tsm-sc-"):
#             try:
#                 idx = int(name.rsplit("-", 1)[1])
#                 existing_indices.append(idx)
#             except ValueError:
#                 pass

#     if existing_indices:
#         print("Killing Stellar on all existing tsm-sc nodes before subrun...")
#         run_parallel(
#             kill_stellar,
#             sorted(existing_indices),
#             max_workers=min(48, len(existing_indices)),
#         )

#     # Compile only once, during the first subrun.
#     if (not COMPILE_ONLY_FIRST_SUBRUN) or (run_idx == 0):
#         print("Compiling Stellar Core on all max nodes. This happens only for first subrun.")

#         run_parallel(
#             git_pull_node,
#             range(MAX_NUM_NODES),
#             max_workers=min(48, MAX_NUM_NODES),
#         )

#         run_parallel(
#             compile_node,
#             range(MAX_NUM_NODES),
#             max_workers=min(48, MAX_NUM_NODES),
#         )
#     else:
#         print("Skipping compile for this subrun.")


#     print("Stopping active nodes...")
#     run_parallel(
#         kill_stellar,
#         active_indices,
#         max_workers=min(48, num_nodes),
#     )

#     # Generate configs for the current node count.
#     # This rewrites tsm_ips.txt so the generated private network has exactly num_nodes validators.
#     write_tsm_ips(active_node_ips)
#     generate_stellar_private_configs(num_nodes)

#     # Clean and copy configs only to active nodes.
#     run_parallel(
#         clean_remote_private,
#         active_indices,
#         max_workers=min(48, num_nodes),
#     )

#     run_parallel(
#         copy_private_to_node,
#         active_indices,
#         max_workers=min(48, num_nodes),
#     )

#     # Start only active nodes.
#     run_parallel(
#         run_stellar_node,
#         active_indices,
#         max_workers=min(48, num_nodes),
#     )

#     print(f"Nodes started. Running for {RUN_DURATION_SEC} seconds...")
#     time.sleep(RUN_DURATION_SEC)

#     print("Stopping active nodes...")
#     run_parallel(
#         kill_stellar,
#         active_indices,
#         max_workers=min(48, num_nodes),
#     )

#     print("Collecting logs...")
#     run_parallel(
#         lambda i: collect_node_log(i, run_dir),
#         active_indices,
#         max_workers=min(48, num_nodes),
#     )

#     leader_log = run_dir / "tsm-sc-000-node1-stellar-core.log"

#     if leader_log.exists():
#         result = write_summary(run_dir, num_nodes, leader_log)
#         result["num_nodes"] = num_nodes
#         all_results.append(result)
#     else:
#         print(f"WARNING: leader log not found: {leader_log}")

#     # Delete nodes that are no longer needed for the next smaller subrun.
#     if run_idx + 1 < len(NUM_NODES_LIST):
#         next_num_nodes = NUM_NODES_LIST[run_idx + 1]
#         delete_nodes_not_needed_next(num_nodes, next_num_nodes)


# # =============================================================================
# # Final CSV summary
# # =============================================================================

# csv_path = RESULTS_DIR / f"stellar_native_accounts_{NUM_TEST_ACCOUNTS}_nodes_sweep_summary.csv"

# with open(csv_path, "w") as f:
#     f.write(
#         "num_nodes,num_accounts,committed_ops,committed_txs,throughput_ops_s,"
#         "avg_ops_per_ledger,avg_txs_per_ledger,"
#         "min_ops_per_ledger,max_ops_per_ledger,"
#         "min_txs_per_ledger,max_txs_per_ledger,"
#         "num_ledgers,first_ts,last_ts\n"
#     )

#     for r in all_results:
#         f.write(
#             f"{r['num_nodes']},"
#             f"{NUM_TEST_ACCOUNTS},"
#             f"{r['ops']},"
#             f"{r['txs']},"
#             f"{r['throughput_ops_s']:.2f},"
#             f"{r['avg_ops_per_ledger']:.2f},"
#             f"{r['avg_txs_per_ledger']:.2f},"
#             f"{r['min_ops_per_ledger']},"
#             f"{r['max_ops_per_ledger']},"
#             f"{r['min_txs_per_ledger']},"
#             f"{r['max_txs_per_ledger']},"
#             f"{r['num_ledgers']},"
#             f"{r['first_ts']},"
#             f"{r['last_ts']}\n"
#         )

# print(f"\nFinal CSV summary: {csv_path}")

# if DELETE_ALL_AFTER_FINAL_RUN:
#     print("Deleting remaining tsm-sc instances after final run...")
#     instances = fetch_existing_instances()
#     tsm_instances = [
#         (name, zone)
#         for name, zone in instances
#         if name.startswith("tsm-sc-")
#     ]

#     run_parallel(
#         delete_instance,
#         tsm_instances,
#         max_workers=min(32, len(tsm_instances)),
#     )

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


Everything up-to-date


Missing instances: ['tsm-sc-000', 'tsm-sc-001', 'tsm-sc-002', 'tsm-sc-003', 'tsm-sc-004', 'tsm-sc-005', 'tsm-sc-006', 'tsm-sc-007', 'tsm-sc-008', 'tsm-sc-009', 'tsm-sc-010', 'tsm-sc-011', 'tsm-sc-012', 'tsm-sc-013', 'tsm-sc-014', 'tsm-sc-015', 'tsm-sc-016', 'tsm-sc-017', 'tsm-sc-018', 'tsm-sc-019', 'tsm-sc-020', 'tsm-sc-021', 'tsm-sc-022', 'tsm-sc-023', 'tsm-sc-024', 'tsm-sc-025', 'tsm-sc-026', 'tsm-sc-027', 'tsm-sc-028', 'tsm-sc-029', 'tsm-sc-030', 'tsm-sc-031', 'tsm-sc-032', 'tsm-sc-033', 'tsm-sc-034', 'tsm-sc-035', 'tsm-sc-036', 'tsm-sc-037', 'tsm-sc-038', 'tsm-sc-039', 'tsm-sc-040', 'tsm-sc-041', 'tsm-sc-042', 'tsm-sc-043', 'tsm-sc-044', 'tsm-sc-045', 'tsm-sc-046', 'tsm-sc-047']
Creating tsm-sc-000
Creating tsm-sc-001
Creating tsm-sc-002
Creating tsm-sc-003
Creating tsm-sc-004
Creating tsm-sc-005
Creating tsm-sc-006
Creating tsm-sc-007
Creating tsm-sc-008
Creating tsm-sc-009
Creating tsm-sc-010
Creating tsm-sc-011
Creating tsm-sc-012
Creating tsm-sc-013
Creating tsm-sc-014
Creating

Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-b/instances/tsm-sc-014].
Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-b/instances/tsm-sc-011].
 - You are creating a global DNS VM. VM instances using global DNS are vulnerable to cross-regional outages. To reduce the risk of widespread service disruption, use zonal DNS instead. Learn more at https://cloud.google.com/compute/docs/networking/zonal-dns?utm_id=vm-insert

Created [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-b/instances/tsm-sc-004].


NAME        ZONE           MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP     STATUS
tsm-sc-014  us-central1-b  e2-standard-2               10.128.0.7   35.253.119.251  RUNNING


...

2026-06-29T22:40:01.802 [default INFO] Config from /home/tejas/stellar-private/node1/stellar-core.cfg
2026-06-29T22:40:01.804 [default INFO] Generated QUORUM_SET: {
   "t" : 3,
   "v" : [ "node2", "GCBKK", "node4", "node3" ]
}

2026-06-29T22:40:01.804 [default WARNING] Adjusted TARGET_PEER_CONNECTIONS to 336 due to insufficient MAX_ADDITIONAL_PEER_CONNECTIONS=1000
2026-06-29T22:40:01.804 [default INFO] Assigning calculated value of 1 to FAILURE_SAFETY
2026-06-29T22:40:01.858 [default INFO] Config from /home/tejas/stellar-private/node2/stellar-core.cfg
2026-06-29T22:40:01.860 [default INFO] Generated QUORUM_SET: {
   "t" : 3,
   "v" : [ "GAHCF", "node1", "node4", "node3" ]
}

2026-06-29T22:40:01.860 [default WARNING] Adjusted TARGET_PEER_CONNECTIONS to 336 due to insufficient MAX_ADDITIONAL_PEER_CONNECTIONS=1000
2026-06-29T22:40:01.860 [default INFO] Assigning calculated value of 1 to FAILURE_SAFETY
2026-06-29T22:40:01.890 [default INFO] Config from /home/tejas/stellar-private/node3/ste

Initializing database for node4...
✅ 4-node private Stellar network setup complete!
Start the nodes with (substituting node number):
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node1/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node2/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node3/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node4/stellar-core.cfg &


2026-06-29T22:40:01.924 [default INFO] Config from /home/tejas/stellar-private/node4/stellar-core.cfg
2026-06-29T22:40:01.926 [default INFO] Generated QUORUM_SET: {
   "t" : 3,
   "v" : [ "node2", "node1", "GDCFX", "node3" ]
}

2026-06-29T22:40:01.926 [default WARNING] Adjusted TARGET_PEER_CONNECTIONS to 336 due to insufficient MAX_ADDITIONAL_PEER_CONNECTIONS=1000
2026-06-29T22:40:01.926 [default INFO] Assigning calculated value of 1 to FAILURE_SAFETY


Nodes started. Running for 210 seconds...
Stopping active nodes...

RESULT num_nodes=4
committed_ops=172800
committed_txs=1728
throughput_ops_s=2880.00
avg_ops_per_ledger=9094.74
avg_txs_per_ledger=90.95
min_ops_per_ledger=6000
max_ops_per_ledger=10000
num_ledgers=19
summary=/home/tejas/work/experiments/stellar_native/stellar_native_nodes_4_accounts_100/summary.txt


Final CSV summary: /home/tejas/work/experiments/stellar_native/stellar_native_accounts_100_nodes_sweep_summary.csv
Deleting remaining tsm-sc instances after final run...
Deleting tsm-sc-000 in us-central1-b
Deleting tsm-sc-001 in us-central1-b
Deleting tsm-sc-002 in us-central1-b
Deleting tsm-sc-003 in us-central1-b


Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-b/instances/tsm-sc-003].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-b/instances/tsm-sc-002].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-b/instances/tsm-sc-001].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-central1-b/instances/tsm-sc-000].


[main e24d2ad] update
 4 files changed, 13740 insertions(+), 1484 deletions(-)


To github.com:tejas-shivanand-mane/stellar-core.git
   f202352..e24d2ad  main -> main


0

In [2]:

    # Fetch all tsm-sc-* instances across ALL zones
    fetch_cmd = f'''
    gcloud compute instances list \
        --project={project} \
        --filter="name~'^tsm-sc-'" \
        --format="value(name,zone)"
    '''
    
    output = subprocess.check_output(fetch_cmd, shell=True).decode().strip()
    instances = []
    
    for line in output.splitlines():
        if line.strip():
            name, zone = line.split()
            instances.append((name, zone))
    
    print("\n➡ Existing instances to delete:")
    for name, zone in instances:
        print(f"  - {name} ({zone})")
    
    def delete_instance(name, zone):
        cmd = f'''
        gcloud compute instances delete {name} \
            --zone={zone} \
            --project={project} \
            --quiet
        '''
        print(f"🗑️ Deleting {name} in {zone}")
        return subprocess.call(cmd, shell=True)
    
    if instances:
        with concurrent.futures.ThreadPoolExecutor(max_workers=32) as executor:
            futures = [
                executor.submit(delete_instance, name, zone)
                for name, zone in instances
            ]
            concurrent.futures.wait(futures)
    
        print("\n🧹 All tsm-sc-* instances deleted across all regions.\n")
    else:
        print("\n✔ No tsm-sc-* instances found.\n")


➡ Existing instances to delete:

✔ No tsm-sc-* instances found.

